## 0.1 Pre-registered audit contract

This notebook performs a full-protocol segmentation audit on the experimental MJ1 charging trajectories. The purpose is not to search for a favorable interpretation, but to assign the measured state-equivalent first-passage gains to predefined protocol segments before any mechanism statement is made.

---

### Locked constants and conventions

| Quantity | Locked value | Meaning |
|---|---:|---|
| `cell_id` | `LG_INR18650_MJ1` | Experimental cell used for the MJ1 dataset |
| `Q_nom_Ah` | `3.4 Ah` | Project-level analysis normalization capacity derived from the standardized 1C/0.5C pre-DC–AC capacity characterization of the same MJ1 cell |
| `Vmax_V` | `4.2 V` | Upper voltage boundary for CC-to-voltage-limited transition |
| `I_cutoff_A` | `0.05 A` | CV termination threshold |
| `I_cutoff_definition` | `absolute_50mA_first_reached` | First time the measured current reaches the 50 mA cutoff |
| `I_charge_onset_threshold_A` | `0.05 A` | Current threshold used only to trim rest/preconditioning before the retained charge interval |
| `phase_convention` | `charge_first` | Experimental MJ1 ARB waveform starts with the charging half-wave |
| `ambient_temperature_C` | `20 °C` | Laboratory ambient boundary condition during the experimental campaign |
| `temperature_control_type` | `ambient_lab_no_chamber` | No temperature chamber was used |
| `temperature_sensor_type` | `Pt100` | Cell-surface temperature sensor |
| `temperature_sensor_placement` | `axial_cell_surface` | Pt100 sensor attached axially to the cell surface |
| `temperature_data_source` | `measured_surface_temperature` | Surface temperature was measured during the experiment |
| `temperature_alignment_method` | `segment_level_summary_only` | Temperature is used only as protocol-level summary metadata |
| `temperature_to_NGU201_alignment_required` | `false` | No sample-by-sample temperature alignment to NGU201 voltage/current is required |
| `Q80_nominal_Ah` | `2.72 Ah` | `0.80 × 3.4 Ah` |
| `Q90_nominal_Ah` | `3.06 Ah` | `0.90 × 3.4 Ah` |

`Q_nom_Ah = 3.4 Ah` is used here as the project-level normalization capacity because the experimental current convention defines `1C = 3.4 A`. This value originates from the standardized 1C and 0.5C charge/discharge characterization performed on the same MJ1 cell before the DC–AC experimental campaign. It is therefore an experiment-specific analysis capacity, not the manufacturer nameplate capacity. The manufacturer nominal capacity of the LG INR18650 MJ1 is higher, but the present audit follows the established capacity and C-rate convention used throughout the DC–AC dataset.

`I_cutoff_A` and `I_charge_onset_threshold_A` both use the numerical value `0.05 A`, but they serve different roles. `I_cutoff_A` defines CV termination at the end of charge, whereas `I_charge_onset_threshold_A` is used only to trim rest/preconditioning before the retained charging interval. They are treated as independent constants even though their numerical values are identical in this audit.

All current signals used for charge integration are converted into an analysis-oriented sign convention:

$$
I_Q(t)>0
$$

means net charge input into the cell. Instrument-specific or model-specific sign conventions are normalized before computing `Q_net(t)`.

For the experimental MJ1 charge-first prescribed waveform, the geometry reference is defined in this charge-positive convention as:

$$
I_{Q,\mathrm{geom,DC}}(t)=I_{\mathrm{DC}}
$$

$$
I_{Q,\mathrm{geom,DCAC}}(t)=I_{\mathrm{DC}}+I_{\mathrm{AC}}\sin(2\pi f_{\mathrm{Hz}} t)
$$

This sinusoidal prescribed waveform expression applies only to Segment A, where AC remains active and the residual is defined. Segment B and Segment D do not consume the prescribed waveform for residual computation. They report only raw first-passage quantities.

The corresponding PyBaMM implementation may use the opposite internal sign convention. That mapping is not used to define the experimental `Q_net(t)`.

---

### Frequency and period convention

The audit distinguishes the experimental frequency label from the actual waveform frequency.

The waveform frequency used in all calculations is the actual frequency in Hz:

$$
f_{\mathrm{Hz}}
$$

The experimental frequency is generated from the time-scale label according to:

$$
f_{\mathrm{Hz}}=\frac{1}{2\pi \tau_{\mathrm{eff}}}
$$

where `τ_eff` is the effective time-scale associated with the experimental frequency label.

For a label multiplier `m_tau`, the effective time-scale is:

$$
\tau_{\mathrm{eff}}=m_{\tau}\tau_{\mathrm{label}}
$$

The AC period is therefore:

$$
T_{\mathrm{AC}}=\frac{1}{f_{\mathrm{Hz}}}=2\pi\tau_{\mathrm{eff}}
$$

For `τ_label = 11.1 s`, the corresponding values are:

| Label | `τ_eff` | `f_Hz` | `T_AC` |
|---|---:|---:|---:|
| `0.1τ` | `1.11 s` | `0.143 Hz` | `6.97 s` |
| `1τ` | `11.1 s` | `0.0143 Hz` | `69.7 s` |
| `10τ` | `111 s` | `0.00143 Hz` | `697 s` |

Thus, the default AC-off persistence window also depends on the frequency label:

$$
AC\_off\_persistence\_s\_default = 3\times T_{\mathrm{AC}}
$$

Examples:

| Label | `AC_off_persistence_s_default` |
|---|---:|
| `1τ` | approximately `209 s` |
| `10τ` | approximately `2092 s`, or `34.9 min` |

This is why a separate low-frequency AC-off exception is required below. For low-frequency cases such as `10τ`, requiring three complete AC periods after the candidate AC-off event may exceed the available post-transition record.

No calculation in this notebook may use a bare variable `f` without specifying whether it denotes `frequency_Hz`, `f_Hz`, or a dimensionless frequency label. All period-based detection logic must use:

$$
T_{\mathrm{AC,s}}=\frac{1}{frequency\_Hz}
$$

---

### Initial-state and Q-coordinate convention

The charge-phase origin is defined by the standardized DC–AC experimental preconditioning protocol:

1. The cell is discharged at 1C to 2.5 V.
2. The discharge endpoint is reached when the current first reaches 25 mA.
3. The cell rests for 63 min 25 s.
4. The DC or DC–AC charging protocol starts.

The 63 min 25 s rest duration follows the frozen DC–AC experimental protocol used in the thesis dataset and is treated as part of the standardized initial-state definition.

This protocol defines the charge-phase start as the experimental `SOC = 0` reference for the present audit.

Accordingly, `Q_net(t)` is a relative charged-capacity coordinate measured from the start of the retained charging interval:

$$
Q_{\mathrm{net}}(t=0)=0
$$

The nominal anchors are therefore expressed in the same relative charged-capacity coordinate:

$$
Q80_{\mathrm{nominal}}=0.80\times Q_{\mathrm{nom}}
$$

$$
Q90_{\mathrm{nominal}}=0.90\times Q_{\mathrm{nom}}
$$

This convention is valid only because the retained charge-phase trajectory starts from the standardized discharged-and-rested state. If a file contains preconditioning, rest, or discharge segments before the charge phase, these segments must be removed before computing `Q_net(t)`, while the retained charging interval preserves the raw signed current waveform.

---

### Measurement source and timebase convention

For the Day21A experimental MJ1 audit, both voltage and current are taken from the NGU201 record.

Therefore:

- `timebase_source = NGU201_single_timebase`
- `voltage_source = NGU201`
- `current_source = NGU201`
- `time_alignment_method = native_same_device_timebase`
- `voltage_current_alignment_status = native_aligned_same_record`

No DAS–NGU cross-device interpolation is used in this audit. If a later dataset uses voltage and current from different instruments, this contract does not apply without an explicit time-alignment audit.

---

### Temperature measurement convention

The experiments were conducted under laboratory ambient conditions of approximately 20 °C without a temperature chamber.

The cell-surface temperature was measured during the experiments using a Pt100 sensor attached axially to the cell surface. Temperature data are used in this audit only as per-protocol summary statistics, not as sample-by-sample signals aligned to the NGU201 voltage/current timebase.

The temperature summary fields are:

- `T_surface_max_C`
- `T_surface_mean_C`
- `temperature_sensor_type = Pt100`
- `temperature_sensor_placement = axial_cell_surface`
- `temperature_data_source = measured_surface_temperature`
- `temperature_alignment_method = segment_level_summary_only`
- `temperature_to_NGU201_alignment_required = false`
- `temperature_control_type = ambient_lab_no_chamber`

The ambient temperature describes the laboratory boundary condition. `T_surface_max_C` and `T_surface_mean_C` describe the measured cell-surface thermal response over the corresponding protocol.

Because temperature is used only as protocol-level summary metadata in Day21A, no interpolation of temperature data onto the NGU201 voltage/current timebase is performed. If a later analysis uses time-resolved temperature together with voltage/current trajectories, a separate temperature-timebase alignment audit is required.

---

### Charge-phase time-zero definition

For all experimental files, the audit time axis is reset before charge integration.

The locked time-zero rule is:

$$
t=0
$$

is the first sample of the retained charging interval where the charge-positive measured current satisfies:

$$
I_Q(t)\ge 0.05\,\mathrm{A}
$$

for at least three consecutive samples.

Segments before this point, including rest, preconditioning, discharge, or file header artifacts, are excluded from the retained audit interval. Within the retained interval, the measured signed current waveform is preserved without rectification or monotonic correction.

The same time-zero definition is used for the measured trajectory and the prescribed geometry reference.

The charge-start threshold is used only to trim rest and preconditioning regions. It does not by itself establish ARB phase alignment. Phase alignment is handled separately through the geometry phase reference status.

---

### Geometry phase reference

The prescribed geometry reference assumes that the retained audit time zero coincides with the ARB phase-zero trigger.

If this alignment can be verified from metadata or from the measured current waveform, the status is:

`geometry_phase_reference_status = verified`

If direct trigger metadata is not available, the phase offset must be estimated from the measured current waveform and persisted as:

- `geometry_phase_offset_s`
- `geometry_phase_offset_rad`
- `geometry_phase_reference_status = estimated_from_current_waveform`

The geometry waveform then becomes:

$$
I_{Q,\mathrm{geom,DCAC}}(t)=I_{\mathrm{DC}}+I_{\mathrm{AC}}\sin(2\pi f_{\mathrm{Hz}} t+\phi)
$$

where `phi = geometry_phase_offset_rad`.

If the phase offset is estimated from the measured current waveform, the locked method is a least-squares fit of the prescribed sinusoidal current model to the early retained charging segment:

$$
I_Q(t)=I_{\mathrm{DC}}+I_{\mathrm{AC}}\sin(2\pi f_{\mathrm{Hz}}t+\phi)
$$

The fit uses protocol metadata for `I_DC`, `I_AC`, and `f_Hz`; only the phase offset `φ` is estimated.

The fitting window is the first available window after charge onset with length:

$$
\min(5T_{\mathrm{AC}},\ t_{\mathrm{ACoff,candidate}})
$$

If this window contains less than one complete AC period, phase estimation is marked as unresolved.

The resulting phase offset is persisted as:

- `geometry_phase_offset_rad`
- `geometry_phase_offset_s = geometry_phase_offset_rad / (2π f_Hz)`
- `geometry_phase_reference_status = estimated_from_current_waveform`

If neither trigger alignment nor waveform-based phase estimation is available, Segment-A residual is not interpretable and must be reported as:

`geometry_phase_reference_status = unresolved`

In that case:

- `segment_A_dt_resid_mean_s = NaN`
- `segment_A_dt_resid_max_abs_s = NaN`
- `segment_A_dt_resid_p95_abs_s = NaN`
- `mechanism_verdict = not_applicable_without_geometry_phase_reference`

---

### Strict-net charge convention

All measured charge trajectories are computed using strict-net signed integration after conversion to the charge-positive analysis convention:

$$
Q_{\mathrm{net}}(t)=\int_0^t I_Q(\tau)\,\mathrm{d}\tau
$$

No rectification is applied. Negative-current intervals in DC–AC trajectories are retained and contribute negatively to `Q_net`. No cumulative maximum is imposed. Target-state times are defined by first passage:

$$
t(Q^\*)=\inf\{t:Q_{\mathrm{net}}(t)\ge Q^\*\}
$$

Raw state-equivalent time gain is defined as:

$$
\Delta t_{\mathrm{raw}}(Q)=t_{\mathrm{DC}}(Q)-t_{\mathrm{DCAC}}(Q)
$$

Positive values indicate that the DC–AC trajectory reaches the same net charge state earlier than the DC reference.

---

### Geometry reference for experimental MJ1 data

For the experimental MJ1 audit, the geometry-only reference is computed from the prescribed current waveform, not from measured current and not from a battery model.

The prescribed geometry charge is:

$$
Q_{\mathrm{geom}}(t)=\int_0^t I_{Q,\mathrm{geom}}(\tau)\,\mathrm{d}\tau
$$

The geometry-only first-passage time is:

$$
t_{\mathrm{geom}}(Q^\*)=\inf\{t:Q_{\mathrm{geom}}(t)\ge Q^\*\}
$$

The geometry-only time gain is:

$$
\Delta t_{\mathrm{geom}}(Q)=t_{\mathrm{geom,DC}}(Q)-t_{\mathrm{geom,DCAC}}(Q)
$$

The Segment-A residual is therefore:

$$
\Delta t_{\mathrm{resid}}(Q)=\Delta t_{\mathrm{raw}}(Q)-\Delta t_{\mathrm{geom}}(Q)
$$

This residual is interpreted only in Segment A. It estimates the portion of the measured first-passage gain that is not explained by the prescribed current-waveform geometry.

---

### Q-grid definition for Segment-A residual statistics

Segment-A residual statistics are computed on a fixed charge grid.

| Quantity | Locked value |
|---|---:|
| `segment_A_Q_lo_Ah` | `0.050 Ah` |
| `Q_grid_step_Ah` | `0.010 Ah` |
| `Q_grid_definition_method` | fixed 10 mAh grid within Segment A |
| `Q_grid_lower_Ah` | first grid value greater than or equal to `segment_A_Q_lo_Ah` |
| `Q_grid_upper_Ah` | last grid value smaller than or equal to `segment_A_Q_hi_Ah` |

The Segment-A residual grid does not start at `Q = 0`. The locked lower bound is:

`segment_A_Q_lo_Ah = 0.050 Ah`

The first 50 mAh are excluded from residual statistics to avoid start-up transients, time-zero quantization, and first-passage sensitivity near the origin. Anchor assignment still uses the full Q trajectory; this lower bound applies only to Segment-A residual statistics.

`segment_A_Q_lo_Ah` is a fixed global audit lower bound and is not recomputed per record. In Day21A it is always set to `0.050 Ah`. The per-record output field `segment_A_Q_lo_Ah` records this fixed value for traceability.

The same Q-grid is used for `median`, `max`, `max_abs`, and `p95_abs` residual statistics. This prevents the spike guard from depending on ad hoc grid density.

Minimum grid-count guard:

| Quantity | Locked value |
|---|---:|
| `Q_grid_min_count_segmentA` | `30` |

If the Segment-A Q-grid contains fewer than 30 points, `segment_A_dt_resid_p95_abs_s` is not reported and must be written as `NaN`. In that case, the spike guard cannot be applied, and the Segment-A residual classification is based only on median, maximum, and maximum absolute residual values. The verdict must carry the caveat:

`segment_A_grid_count_below_p95_requirement`

---

### Segment definitions

The full protocol is partitioned by the first voltage-boundary events of the DC and DC–AC trajectories.

#### Segment A — shared prescribed-current region

Segment A covers the region before the DC–AC voltage-boundary / AC-off boundary:

$$
Q \le Q_{\mathrm{segmentB,start}}
$$

The Segment-A upper boundary is defined as:

$$
Q_{\mathrm{segmentB,start}}=Q_{\mathrm{DCAC}}(t_{\mathrm{B,start}})
$$

where, under non-negative AC-off lag:

$$
t_{\mathrm{B,start}}=\min(t_{\mathrm{Vmax,DCAC}},t_{\mathrm{ACoff,DCAC}})
$$

`Q_segmentB_start_Ah` is evaluated from the measured DC–AC strict-net charge trajectory, not from the prescribed geometry reference.

In the usual case, this is equal to or very close to `Q_DCAC,Vmax_Ah`. The audit table must nevertheless report both:

- `Q_Vmax_DCAC_Ah`
- `Q_segmentB_start_Ah`
- `segment_A_Q_hi_definition`

Both trajectories are still evaluated within their prescribed-current region in Segment A. Only in this segment is the geometry-corrected residual defined:

$$
\Delta t_{\mathrm{resid}}(Q)=\Delta t_{\mathrm{raw}}(Q)-\Delta t_{\mathrm{geom}}(Q)
$$

#### Segment B — voltage-boundary / control-state split

Segment B covers:

$$
Q_{\mathrm{segmentB,start}} < Q \le Q_{\mathrm{DC,Vmax}}
$$

The DC–AC trajectory has reached `Vmax`, switched off AC, or entered a voltage-limited / CV-coupled control state, while the DC reference remains in CC. Segment B reports only raw `Δt(Q)`. No residual is defined or interpreted here.

#### Segment D — late CV feedback region

Segment D covers:

$$
Q > Q_{\mathrm{DC,Vmax}}
$$

Both trajectories are in voltage-limited / CV-feedback-controlled regions. Segment D reports only raw `Δt(Q)`. No residual is defined or interpreted here.

#### Segment C handling

Segment C is not used as a separate class in Day21A.

`segment_C_handling = absorbed_into_segment_B_start_no_separate_segment`

The AC-off transition is absorbed into Segment B, which starts at the DC–AC voltage-boundary / AC-off event.

#### Residual scope

`dt_resid_scope = Segment_A_only_B_and_D_use_raw_dt`

The notebook must not generate or report:

- `dt_resid_segB`
- `dt_resid_segD`

Segment B and Segment D may only contain raw first-passage quantities and boundary/control-state descriptors.

---

### Outside-window anchor handling

An anchor is classified as:

`outside_common_Q_window`

if:

$$
Q^\*>\min(Q_{\mathrm{final,DC}},Q_{\mathrm{final,DCAC}})
$$

Such an anchor is physically unreachable by at least one trajectory and must not be assigned to Segment A, Segment B, or Segment D.

Decision handling:

- If all evaluated anchors are outside the common Q-window, the verdict is `not_applicable_protocol_does_not_reach_target_state`.
- If only some anchors are outside the common Q-window, the in-window anchors remain decisive.
- The verdict must carry the caveat `nominal_anchor_outside_common_Q_window`.

This rule is especially relevant for nominal anchors such as `Q90_nominal_Ah = 3.06 Ah` when the experimentally reached final charge is lower than the nominal target.

---

### Voltage-boundary and AC-off event detection hierarchy

The voltage-boundary and AC-off events are detected using the following priority order:

1. Actual cycler or protocol transition timestamp, if available.
2. Detected AC-off / CV transition timestamp from the current waveform.
3. First `V >= 4.2 V` with deglitch validation.

The actual method used for each event must be persisted separately as:

- `Vmax_detection_method_used`
- `AC_off_detection_method_used`

The notebook must output:

- `Vmax_detection_method_used`
- `AC_off_detection_method_used`
- `t_Vmax_detected_s`
- `t_AC_off_detected_s`
- `t_segmentB_start_s`
- `AC_off_lag_s = t_AC_off_DCAC_s - t_Vmax_DCAC_s`

#### Priority 2 current-waveform AC-off detection

If no explicit stage marker is available, AC-off is detected from the measured current waveform using a pre-registered envelope-disappearance rule.

For a DC–AC protocol with AC amplitude `I_AC`, the frequency used in the audit is the actual waveform frequency in Hz, denoted as `f_Hz` or `frequency_Hz`.

The corresponding AC period is:

$$
T_{\mathrm{AC}}=\frac{1}{f_{\mathrm{Hz}}}=2\pi\tau_{\mathrm{eff}}
$$

The locked default detection parameters are:

| Quantity | Locked value |
|---|---:|
| `AC_off_envelope_tolerance_coefficient` | `0.05` |
| `AC_off_envelope_tolerance_A` | `0.05 × I_AC`, protocol-specific |
| `AC_off_persistence_s_default` | `3 × T_AC` |
| `AC_off_detection_min_samples` | all samples within the persistence window |
| `AC_off_unresolved_rule` | unresolved if the record after candidate AC-off is shorter than the required persistence window |

The locked quantity is the relative coefficient `0.05`. The absolute tolerance in ampere is protocol-specific because it scales with `I_AC`.

Default detection sequence:

1. Estimate the local non-oscillatory current baseline after the candidate transition.
2. Compute the deviation of the measured current from this local baseline.
3. Identify the first candidate time at which the AC envelope remains below `0.05 × I_AC`.
4. Require this condition to persist for at least `3 × T_AC`.
5. Validate that the subsequent current trajectory is compatible with pure DC or CV decay rather than continued sinusoidal modulation.

#### Low-frequency AC-off exception

The low-frequency AC-off exception is a fallback rule, not a pre-emptive replacement of the default priority-2 detection rule.

The default rule is attempted first. If the default `3 × T_AC` persistence requirement cannot be satisfied within the available post-transition record, the low-frequency exception may be applied.

This situation is expected mainly for low-frequency labels such as `10τ`, where `T_AC` is approximately 697 s and `3 × T_AC` is approximately 2092 s. In such cases, requiring three complete post-transition AC periods may exceed the useful CV-transition record length.

The locked low-frequency exception parameters are:

| Quantity | Locked value |
|---|---:|
| `AC_off_low_frequency_voltage_window_V` | `[4.195, 4.205]` |
| `AC_off_low_frequency_min_persistence_s` | `120 s` |
| `AC_off_low_frequency_envelope_tolerance_coefficient` | `0.05` |
| `AC_off_low_frequency_envelope_tolerance_A` | `0.05 × I_AC`, protocol-specific |
| `AC_off_low_frequency_required_trend` | monotonic CV-current decay or pure-DC continuation |

Under this exception, AC-off detection may be accepted if all conditions are met:

1. The candidate transition lies within the voltage-boundary neighborhood `[4.195 V, 4.205 V]`.
2. The programmed sinusoidal component disappears according to the `0.05 × I_AC` envelope tolerance.
3. The condition persists for at least 120 s.
4. The subsequent current trajectory is compatible with monotonic CV-current decay or pure-DC continuation.
5. Continued sinusoidal modulation is not observed after the candidate transition.

If these conditions are not satisfied, priority 2 is marked as unresolved and the method falls back to priority 3.

#### Priority 3 voltage deglitch rule

The voltage deglitch rule applies only to priority 3.

`deglitch_applies_to_priorities = [3]`

The fallback voltage event is the first index where `V >= 4.2 V`, provided that at least three consecutive samples remain near or above `Vmax`, or the event is followed by a protocol transition consistent with AC-off / CV entry.

---

### AC-off lag guard

Normally, the expected ordering is:

$$
t_{\mathrm{ACoff,DCAC}} \ge t_{\mathrm{Vmax,DCAC}}
$$

If:

$$
t_{\mathrm{ACoff,DCAC}} - t_{\mathrm{Vmax,DCAC}} < 0
$$

then AC-off precedes the detected voltage-boundary event. In this case:

- `segment_framework_status = AC_off_precedes_Vmax`
- `mechanism_verdict = not_applicable_without_event_reaudit`
- no Interpretation A or Interpretation B verdict is allowed

This guard prevents a prematurely detected AC-off event from being misclassified as a voltage-boundary / control-state split.

If the lag is non-negative, the Segment B start is defined as:

$$
t_{\mathrm{B,start}}=\min(t_{\mathrm{Vmax,DCAC}},t_{\mathrm{ACoff,DCAC}})
$$

and:

$$
Q_{\mathrm{B,start}}=Q_{\mathrm{DCAC}}(t_{\mathrm{B,start}})
$$

---

### Segment ordering guard

The expected charge-ordering is:

$$
Q_{\mathrm{segmentB,start}} < Q_{\mathrm{DC,Vmax}}
$$

This ordering is not assumed. It is tested explicitly.

If `Q_segmentB_start_Ah` is numerically smaller than `Q_DC,Vmax_Ah`, the ordering status is valid.

If `Q_segmentB_start_Ah` is numerically equal to `Q_DC,Vmax_Ah` within the locked degeneracy tolerance, Segment B is classified as degenerate rather than ordering-violated.

The locked tolerance is:

`Q_segmentB_degenerate_tolerance_Ah = 0.001 Ah`

This is a physical audit tolerance of 1 mAh, not merely a floating-point tolerance. It is used to avoid classifying a numerically negligible boundary coincidence as a protocol-ordering violation.

In the `segment_B_degenerate` case, Segment B has no physically meaningful charge interval or only a numerically negligible interval. Therefore, Interpretation A strong support is not available because strong support requires at least one in-window anchor in Segment B.

The verdict path under `segment_B_degenerate` is:

- if Segment-A residual is floor-compatible, the result may enter Interpretation A weak support;
- if in-window anchors are mixed across Segment A and Segment D, the subclass is `non_geometric_A_not_supported_mixed_anchor_distribution`;
- if all in-window anchors lie before the boundary, the subclass is `geometry_only_anchors_pre_boundary`;
- if in-window anchors lie in Segment D and late-CV preservation is satisfied, the subclass is `late_CV_preservation_consistent`;
- if the residual is intermediate or above the reopen threshold, the ordinary ambiguous / reopened rules apply.

The verdict must carry the caveat:

`segment_B_degenerate`

This degeneracy classification is sensitive to the selected charge-domain tolerance. Day21A locks the tolerance at 1 mAh to prevent post hoc switching between `segment_B_degenerate` and `ordering_violated`.

If `Q_segmentB_start_Ah` is greater than `Q_DC,Vmax_Ah` beyond the degeneracy tolerance, the notebook must report:

- `Q_Vmax_ordering_status = ordering_violated`
- `segment_framework_status = ordering_violated`
- `mechanism_verdict = not_applicable_without_redefinition`

The notebook must not crash before writing the diagnostic information.

---

### Q80/Q90 anchor definitions

Both nominal and common-capacity anchors are reported.

Nominal anchors:

$$
Q80_{\mathrm{nominal}}=0.80\times3.4\,\mathrm{Ah}=2.72\,\mathrm{Ah}
$$

$$
Q90_{\mathrm{nominal}}=0.90\times3.4\,\mathrm{Ah}=3.06\,\mathrm{Ah}
$$

Common-capacity anchors:

$$
Q80_{\mathrm{common}}=0.80\times\min(Q_{\mathrm{final,DC}},Q_{\mathrm{final,DCAC}})
$$

$$
Q90_{\mathrm{common}}=0.90\times\min(Q_{\mathrm{final,DC}},Q_{\mathrm{final,DCAC}})
$$

The label `Q80` must not be used without specifying whether it refers to the nominal or common-capacity anchor.

For cross-source comparison, nominal and common anchors must be normalized separately. The unified verdict table therefore reports four anchor fractions:

- `Q80_nominal_fraction_of_Q_nom`
- `Q90_nominal_fraction_of_Q_nom`
- `Q80_common_fraction_of_Q_nom`
- `Q90_common_fraction_of_Q_nom`

Nominal fractions are fixed by definition:

$$
Q80_{\mathrm{nominal,fraction}}=0.80
$$

$$
Q90_{\mathrm{nominal,fraction}}=0.90
$$

Common fractions depend on the shared final charge window:

$$
Q80_{\mathrm{common,fraction}}=\frac{Q80_{\mathrm{common}}}{Q_{\mathrm{nom}}}
$$

$$
Q90_{\mathrm{common,fraction}}=\frac{Q90_{\mathrm{common}}}{Q_{\mathrm{nom}}}
$$

Common anchor fractions may be smaller than the nominal fractions if the final charge is below `Q_nom_Ah`.

---

### Final-Q consistency

The notebook must report:

- `Q_final_DC_Ah`
- `Q_final_DCAC_Ah`
- `Q_final_diff_Ah`
- `Q_final_diff_status`

The pre-registered consistency threshold is:

$$
\left|Q_{\mathrm{final,DC}}-Q_{\mathrm{final,DCAC}}\right|\le 0.010\,\mathrm{Ah}
$$

If the difference is at most 10 mAh, the status is `final_Q_consistent`; otherwise, it is `final_Q_mismatch_warning`.

If `Q_final_diff_status = final_Q_mismatch_warning`, common anchors remain computable because they are defined within the shared reachable charge window. However, common anchors are valid only as shared reachable absolute-Q targets. They must not be interpreted as equal final-SOC fractions of both trajectories.

The verdict must carry the caveat:

`asymmetric_final_Q`

Any verdict based on common anchors under this warning is conservatively downgraded by one evidence level. No automatic exception is applied in Day21A.

---

### Experimental Segment-A residual floor

The MJ1 experimental residual floor is not a statistical noise floor. It is a single-condition experimental residual floor estimate.

Locked values:

- `MJ1_FLOOR_MAX_ABS_S = 1.35 s`
- `MJ1_FLOOR_MEAN_S ≈ -0.17 s`
- `MJ1_FLOOR_N = 1`
- replicate standard deviation is not available

The PyBaMM Segment-A numerical-null floor and the MJ1 experimental-floor estimate must remain distinguished.

PyBaMM numerical-null floor:

$$
\max |\Delta t_{\mathrm{resid}}| \approx 0.014\text{–}0.044\,\mathrm{s}
$$

MJ1 experimental-floor estimate:

$$
\max |\Delta t_{\mathrm{resid}}| \approx 1.35\,\mathrm{s}
$$

---

### Floor uncertainty propagation

The decision thresholds derived from the MJ1 floor are heuristic audit thresholds, not statistical confidence bounds.

```text
floor_uncertainty_propagation = {
  "method": "single_trial_max_inflation_by_factor_N",
  "no_replicate_SD_available": true,
  "threshold_inflation_is_heuristic_not_statistical": true,
  "known_conservative_bias": true
}
```

The threshold inflation is intentionally conservative. It protects against over-interpreting a single residual excursion as evidence for non-geometric Segment-A acceleration.

---

### Late-CV preservation threshold

The late-CV preservation threshold is defined as an independent operational threshold:

`late_CV_preservation_threshold_s = 2.70 s`

This value is numerically matched to the Segment-A floor-compatible threshold for cross-section consistency, but it is logically independent. Segment A uses this value to classify residual magnitude, whereas Segment D uses it only to test whether raw first-passage gain remains positive above a conservative operational margin.

---

### Pre-registered decision tree

Thresholds:

$$
\mathrm{floor\ compatible}=2\times1.35=2.70\,\mathrm{s}
$$

$$
\mathrm{reopen}=5\times1.35=6.75\,\mathrm{s}
$$

#### Interpretation A strong support

Condition:

- Segment-A residual is floor-compatible:

$$
\max|\Delta t_{\mathrm{resid,SegmentA}}|\le 2.70\,\mathrm{s}
$$

- No in-window anchor lies in Segment A.
- At least one in-window anchor lies in Segment B.
- All in-window anchors lie in Segment B or Segment D.
- If any anchor lies in Segment D, late-CV preservation must be explicitly satisfied.

Late-CV preservation is satisfied only if:

$$
\Delta t_{\mathrm{raw}}(Q_{\mathrm{D-anchor}})>2.70\,\mathrm{s}
$$

for every Segment-D anchor, and:

$$
\mathrm{median}(\Delta t_{\mathrm{raw,SegmentD}})>2.70\,\mathrm{s}
$$

The 2.70 s threshold used for late-CV preservation is an operational consistency threshold. It is not a separately estimated Segment-D noise floor.

When strong support is reached:

MJ1 aligns with a boundary/control-state mediated first-passage gain interpretation, with late-CV preservation where Segment-D anchors are involved. Segment-A non-geometric acceleration is not supported.

If Segment-D anchors are present but `late_CV_preservation_satisfied` cannot be resolved because one or more Segment-D anchor gains or Segment-D median statistics are unavailable, the result is not allowed to enter Interpretation A strong support.

In that case:

- `evidence_status = ambiguous`
- `interpretation_class = late_CV_preservation_unresolved`
- `caveat = late_CV_preservation_unresolved`

Meaning:

MJ1 aligns with a boundary/control-state mediated first-passage gain interpretation, with late-CV preservation where Segment-D anchors are involved. Segment-A non-geometric acceleration is not supported.

If some nominal anchors are outside the common Q-window but all common anchors remain in-window and satisfy the above conditions, the verdict remains eligible for strong support but must carry the caveat:

`nominal_anchor_outside_common_Q_window`

#### Interpretation A weak support

Condition:

- Segment-A residual is floor-compatible:

$$
\max|\Delta t_{\mathrm{resid,SegmentA}}|\le 2.70\,\mathrm{s}
$$

- The anchor distribution does not satisfy the strong-support pattern.
- No in-window Segment-A anchor satisfies the reopen threshold.

Meaning:

Non-geometric Segment-A acceleration is not supported. The observed gain is compatible with current-geometry contribution, boundary/control-state split, and/or late-CV preservation, depending on where the anchors fall.

Possible interpretation subclasses:

- `geometry_only_anchors_pre_boundary`: all in-window anchors lie in Segment A and Segment-A residual is floor-compatible.
- `non_geometric_A_not_supported_mixed_anchor_distribution`: anchors are distributed across Segment A, Segment B, Segment D, or outside the common Q-window.
- `late_CV_preservation_consistent`: in-window anchors lie in Segment D and raw Segment-D gains remain positive above the late-CV preservation threshold.

Evidence status:

`partial_support`

#### Interpretation B reopened

Condition:

At least one in-window anchor lies in Segment A:

- `Q80_common`
- `Q90_common`
- `Q80_nominal`
- `Q90_nominal`

and:

$$
\max |\Delta t_{\mathrm{resid,SegmentA}}| \ge 5\times MJ1\_FLOOR\_MAX\_ABS\_S
$$

that is:

$$
\max |\Delta t_{\mathrm{resid,SegmentA}}| \ge 6.75\,\mathrm{s}
$$

Meaning:

Possible above-floor non-geometric Segment-A acceleration beyond the current PyBaMM-DFN response basin.

#### Intermediate / ambiguous zone

If:

$$
2.70\,\mathrm{s}<\max|\Delta t_{\mathrm{resid,SegmentA}}|<6.75\,\mathrm{s}
$$

then the result is classified as:

`ambiguous_defer_mechanism_commitment`

The residual exceeds the floor-compatible range but does not satisfy the pre-registered reopen threshold.

#### Spike guard

If the maximum residual exceeds the reopen threshold:

$$
6.75\,\mathrm{s}
$$

but the 95th percentile does not, the result is classified as:

`spike_or_transition_artifact`

This is reported as a sub-class of ambiguous:

- `evidence_status = ambiguous`
- `interpretation_class = spike_or_transition_artifact`

It must not directly reopen Interpretation B.

---

### Required output fields for traceability

The unified verdict table must persist the following event-detection, phase, segmentation, anchor, and thermal fields:

- `phase_convention`
- `Vmax_detection_method_used`
- `AC_off_detection_method_used`
- `t_AC_off_DC_s`
- `t_AC_off_DCAC_s`
- `geometry_phase_offset_s`
- `geometry_phase_offset_rad`
- `geometry_phase_reference_status`
- `t_segmentB_start_s`
- `Q_segmentB_start_Ah`
- `segment_A_Q_hi_definition`
- `Q80_nominal_fraction_of_Q_nom`
- `Q90_nominal_fraction_of_Q_nom`
- `Q80_common_fraction_of_Q_nom`
- `Q90_common_fraction_of_Q_nom`
- `T_surface_max_C`
- `T_surface_mean_C`
- `temperature_sensor_type`
- `temperature_sensor_placement`
- `temperature_data_source`
- `temperature_alignment_method`
- `temperature_to_NGU201_alignment_required`

For pure DC reference data, `t_AC_off_DC_s = NaN`, because no AC-off event exists in pure DC charging.

---

### Provenance requirements

The file inventory must include:

- `source_type`
- `cell_id`
- `voltage_source`
- `current_source`
- `sampling_rate_Hz`
- `ambient_temperature_C`
- `temperature_control_type`
- `temperature_sensor_type`
- `temperature_sensor_placement`
- `temperature_data_source`
- `temperature_alignment_method`
- `temperature_to_NGU201_alignment_required`
- `T_surface_max_C`
- `T_surface_mean_C`
- `timebase_source`
- `time_alignment_method`
- `voltage_current_alignment_status`
- `phase_convention`
- `candidate_for_DC_reference`
- `candidate_for_DCAC`

`candidate_for_DC_reference` and `candidate_for_DCAC` must be mutually exclusive for a single file. If both are true, the file must be flagged for manual review before being used.

For unknown metadata strings, use:

`unknown_not_recorded`

For non-applicable numeric fields, use:

`NaN`

No empty provenance field is allowed.

---

### Unified MJ1–PyBaMM alignment

The Day21A verdict table is designed to align with the Day20B PyBaMM full-protocol segmentation outputs, including Segment A, Segment B, Segment D, voltage-boundary charge, and full-protocol first-passage gain fields.

When merged with PyBaMM historical or Day20B outputs, phase convention must remain explicit. Discharge-first historical PyBaMM cases from Day19A must not be silently merged with charge-first MJ1 experimental evidence.

For cross-source comparison between MJ1 and PyBaMM parameter sets, absolute Ah values are not directly comparable because the nominal capacities differ between the experimental MJ1 cell and the PyBaMM parameter sets. Therefore, the unified verdict table must use anchor fractions and segment assignment, not raw absolute Ah values alone.

Nominal and common anchors must not be collapsed into a single generic fraction field. The following four fields are required:

- `Q80_nominal_fraction_of_Q_nom`
- `Q90_nominal_fraction_of_Q_nom`
- `Q80_common_fraction_of_Q_nom`
- `Q90_common_fraction_of_Q_nom`

Mechanism-level comparisons must use the relevant anchor fraction and segment assignment together.

---

### Required outputs

The Day21A output must include:

1. MJ1 file provenance inventory.
2. Event / AC-off detection audit.
3. Strict-net Q integration audit.
4. Final-Q consistency audit.
5. Segment A/B/D/outside assignment for Q80/Q90 nominal and common anchors.
6. Segment-A residual floor test.
7. MJ1 mechanism verdict.
8. Unified MJ1–PyBaMM mechanism verdict table.

The unified verdict table schema is frozen in Cell 1 and must not be extended ad hoc during data scanning.

In [1]:
# Cell 1 v4.1 — setup, constants, thresholds, schemas, and decision helpers
# Day21A: MJ1 experimental full-protocol segmentation audit
#
# Contract rule:
# - No data scanning in this cell.
# - No MJ1 file loading in this cell.
# - No runtime threshold tuning in this cell.
# - This cell freezes constants, schemas, helper functions, and audit-contract metadata.

from __future__ import annotations

from pathlib import Path
from datetime import datetime, timezone
from typing import Any, Dict, Iterable, List, Optional, Sequence
import json
import subprocess

import numpy as np
import pandas as pd


# =============================================================================
# 1. Paths and run metadata
# =============================================================================

REPO = Path("/Users/louislu/pybamm-dcac-superimposed").expanduser()
DATA_DIR = REPO / "data"
NOTEBOOK_NAME = "25_day21A_MJ1_experimental_segment_audit.ipynb"

OUT_FILE_INVENTORY = DATA_DIR / "day21A_step0_MJ1_file_inventory.csv"
OUT_UNIFIED_VERDICT = DATA_DIR / "day21A_step2_unified_MJ1_PyBaMM_mechanism_verdict.csv"
OUT_AUDIT_CONTRACT_JSON = DATA_DIR / "day21A_audit_contract_schema_thresholds.json"

DATA_DIR.mkdir(parents=True, exist_ok=True)

RUN_TIMESTAMP_UTC = datetime.now(timezone.utc).isoformat()


def get_git_head(repo: Path) -> str:
    """Return current git HEAD. Does not fail the notebook if git metadata is unavailable."""
    try:
        result = subprocess.run(
            ["git", "rev-parse", "HEAD"],
            cwd=repo,
            capture_output=True,
            text=True,
            check=True,
        )
        return result.stdout.strip()
    except Exception as exc:
        return f"unknown_not_recorded:{type(exc).__name__}"


GIT_HEAD = get_git_head(REPO)


# =============================================================================
# 2. Locked constants and conventions
# =============================================================================

UNKNOWN = "unknown_not_recorded"

SOURCE_TYPE_MJ1 = "experimental_MJ1"
CELL_ID = "LG_INR18650_MJ1"
CHEMISTRY_FAMILY = "NMC_layered_oxide"

# Project-level analysis capacity.
# Origin: standardized 1C / 0.5C pre-DC–AC characterization of the same MJ1 cell.
Q_NOM_AH = 3.4
ONE_C_A = 3.4

VMAX_V = 4.2

# Important: these two constants have the same numerical value but different semantics.
# I_CUTOFF_A: CV termination threshold at the end of charge.
# I_CHARGE_ONSET_THRESHOLD_A: onset threshold only for trimming rest/preconditioning.
I_CUTOFF_A = 0.05
I_CUTOFF_DEFINITION = "absolute_50mA_first_reached"

I_CHARGE_ONSET_THRESHOLD_A = 0.05
I_CHARGE_ONSET_MIN_CONSECUTIVE_SAMPLES = 3

PHASE_CONVENTION = "charge_first"

# Environment and temperature measurement convention
AMBIENT_TEMPERATURE_C = 20.0
TEMPERATURE_CONTROL_TYPE = "ambient_lab_no_chamber"
TEMPERATURE_SENSOR_TYPE = "Pt100"
TEMPERATURE_SENSOR_PLACEMENT = "axial_cell_surface"
TEMPERATURE_DATA_SOURCE = "measured_surface_temperature"
TEMPERATURE_ALIGNMENT_METHOD = "segment_level_summary_only"
TEMPERATURE_TO_NGU201_ALIGNMENT_REQUIRED = False

# Measurement source defaults for Day21A.
# Inventory still stores these as per-record fields for traceability.
DEFAULT_VOLTAGE_SOURCE = "NGU201"
DEFAULT_CURRENT_SOURCE = "NGU201"
DEFAULT_TIMEBASE_SOURCE = "NGU201_single_timebase"
DEFAULT_TIME_ALIGNMENT_METHOD = "native_same_device_timebase"
DEFAULT_VOLTAGE_CURRENT_ALIGNMENT_STATUS = "native_aligned_same_record"

# Anchor fractions
Q80_NOMINAL_FRACTION_OF_Q_NOM = 0.80
Q90_NOMINAL_FRACTION_OF_Q_NOM = 0.90

Q80_NOMINAL_AH = Q80_NOMINAL_FRACTION_OF_Q_NOM * Q_NOM_AH
Q90_NOMINAL_AH = Q90_NOMINAL_FRACTION_OF_Q_NOM * Q_NOM_AH

FINAL_Q_DIFF_THRESHOLD_AH = 0.010  # 10 mAh


# =============================================================================
# 3. Q-grid, Vmax, AC-off, late-CV, and residual-floor constants
# =============================================================================

# Segment-A residual grid
SEGMENT_A_Q_LO_AH = 0.050
Q_GRID_STEP_AH = 0.010
Q_GRID_DEFINITION_METHOD = "fixed_10mAh_grid_within_segment_A"
Q_GRID_MIN_COUNT_SEGMENT_A = 30

# Segment-B degeneracy
Q_SEGMENTB_DEGENERATE_TOLERANCE_AH = 0.001  # 1 mAh physical audit tolerance

# Geometry phase estimation
GEOMETRY_PHASE_FIT_N_T_AC = 5
GEOMETRY_PHASE_MIN_REQUIRED_CYCLES = 1

# AC-off detection — default rule
AC_OFF_ENVELOPE_TOLERANCE_COEFFICIENT = 0.05
AC_OFF_PERSISTENCE_N_T_AC = 3

# AC-off low-frequency exception
AC_OFF_LOW_FREQUENCY_VOLTAGE_WINDOW_V = [4.195, 4.205]
AC_OFF_LOW_FREQUENCY_MIN_PERSISTENCE_S = 120.0
AC_OFF_LOW_FREQUENCY_ENVELOPE_TOLERANCE_COEFFICIENT = 0.05
AC_OFF_LOW_FREQUENCY_REQUIRED_TREND = (
    "monotonic_CV_current_decay_or_pure_DC_continuation"
)

# Vmax detection / voltage deglitch
VMAX_DETECTION_PRIORITY = [
    "actual_cycler_or_protocol_transition_timestamp_if_available",
    "detected_AC_off_or_CV_transition_from_current_waveform",
    "first_V_ge_4p2V_with_deglitch_rule",
]

DEGLITCH_APPLIES_TO_PRIORITIES = [3]
DEGLITCH_MIN_CONSECUTIVE_SAMPLES = 3
DEGLITCH_NEAR_VMAX_TOL_V = 0.002

# Experimental Segment-A residual floor
MJ1_FLOOR_MAX_ABS_S = 1.35
MJ1_FLOOR_MEAN_S = -0.17
MJ1_FLOOR_N = 1
MJ1_FLOOR_TYPE = "single_condition_experimental_residual_floor_estimate"

N_ALLOW = 2
M_REOPEN = 5

SEG_A_FLOOR_COMPATIBLE_THRESHOLD_S = N_ALLOW * MJ1_FLOOR_MAX_ABS_S  # 2.70 s
SEG_A_REOPEN_THRESHOLD_S = M_REOPEN * MJ1_FLOOR_MAX_ABS_S           # 6.75 s

# Logically independent from Segment-A residual threshold, but numerically matched.
LATE_CV_PRESERVATION_THRESHOLD_S = 2.70

PYBAMM_NUMERICAL_NULL_MAX_ABS_RANGE_S = [0.014, 0.044]


# =============================================================================
# 4. Frequency / period helpers
# =============================================================================

TAU_LABEL_S = 11.1  # experimental label reference; actual frequency_Hz is used when available


def compute_tau_eff_s(tau_label_s: float, m_tau: float) -> float:
    """tau_eff = m_tau * tau_label."""
    if not np.isfinite(tau_label_s) or not np.isfinite(m_tau):
        return np.nan
    return float(m_tau * tau_label_s)


def compute_frequency_hz_from_tau_eff(tau_eff_s: float) -> float:
    """f_Hz = 1 / (2*pi*tau_eff)."""
    if not np.isfinite(tau_eff_s) or tau_eff_s <= 0:
        return np.nan
    return float(1.0 / (2.0 * np.pi * tau_eff_s))


def compute_t_ac_s(frequency_hz: float) -> float:
    """T_AC = 1 / frequency_Hz."""
    if not np.isfinite(frequency_hz) or frequency_hz <= 0:
        return np.nan
    return float(1.0 / frequency_hz)


def frequency_period_example(tau_label_s: float, m_tau: float) -> Dict[str, float]:
    """Compute example tau_eff, frequency_Hz, T_AC, and default AC-off persistence."""
    tau_eff_s = compute_tau_eff_s(tau_label_s=tau_label_s, m_tau=m_tau)
    frequency_hz = compute_frequency_hz_from_tau_eff(tau_eff_s=tau_eff_s)
    t_ac_s = compute_t_ac_s(frequency_hz=frequency_hz)
    return {
        "m_tau": float(m_tau),
        "tau_eff_s": tau_eff_s,
        "frequency_Hz": frequency_hz,
        "T_AC_s": t_ac_s,
        "AC_off_persistence_s_default": AC_OFF_PERSISTENCE_N_T_AC * t_ac_s
        if np.isfinite(t_ac_s)
        else np.nan,
    }


# =============================================================================
# 5. Segment, framework, evidence, and verdict status labels
# =============================================================================

SEGMENT_A = "A_shared_prescribed_current"
SEGMENT_B = "B_voltage_boundary_control_state_split"
SEGMENT_D = "D_late_CV_feedback"
SEGMENT_OUTSIDE = "outside_common_Q_window"
SEGMENT_UNRESOLVED = "segment_unresolved"

SEGMENT_FRAMEWORK_OK = "segment_framework_ok"
SEGMENT_FRAMEWORK_ORDERING_VIOLATED = "ordering_violated"
SEGMENT_FRAMEWORK_AC_OFF_PRECEDES_VMAX = "AC_off_precedes_Vmax"
SEGMENT_FRAMEWORK_GEOMETRY_PHASE_UNRESOLVED = "geometry_phase_unresolved"
SEGMENT_FRAMEWORK_SEGMENT_B_DEGENERATE = "segment_B_degenerate"
SEGMENT_FRAMEWORK_UNRESOLVED = "segment_framework_unresolved"

ORDERING_EXPECTED = "expected_Q_segmentB_start_lt_Q_DC_Vmax"
ORDERING_DEGENERATE = "segment_B_degenerate_Q_segmentB_start_eq_Q_DC_Vmax"
ORDERING_VIOLATED = "ordering_violated"
ORDERING_UNRESOLVED = "ordering_unresolved"

FINAL_Q_CONSISTENT = "final_Q_consistent"
FINAL_Q_MISMATCH_WARNING = "final_Q_mismatch_warning"
FINAL_Q_UNRESOLVED = "final_Q_unresolved"

GEOM_PHASE_VERIFIED = "verified"
GEOM_PHASE_ESTIMATED = "estimated_from_current_waveform"
GEOM_PHASE_UNRESOLVED = "unresolved"

ABOVE_FLOOR_NO = "floor_compatible"
ABOVE_FLOOR_INTERMEDIATE = "intermediate_between_floor_and_reopen_threshold"
ABOVE_FLOOR_YES = "above_floor"
ABOVE_FLOOR_YES_NO_P95_GUARD = "above_floor_no_p95_guard"
ABOVE_FLOOR_SPIKE = "spike_or_transition_artifact"
ABOVE_FLOOR_UNRESOLVED = "above_floor_unresolved"

LATE_CV_SATISFIED = "satisfied"
LATE_CV_NOT_SATISFIED = "not_satisfied"
LATE_CV_NOT_REQUIRED = "not_required_no_segmentD_anchor"
LATE_CV_UNRESOLVED = "unresolved"

EVIDENCE_SUPPORTED = "supported"
EVIDENCE_SUPPORT_WITH_WARNING = "support_with_warning"
EVIDENCE_PARTIAL_SUPPORT = "partial_support"
EVIDENCE_PARTIAL_SUPPORT_WITH_WARNING = "partial_support_with_warning"
EVIDENCE_REOPENED = "reopened"
EVIDENCE_REOPENED_WITH_WARNING = "reopened_with_warning"
EVIDENCE_AMBIGUOUS = "ambiguous"
EVIDENCE_NOT_APPLICABLE = "not_applicable"

VERDICT_A_STRONG = (
    "interpretation_A_strong_boundary_control_state_late_CV_preservation"
)
VERDICT_A_WEAK = "interpretation_A_weak_non_geometric_A_not_supported"
VERDICT_B_REOPENED = (
    "interpretation_B_reopened_possible_non_geometric_segment_A"
)
VERDICT_AMBIG = "ambiguous_defer_mechanism_commitment"
VERDICT_NOT_APPLICABLE = "not_applicable"

INTERPRET_A_STRONG_CLASS = (
    "boundary_control_state_mediated_first_passage_gain_with_late_CV_preservation"
)
INTERPRET_GEOMETRY_ONLY_A = "geometry_only_anchors_pre_boundary"
INTERPRET_MIXED_A_NOT_SUPPORTED = (
    "non_geometric_A_not_supported_mixed_anchor_distribution"
)
INTERPRET_LATE_CV_CONSISTENT = "late_CV_preservation_consistent"
INTERPRET_LATE_CV_UNRESOLVED = "late_CV_preservation_unresolved"
INTERPRET_B_CLASS = "possible_non_geometric_segment_A_acceleration"
INTERPRET_SPIKE_CLASS = "spike_or_transition_artifact"
INTERPRET_INTERMEDIATE_CLASS = "intermediate_residual_defer_mechanism_commitment"


# =============================================================================
# 6. Allowed status sets for strict validation
# =============================================================================

ALLOWED_Q_SEGMENTS = {
    SEGMENT_A,
    SEGMENT_B,
    SEGMENT_D,
    SEGMENT_OUTSIDE,
    SEGMENT_UNRESOLVED,
}

ALLOWED_ORDERING_STATUS = {
    ORDERING_EXPECTED,
    ORDERING_DEGENERATE,
    ORDERING_VIOLATED,
    ORDERING_UNRESOLVED,
}

ALLOWED_SEGMENT_FRAMEWORK_STATUS = {
    SEGMENT_FRAMEWORK_OK,
    SEGMENT_FRAMEWORK_ORDERING_VIOLATED,
    SEGMENT_FRAMEWORK_AC_OFF_PRECEDES_VMAX,
    SEGMENT_FRAMEWORK_GEOMETRY_PHASE_UNRESOLVED,
    SEGMENT_FRAMEWORK_SEGMENT_B_DEGENERATE,
    SEGMENT_FRAMEWORK_UNRESOLVED,
}

ALLOWED_GEOMETRY_PHASE_STATUS = {
    GEOM_PHASE_VERIFIED,
    GEOM_PHASE_ESTIMATED,
    GEOM_PHASE_UNRESOLVED,
}

ALLOWED_FINAL_Q_STATUS = {
    FINAL_Q_CONSISTENT,
    FINAL_Q_MISMATCH_WARNING,
    FINAL_Q_UNRESOLVED,
}

ALLOWED_LATE_CV_STATUS = {
    LATE_CV_SATISFIED,
    LATE_CV_NOT_SATISFIED,
    LATE_CV_NOT_REQUIRED,
    LATE_CV_UNRESOLVED,
}


# =============================================================================
# 7. Caveat constants
# =============================================================================

CAVEAT_ASYMMETRIC_FINAL_Q = "asymmetric_final_Q"
CAVEAT_NOMINAL_ANCHOR_OUTSIDE = "nominal_anchor_outside_common_Q_window"
CAVEAT_ALL_ANCHORS_OUTSIDE = "all_anchors_outside_common_Q_window"
CAVEAT_SEGMENT_A_GRID_LOW = "segment_A_grid_count_below_p95_requirement"
CAVEAT_GEOMETRY_PHASE_UNRESOLVED = "geometry_phase_reference_unresolved"
CAVEAT_EVENT_REAUDIT_REQUIRED = "event_reaudit_required"
CAVEAT_ORDERING_VIOLATED = "segment_ordering_violated"
CAVEAT_SEGMENT_B_DEGENERATE = "segment_B_degenerate"
CAVEAT_P95_NOT_AVAILABLE = "p95_not_available_spike_guard_not_applied"
CAVEAT_LATE_CV_PRESERVATION_UNRESOLVED = "late_CV_preservation_unresolved"
CAVEAT_SEGMENT_A_ABOVE_FLOOR_NO_A_ANCHOR = (
    "segment_A_above_floor_without_inwindow_A_anchor"
)


# =============================================================================
# 8. Frozen schema: MJ1 file provenance inventory
# =============================================================================

INVENTORY_SCHEMA: List[str] = [
    "file_path",
    "file_name",
    "file_mtime",
    "file_size_kb",
    "read_ok",

    "source_type",
    "cell_id",
    "source_session_date",

    "protocol_label",
    "protocol_role",
    "DC_C",
    "AC_C",
    "frequency_Hz",
    "m_tau",
    "tau_label_s",
    "tau_eff_s",
    "phase_convention",

    "Vmax_V",
    "I_cutoff_A",
    "I_charge_onset_threshold_A",
    "Q_nom_Ah",

    "voltage_source",
    "current_source",
    "sampling_rate_Hz",
    "ambient_temperature_C",
    "temperature_control_type",
    "temperature_sensor_type",
    "temperature_sensor_placement",
    "temperature_data_source",
    "temperature_alignment_method",
    "temperature_to_NGU201_alignment_required",
    "T_surface_max_C",
    "T_surface_mean_C",

    "timebase_source",
    "time_alignment_method",
    "voltage_current_alignment_status",

    "has_time",
    "has_voltage",
    "has_current",
    "has_temperature",
    "has_stage_marker",
    "has_AC_off_marker",

    "candidate_for_DC_reference",
    "candidate_for_DCAC",
    "notes",
]


# =============================================================================
# 9. Frozen schema: unified MJ1–PyBaMM mechanism verdict table
# =============================================================================

UNIFIED_VERDICT_SCHEMA: List[str] = [
    # Source / protocol identity
    "source_type",
    "source_id",
    "cell_or_param_set",
    "chemistry_family",
    "protocol_pair",
    "protocol_label_DC",
    "protocol_label_DCAC",
    "phase_convention",

    # Measurement / provenance fields
    "voltage_source",
    "current_source",
    "sampling_rate_Hz",
    "ambient_temperature_C",
    "temperature_control_type",
    "temperature_sensor_type",
    "temperature_sensor_placement",
    "temperature_data_source",
    "temperature_alignment_method",
    "temperature_to_NGU201_alignment_required",
    "T_surface_max_C",
    "T_surface_mean_C",
    "timebase_source",
    "time_alignment_method",
    "voltage_current_alignment_status",

    # Capacity and final-Q consistency
    "Q_nom_Ah",
    "Q80_nominal_fraction_of_Q_nom",
    "Q90_nominal_fraction_of_Q_nom",
    "Q80_common_fraction_of_Q_nom",
    "Q90_common_fraction_of_Q_nom",
    "Q_final_DC_Ah",
    "Q_final_DCAC_Ah",
    "Q_final_diff_Ah",
    "Q_final_diff_status",

    # Protocol constants
    "Vmax_V",
    "I_cutoff_A",
    "I_cutoff_definition",
    "I_charge_onset_threshold_A",

    # Event detection methods and times
    "Vmax_detection_method_used",
    "AC_off_detection_method_used",
    "t_Vmax_DC_s",
    "t_Vmax_DCAC_s",
    "t_AC_off_DC_s",
    "t_AC_off_DCAC_s",
    "AC_off_lag_s",
    "t_segmentB_start_s",

    # Voltage-boundary charges and segment boundary
    "Q_Vmax_DC_Ah",
    "Q_Vmax_DCAC_Ah",
    "Q_segmentB_start_Ah",
    "segment_A_Q_hi_definition",
    "Q_Vmax_shift_Ah",
    "Q_Vmax_ordering_status",
    "segment_framework_status",

    # Geometry phase reference
    "geometry_phase_offset_s",
    "geometry_phase_offset_rad",
    "geometry_phase_reference_status",

    # Anchor definitions
    "Q80_nominal_Ah",
    "Q90_nominal_Ah",
    "Q80_common_Ah",
    "Q90_common_Ah",

    # Anchor segment assignment
    "Q80_nominal_segment",
    "Q90_nominal_segment",
    "Q80_common_segment",
    "Q90_common_segment",

    # Anchor raw time gains
    "dt_Q80_nominal_raw_s",
    "dt_Q90_nominal_raw_s",
    "dt_Q80_common_raw_s",
    "dt_Q90_common_raw_s",

    # Segment A statistics
    "segment_A_Q_lo_Ah",
    "segment_A_Q_hi_Ah",
    "segment_A_Q_grid_count",
    "segment_A_dt_raw_median_s",
    "segment_A_dt_raw_max_s",
    "segment_A_dt_resid_mean_s",
    "segment_A_dt_resid_max_abs_s",
    "segment_A_dt_resid_p95_abs_s",
    "segment_A_resid_floor_s",
    "segment_A_resid_floor_type",
    "segment_A_resid_floor_n",
    "segment_A_above_floor_status",

    # Segment B statistics
    "segment_B_Q_lo_Ah",
    "segment_B_Q_hi_Ah",
    "segment_B_dt_raw_median_s",
    "segment_B_dt_raw_max_s",

    # Segment D statistics and late-CV preservation
    "segment_D_Q_lo_Ah",
    "segment_D_Q_hi_Ah",
    "segment_D_dt_raw_min_s",
    "segment_D_dt_raw_median_s",
    "segment_D_dt_raw_max_s",
    "late_CV_preservation_threshold_s",
    "late_CV_preservation_satisfied",

    # Verdict
    "evidence_status",
    "mechanism_verdict",
    "interpretation_class",
    "caveat",
]


# =============================================================================
# 10. Numeric columns for strict fill/serialization
# =============================================================================

INVENTORY_NUMERIC_COLUMNS: List[str] = [
    "file_size_kb",
    "DC_C",
    "AC_C",
    "frequency_Hz",
    "m_tau",
    "tau_label_s",
    "tau_eff_s",
    "Vmax_V",
    "I_cutoff_A",
    "I_charge_onset_threshold_A",
    "Q_nom_Ah",
    "sampling_rate_Hz",
    "ambient_temperature_C",
    "T_surface_max_C",
    "T_surface_mean_C",
]

UNIFIED_VERDICT_NUMERIC_COLUMNS: List[str] = [
    "sampling_rate_Hz",
    "ambient_temperature_C",
    "T_surface_max_C",
    "T_surface_mean_C",

    "Q_nom_Ah",
    "Q80_nominal_fraction_of_Q_nom",
    "Q90_nominal_fraction_of_Q_nom",
    "Q80_common_fraction_of_Q_nom",
    "Q90_common_fraction_of_Q_nom",
    "Q_final_DC_Ah",
    "Q_final_DCAC_Ah",
    "Q_final_diff_Ah",

    "Vmax_V",
    "I_cutoff_A",
    "I_charge_onset_threshold_A",

    "t_Vmax_DC_s",
    "t_Vmax_DCAC_s",
    "t_AC_off_DC_s",
    "t_AC_off_DCAC_s",
    "AC_off_lag_s",
    "t_segmentB_start_s",

    "Q_Vmax_DC_Ah",
    "Q_Vmax_DCAC_Ah",
    "Q_segmentB_start_Ah",
    "Q_Vmax_shift_Ah",

    "geometry_phase_offset_s",
    "geometry_phase_offset_rad",

    "Q80_nominal_Ah",
    "Q90_nominal_Ah",
    "Q80_common_Ah",
    "Q90_common_Ah",

    "dt_Q80_nominal_raw_s",
    "dt_Q90_nominal_raw_s",
    "dt_Q80_common_raw_s",
    "dt_Q90_common_raw_s",

    "segment_A_Q_lo_Ah",
    "segment_A_Q_hi_Ah",
    "segment_A_Q_grid_count",
    "segment_A_dt_raw_median_s",
    "segment_A_dt_raw_max_s",
    "segment_A_dt_resid_mean_s",
    "segment_A_dt_resid_max_abs_s",
    "segment_A_dt_resid_p95_abs_s",
    "segment_A_resid_floor_s",
    "segment_A_resid_floor_n",

    "segment_B_Q_lo_Ah",
    "segment_B_Q_hi_Ah",
    "segment_B_dt_raw_median_s",
    "segment_B_dt_raw_max_s",

    "segment_D_Q_lo_Ah",
    "segment_D_Q_hi_Ah",
    "segment_D_dt_raw_min_s",
    "segment_D_dt_raw_median_s",
    "segment_D_dt_raw_max_s",
    "late_CV_preservation_threshold_s",
]


# =============================================================================
# 11. Schema guards
# =============================================================================

def assert_unique_columns(schema: Iterable[str], schema_name: str) -> None:
    cols = list(schema)
    duplicates = sorted({c for c in cols if cols.count(c) > 1})
    if duplicates:
        raise ValueError(f"{schema_name} contains duplicate columns: {duplicates}")


def make_empty_schema_df(schema: Sequence[str]) -> pd.DataFrame:
    return pd.DataFrame(columns=list(schema))


def assert_exact_schema(df: pd.DataFrame, schema: Sequence[str], schema_name: str) -> None:
    actual = list(df.columns)
    expected = list(schema)
    if actual != expected:
        missing = [c for c in expected if c not in actual]
        extra = [c for c in actual if c not in expected]
        wrong_order_only = (not missing and not extra and actual != expected)
        raise ValueError(
            f"{schema_name} schema mismatch.\n"
            f"Missing: {missing}\n"
            f"Extra: {extra}\n"
            f"Wrong order only: {wrong_order_only}"
        )


def assert_numeric_columns_in_schema(
    numeric_columns: Sequence[str],
    schema: Sequence[str],
    schema_name: str,
) -> None:
    missing = [c for c in numeric_columns if c not in schema]
    if missing:
        raise ValueError(
            f"{schema_name} numeric column list contains fields not in schema: {missing}"
        )


assert_unique_columns(INVENTORY_SCHEMA, "INVENTORY_SCHEMA")
assert_unique_columns(UNIFIED_VERDICT_SCHEMA, "UNIFIED_VERDICT_SCHEMA")
assert_numeric_columns_in_schema(INVENTORY_NUMERIC_COLUMNS, INVENTORY_SCHEMA, "INVENTORY_SCHEMA")
assert_numeric_columns_in_schema(
    UNIFIED_VERDICT_NUMERIC_COLUMNS,
    UNIFIED_VERDICT_SCHEMA,
    "UNIFIED_VERDICT_SCHEMA",
)

_empty_inventory = make_empty_schema_df(INVENTORY_SCHEMA)
_empty_verdict = make_empty_schema_df(UNIFIED_VERDICT_SCHEMA)

assert_exact_schema(_empty_inventory, INVENTORY_SCHEMA, "INVENTORY_SCHEMA")
assert_exact_schema(_empty_verdict, UNIFIED_VERDICT_SCHEMA, "UNIFIED_VERDICT_SCHEMA")


# =============================================================================
# 12. Generic helpers
# =============================================================================

def is_finite_number(x: Any) -> bool:
    try:
        return bool(np.isfinite(float(x)))
    except Exception:
        return False


def parse_bool_strict(x: Any) -> bool:
    """
    Strict boolean parser.
    Avoids pandas/object pitfalls where non-empty strings cast to True.
    """
    if isinstance(x, (bool, np.bool_)):
        return bool(x)
    if isinstance(x, (int, np.integer)) and x in [0, 1]:
        return bool(x)
    if isinstance(x, str):
        s = x.strip().lower()
        if s == "true":
            return True
        if s == "false":
            return False
    raise ValueError(f"Cannot parse strict bool from {x!r}")


def append_caveat(existing: Any, new: str) -> str:
    """Append a caveat string without duplicates; semicolon-delimited."""
    if new is None or new == "":
        if existing is None or pd.isna(existing):
            return ""
        return str(existing)

    if existing is None or pd.isna(existing):
        items: List[str] = []
    else:
        existing_str = str(existing).strip()
        items = [] if existing_str in ["", "nan", "NaN", "<NA>"] else [
            x.strip() for x in existing_str.split(";") if x.strip()
        ]

    if new not in items:
        items.append(new)

    return ";".join(items)


def compute_common_anchors(
    q_final_dc_ah: float,
    q_final_dcac_ah: float,
    q80_fraction: float = Q80_NOMINAL_FRACTION_OF_Q_NOM,
    q90_fraction: float = Q90_NOMINAL_FRACTION_OF_Q_NOM,
    q_nom_ah: float = Q_NOM_AH,
) -> Dict[str, float]:
    """
    Compute common reachable Q80/Q90 anchors and their fractions relative to Q_nom.

    Common anchor Ah values:
    Q80_common = 0.80 * min(Q_final_DC, Q_final_DCAC)
    Q90_common = 0.90 * min(Q_final_DC, Q_final_DCAC)

    Common fractions:
    Q80_common_fraction_of_Q_nom = Q80_common / Q_nom
    Q90_common_fraction_of_Q_nom = Q90_common / Q_nom

    Q_common_final_Ah is an internal intermediate and is not part of UNIFIED_VERDICT_SCHEMA.
    """
    if not is_finite_number(q_final_dc_ah) or not is_finite_number(q_final_dcac_ah):
        return {
            "Q_common_final_Ah": np.nan,
            "Q80_common_Ah": np.nan,
            "Q90_common_Ah": np.nan,
            "Q80_common_fraction_of_Q_nom": np.nan,
            "Q90_common_fraction_of_Q_nom": np.nan,
        }

    q_common_final_ah = min(float(q_final_dc_ah), float(q_final_dcac_ah))
    q80_common_ah = q80_fraction * q_common_final_ah
    q90_common_ah = q90_fraction * q_common_final_ah

    if not is_finite_number(q_nom_ah) or float(q_nom_ah) <= 0:
        q80_common_fraction = np.nan
        q90_common_fraction = np.nan
    else:
        q80_common_fraction = q80_common_ah / float(q_nom_ah)
        q90_common_fraction = q90_common_ah / float(q_nom_ah)

    return {
        "Q_common_final_Ah": q_common_final_ah,
        "Q80_common_Ah": q80_common_ah,
        "Q90_common_Ah": q90_common_ah,
        "Q80_common_fraction_of_Q_nom": q80_common_fraction,
        "Q90_common_fraction_of_Q_nom": q90_common_fraction,
    }


def final_q_status(q_final_dc_ah: float, q_final_dcac_ah: float) -> str:
    if not is_finite_number(q_final_dc_ah) or not is_finite_number(q_final_dcac_ah):
        return FINAL_Q_UNRESOLVED

    diff = abs(float(q_final_dc_ah) - float(q_final_dcac_ah))
    return FINAL_Q_CONSISTENT if diff <= FINAL_Q_DIFF_THRESHOLD_AH else FINAL_Q_MISMATCH_WARNING


def q_boundary_ordering_status(
    q_segmentB_start_ah: float,
    q_vmax_dc_ah: float,
    tol_ah: float = Q_SEGMENTB_DEGENERATE_TOLERANCE_AH,
) -> str:
    """
    Expected ordering:
    Q_segmentB_start_Ah < Q_Vmax_DC_Ah

    If equal within tol_ah, classify as Segment-B degenerate rather than violated.
    """
    if not is_finite_number(q_segmentB_start_ah) or not is_finite_number(q_vmax_dc_ah):
        return ORDERING_UNRESOLVED

    q_b = float(q_segmentB_start_ah)
    q_dc = float(q_vmax_dc_ah)

    if q_b < q_dc - tol_ah:
        return ORDERING_EXPECTED

    if abs(q_b - q_dc) <= tol_ah:
        return ORDERING_DEGENERATE

    return ORDERING_VIOLATED


def segment_framework_status_from_ordering(q_ordering_status: str) -> str:
    if q_ordering_status == ORDERING_EXPECTED:
        return SEGMENT_FRAMEWORK_OK
    if q_ordering_status == ORDERING_DEGENERATE:
        return SEGMENT_FRAMEWORK_SEGMENT_B_DEGENERATE
    if q_ordering_status == ORDERING_VIOLATED:
        return SEGMENT_FRAMEWORK_ORDERING_VIOLATED
    return SEGMENT_FRAMEWORK_UNRESOLVED


def assign_segment_by_q(
    q_ah: float,
    q_segmentB_start_ah: float,
    q_vmax_dc_ah: float,
    q_final_dc_ah: float,
    q_final_dcac_ah: float,
    eps_ah: float = 1e-9,
) -> str:
    """
    Assign a target charge Q to Segment A/B/D/outside.

    Rules:
    - If Q > min(Q_final_DC, Q_final_DCAC), return SEGMENT_OUTSIDE.
    - Segment A: Q <= Q_segmentB_start_Ah
    - Segment B: Q_segmentB_start_Ah < Q <= Q_Vmax_DC_Ah
    - Segment D: Q > Q_Vmax_DC_Ah
    """
    required = [
        q_ah,
        q_segmentB_start_ah,
        q_vmax_dc_ah,
        q_final_dc_ah,
        q_final_dcac_ah,
    ]
    if not all(is_finite_number(x) for x in required):
        return SEGMENT_UNRESOLVED

    q = float(q_ah)
    q_b_start = float(q_segmentB_start_ah)
    q_vmax_dc = float(q_vmax_dc_ah)
    q_common_final = min(float(q_final_dc_ah), float(q_final_dcac_ah))

    if q > q_common_final + eps_ah:
        return SEGMENT_OUTSIDE

    if q <= q_b_start + eps_ah:
        return SEGMENT_A

    if q <= q_vmax_dc + eps_ah:
        return SEGMENT_B

    return SEGMENT_D


def segment_A_grid_count(
    segment_A_Q_lo_Ah: float = SEGMENT_A_Q_LO_AH,
    segment_A_Q_hi_Ah: float = np.nan,
    q_grid_step_Ah: float = Q_GRID_STEP_AH,
) -> int:
    """Number of fixed-step Q-grid points in Segment A."""
    if not all(is_finite_number(x) for x in [segment_A_Q_lo_Ah, segment_A_Q_hi_Ah, q_grid_step_Ah]):
        return 0
    if q_grid_step_Ah <= 0:
        return 0

    q_lo = float(segment_A_Q_lo_Ah)
    q_hi = float(segment_A_Q_hi_Ah)
    step = float(q_grid_step_Ah)

    if q_hi < q_lo:
        return 0

    # Inclusive fixed grid. segment_A_Q_lo_Ah is globally fixed at 0.050 Ah in Day21A.
    return int(np.floor((q_hi - q_lo) / step)) + 1


def segment_A_above_floor_status(
    max_abs_s: float,
    p95_abs_s: float,
    q_grid_count: Optional[int] = None,
) -> str:
    """
    Segment-A residual floor classification.

    Spike guard:
    - Applies only when p95 is available and Segment-A grid count is sufficient.
    - If max >= reopen but p95 < reopen, classify as spike_or_transition_artifact.
    """
    if not is_finite_number(max_abs_s):
        return ABOVE_FLOOR_UNRESOLVED

    max_abs = abs(float(max_abs_s))

    if max_abs <= SEG_A_FLOOR_COMPATIBLE_THRESHOLD_S:
        return ABOVE_FLOOR_NO

    if max_abs < SEG_A_REOPEN_THRESHOLD_S:
        return ABOVE_FLOOR_INTERMEDIATE

    # max_abs >= reopen threshold
    grid_ok = q_grid_count is None or q_grid_count >= Q_GRID_MIN_COUNT_SEGMENT_A
    p95_available = is_finite_number(p95_abs_s)

    if (not grid_ok) or (not p95_available):
        return ABOVE_FLOOR_YES_NO_P95_GUARD

    p95_abs = abs(float(p95_abs_s))
    if p95_abs < SEG_A_REOPEN_THRESHOLD_S:
        return ABOVE_FLOOR_SPIKE

    return ABOVE_FLOOR_YES


def late_cv_preservation_status(
    segment_D_anchor_dt_raw_s: Sequence[float],
    segment_D_dt_raw_median_s: float,
    threshold_s: float = LATE_CV_PRESERVATION_THRESHOLD_S,
) -> str:
    """
    Late-CV preservation rule:
    - If no Segment-D anchor exists, late-CV preservation is not required.
    - If Segment-D anchors exist, every Segment-D anchor must have dt_raw > threshold.
    - Segment-D median raw gain must also be > threshold.

    Note:
    threshold_s is logically independent from Segment-A residual floor,
    even though it is numerically matched in this audit.
    """
    anchor_values = [float(x) for x in segment_D_anchor_dt_raw_s if is_finite_number(x)]

    if len(segment_D_anchor_dt_raw_s) == 0:
        return LATE_CV_NOT_REQUIRED

    if len(anchor_values) != len(segment_D_anchor_dt_raw_s):
        return LATE_CV_UNRESOLVED

    if not is_finite_number(segment_D_dt_raw_median_s):
        return LATE_CV_UNRESOLVED

    anchors_ok = all(x > threshold_s for x in anchor_values)
    median_ok = float(segment_D_dt_raw_median_s) > threshold_s

    return LATE_CV_SATISFIED if (anchors_ok and median_ok) else LATE_CV_NOT_SATISFIED


# =============================================================================
# 13. Pre-registered verdict helper
# =============================================================================

def _require_allowed(value: str, allowed: set[str], name: str) -> None:
    if value not in allowed:
        raise ValueError(
            f"Invalid {name}: {value!r}. "
            f"Allowed values are: {sorted(allowed)}"
        )


def pre_registered_mechanism_verdict(
    *,
    q80_nominal_segment: str,
    q90_nominal_segment: str,
    q80_common_segment: str,
    q90_common_segment: str,
    segment_A_dt_resid_max_abs_s: float,
    segment_A_dt_resid_p95_abs_s: float,
    segment_A_Q_grid_count: Optional[int],
    q_vmax_ordering_status: str,
    segment_framework_status: str,
    geometry_phase_reference_status: str,
    final_q_diff_status: str,
    late_CV_preservation_satisfied: str,
) -> Dict[str, str]:
    """
    Implements the Markdown Cell 0.1 v9 decision tree.

    Returns:
    - evidence_status
    - mechanism_verdict
    - interpretation_class
    - caveat
    - segment_A_above_floor_status
    """

    caveat = ""

    # Strict input validation to prevent silent misclassification.
    for name, seg in {
        "q80_nominal_segment": q80_nominal_segment,
        "q90_nominal_segment": q90_nominal_segment,
        "q80_common_segment": q80_common_segment,
        "q90_common_segment": q90_common_segment,
    }.items():
        _require_allowed(seg, ALLOWED_Q_SEGMENTS, name)

    _require_allowed(q_vmax_ordering_status, ALLOWED_ORDERING_STATUS, "q_vmax_ordering_status")
    _require_allowed(segment_framework_status, ALLOWED_SEGMENT_FRAMEWORK_STATUS, "segment_framework_status")
    _require_allowed(geometry_phase_reference_status, ALLOWED_GEOMETRY_PHASE_STATUS, "geometry_phase_reference_status")
    _require_allowed(final_q_diff_status, ALLOWED_FINAL_Q_STATUS, "final_q_diff_status")
    _require_allowed(late_CV_preservation_satisfied, ALLOWED_LATE_CV_STATUS, "late_CV_preservation_satisfied")

    # Consistency check between ordering status and segment framework status.
    if q_vmax_ordering_status == ORDERING_DEGENERATE:
        if segment_framework_status != SEGMENT_FRAMEWORK_SEGMENT_B_DEGENERATE:
            raise ValueError(
                "Inconsistent status: ORDERING_DEGENERATE requires "
                "segment_framework_status == SEGMENT_FRAMEWORK_SEGMENT_B_DEGENERATE."
            )

    if segment_framework_status == SEGMENT_FRAMEWORK_SEGMENT_B_DEGENERATE:
        if q_vmax_ordering_status != ORDERING_DEGENERATE:
            raise ValueError(
                "Inconsistent status: SEGMENT_FRAMEWORK_SEGMENT_B_DEGENERATE requires "
                "q_vmax_ordering_status == ORDERING_DEGENERATE."
            )

    anchor_segments = {
        "Q80_nominal": q80_nominal_segment,
        "Q90_nominal": q90_nominal_segment,
        "Q80_common": q80_common_segment,
        "Q90_common": q90_common_segment,
    }

    if final_q_diff_status == FINAL_Q_MISMATCH_WARNING:
        caveat = append_caveat(caveat, CAVEAT_ASYMMETRIC_FINAL_Q)

    if any(seg == SEGMENT_OUTSIDE for seg in anchor_segments.values()):
        caveat = append_caveat(caveat, CAVEAT_NOMINAL_ANCHOR_OUTSIDE)

    if segment_A_Q_grid_count is not None and segment_A_Q_grid_count < Q_GRID_MIN_COUNT_SEGMENT_A:
        caveat = append_caveat(caveat, CAVEAT_SEGMENT_A_GRID_LOW)

    if q_vmax_ordering_status == ORDERING_DEGENERATE or (
        segment_framework_status == SEGMENT_FRAMEWORK_SEGMENT_B_DEGENERATE
    ):
        caveat = append_caveat(caveat, CAVEAT_SEGMENT_B_DEGENERATE)

    # Geometry phase guard
    if geometry_phase_reference_status == GEOM_PHASE_UNRESOLVED:
        caveat = append_caveat(caveat, CAVEAT_GEOMETRY_PHASE_UNRESOLVED)
        return {
            "evidence_status": EVIDENCE_NOT_APPLICABLE,
            "mechanism_verdict": "not_applicable_without_geometry_phase_reference",
            "interpretation_class": "geometry_phase_unresolved",
            "caveat": caveat,
            "segment_A_above_floor_status": ABOVE_FLOOR_UNRESOLVED,
        }

    # Event / framework guard
    if segment_framework_status == SEGMENT_FRAMEWORK_AC_OFF_PRECEDES_VMAX:
        caveat = append_caveat(caveat, CAVEAT_EVENT_REAUDIT_REQUIRED)
        return {
            "evidence_status": EVIDENCE_NOT_APPLICABLE,
            "mechanism_verdict": "not_applicable_without_event_reaudit",
            "interpretation_class": "AC_off_precedes_Vmax",
            "caveat": caveat,
            "segment_A_above_floor_status": ABOVE_FLOOR_UNRESOLVED,
        }

    # Ordering guard
    if (
        q_vmax_ordering_status == ORDERING_VIOLATED
        or segment_framework_status == SEGMENT_FRAMEWORK_ORDERING_VIOLATED
    ):
        caveat = append_caveat(caveat, CAVEAT_ORDERING_VIOLATED)
        return {
            "evidence_status": EVIDENCE_NOT_APPLICABLE,
            "mechanism_verdict": "not_applicable_without_redefinition",
            "interpretation_class": "ordering_violated",
            "caveat": caveat,
            "segment_A_above_floor_status": ABOVE_FLOOR_UNRESOLVED,
        }

    in_window_segments = [
        seg for seg in anchor_segments.values()
        if seg not in [SEGMENT_OUTSIDE, SEGMENT_UNRESOLVED]
    ]

    if len(in_window_segments) == 0:
        caveat = append_caveat(caveat, CAVEAT_ALL_ANCHORS_OUTSIDE)
        return {
            "evidence_status": EVIDENCE_NOT_APPLICABLE,
            "mechanism_verdict": "not_applicable_protocol_does_not_reach_target_state",
            "interpretation_class": "all_anchors_outside_common_Q_window",
            "caveat": caveat,
            "segment_A_above_floor_status": ABOVE_FLOOR_UNRESOLVED,
        }

    floor_status = segment_A_above_floor_status(
        max_abs_s=segment_A_dt_resid_max_abs_s,
        p95_abs_s=segment_A_dt_resid_p95_abs_s,
        q_grid_count=segment_A_Q_grid_count,
    )

    any_A = any(seg == SEGMENT_A for seg in in_window_segments)
    any_B = any(seg == SEGMENT_B for seg in in_window_segments)
    any_D = any(seg == SEGMENT_D for seg in in_window_segments)

    all_A = all(seg == SEGMENT_A for seg in in_window_segments)
    all_D = all(seg == SEGMENT_D for seg in in_window_segments)
    all_B_or_D = all(seg in [SEGMENT_B, SEGMENT_D] for seg in in_window_segments)

    # Spike guard
    if floor_status == ABOVE_FLOOR_SPIKE:
        return {
            "evidence_status": EVIDENCE_AMBIGUOUS,
            "mechanism_verdict": VERDICT_AMBIG,
            "interpretation_class": INTERPRET_SPIKE_CLASS,
            "caveat": caveat,
            "segment_A_above_floor_status": floor_status,
        }

    # Intermediate zone
    if floor_status == ABOVE_FLOOR_INTERMEDIATE:
        return {
            "evidence_status": EVIDENCE_AMBIGUOUS,
            "mechanism_verdict": VERDICT_AMBIG,
            "interpretation_class": INTERPRET_INTERMEDIATE_CLASS,
            "caveat": caveat,
            "segment_A_above_floor_status": floor_status,
        }

    # Interpretation B reopened
    if any_A and floor_status in [ABOVE_FLOOR_YES, ABOVE_FLOOR_YES_NO_P95_GUARD]:
        if floor_status == ABOVE_FLOOR_YES_NO_P95_GUARD:
            caveat = append_caveat(caveat, CAVEAT_P95_NOT_AVAILABLE)
            evidence_status = EVIDENCE_REOPENED_WITH_WARNING
        else:
            evidence_status = EVIDENCE_REOPENED

        return {
            "evidence_status": evidence_status,
            "mechanism_verdict": VERDICT_B_REOPENED,
            "interpretation_class": INTERPRET_B_CLASS,
            "caveat": caveat,
            "segment_A_above_floor_status": floor_status,
        }

    # Segment A above floor, but no in-window A anchor.
    if (not any_A) and floor_status in [ABOVE_FLOOR_YES, ABOVE_FLOOR_YES_NO_P95_GUARD]:
        caveat = append_caveat(caveat, CAVEAT_SEGMENT_A_ABOVE_FLOOR_NO_A_ANCHOR)
        if floor_status == ABOVE_FLOOR_YES_NO_P95_GUARD:
            caveat = append_caveat(caveat, CAVEAT_P95_NOT_AVAILABLE)

        return {
            "evidence_status": EVIDENCE_AMBIGUOUS,
            "mechanism_verdict": VERDICT_AMBIG,
            "interpretation_class": "segmentA_above_floor_without_inwindow_A_anchor",
            "caveat": caveat,
            "segment_A_above_floor_status": floor_status,
        }

    # Floor-compatible branches
    if floor_status == ABOVE_FLOOR_NO:
        # Late-CV unresolved explicitly blocks strong support if Segment-D anchors exist.
        if any_D and late_CV_preservation_satisfied == LATE_CV_UNRESOLVED:
            caveat = append_caveat(caveat, CAVEAT_LATE_CV_PRESERVATION_UNRESOLVED)
            return {
                "evidence_status": EVIDENCE_AMBIGUOUS,
                "mechanism_verdict": VERDICT_AMBIG,
                "interpretation_class": INTERPRET_LATE_CV_UNRESOLVED,
                "caveat": caveat,
                "segment_A_above_floor_status": floor_status,
            }

        # Strong support:
        # - no in-window A anchor
        # - at least one in-window B anchor
        # - all in-window anchors are B or D
        # - if D exists, late-CV preservation must be satisfied
        # - not allowed under segment_B_degenerate
        strong_late_cv_ok = (not any_D) or (
            late_CV_preservation_satisfied == LATE_CV_SATISFIED
        )
        strong_support_allowed_by_framework = (
            segment_framework_status != SEGMENT_FRAMEWORK_SEGMENT_B_DEGENERATE
        )

        if (
            strong_support_allowed_by_framework
            and (not any_A)
            and any_B
            and all_B_or_D
            and strong_late_cv_ok
        ):
            evidence_status = (
                EVIDENCE_SUPPORT_WITH_WARNING
                if final_q_diff_status == FINAL_Q_MISMATCH_WARNING
                else EVIDENCE_SUPPORTED
            )
            return {
                "evidence_status": evidence_status,
                "mechanism_verdict": VERDICT_A_STRONG,
                "interpretation_class": INTERPRET_A_STRONG_CLASS,
                "caveat": caveat,
                "segment_A_above_floor_status": floor_status,
            }

        # Weak support subclasses
        if all_A:
            interpretation_class = INTERPRET_GEOMETRY_ONLY_A
        elif all_D and late_CV_preservation_satisfied == LATE_CV_SATISFIED:
            interpretation_class = INTERPRET_LATE_CV_CONSISTENT
        else:
            # Includes mixed A/D, A/B, B/D, A/B/D, outside-reduced in-window distributions.
            interpretation_class = INTERPRET_MIXED_A_NOT_SUPPORTED

        evidence_status = (
            EVIDENCE_PARTIAL_SUPPORT_WITH_WARNING
            if final_q_diff_status == FINAL_Q_MISMATCH_WARNING
            else EVIDENCE_PARTIAL_SUPPORT
        )

        return {
            "evidence_status": evidence_status,
            "mechanism_verdict": VERDICT_A_WEAK,
            "interpretation_class": interpretation_class,
            "caveat": caveat,
            "segment_A_above_floor_status": floor_status,
        }

    # Unresolved residual state
    if floor_status == ABOVE_FLOOR_UNRESOLVED:
        return {
            "evidence_status": EVIDENCE_AMBIGUOUS,
            "mechanism_verdict": VERDICT_AMBIG,
            "interpretation_class": "segment_A_residual_unresolved",
            "caveat": caveat,
            "segment_A_above_floor_status": floor_status,
        }

    # Fallback
    return {
        "evidence_status": EVIDENCE_AMBIGUOUS,
        "mechanism_verdict": VERDICT_AMBIG,
        "interpretation_class": "fallback_unclassified_contract_state",
        "caveat": caveat,
        "segment_A_above_floor_status": floor_status,
    }


# =============================================================================
# 14. Inventory guards for Step 1
# =============================================================================

def assert_candidate_roles_mutually_exclusive(df_inventory: pd.DataFrame) -> None:
    """
    A single file must not be both DC reference and DCAC candidate.
    This is intended for Step 1 after inventory creation.
    """
    required = ["candidate_for_DC_reference", "candidate_for_DCAC"]
    missing = [c for c in required if c not in df_inventory.columns]
    if missing:
        raise ValueError(f"Inventory missing required candidate columns: {missing}")

    dc_ref = df_inventory["candidate_for_DC_reference"].map(parse_bool_strict)
    dcac = df_inventory["candidate_for_DCAC"].map(parse_bool_strict)

    both = dc_ref & dcac

    if both.any():
        bad = df_inventory.loc[both, ["file_name", "protocol_label", *required]]
        raise ValueError(
            "candidate_for_DC_reference and candidate_for_DCAC must be mutually exclusive.\n"
            f"{bad.to_string(index=False)}"
        )


def fill_unknown_strings_and_nan_numeric(
    df: pd.DataFrame,
    numeric_columns: Sequence[str],
    unknown: str = UNKNOWN,
) -> pd.DataFrame:
    """
    Contract rule:
    - Unknown metadata strings -> 'unknown_not_recorded'
    - Non-applicable numeric fields -> NaN
    """
    out = df.copy()
    numeric_set = set(numeric_columns)

    for col in out.columns:
        if col in numeric_set:
            out[col] = pd.to_numeric(out[col], errors="coerce")
        else:
            out[col] = out[col].where(out[col].notna(), unknown)
            out[col] = out[col].replace("", unknown)

    return out


# =============================================================================
# 15. Audit-contract JSON for reproducibility
# =============================================================================

FREQ_EXAMPLES = {
    "0.1tau": frequency_period_example(TAU_LABEL_S, 0.1),
    "1tau": frequency_period_example(TAU_LABEL_S, 1.0),
    "10tau": frequency_period_example(TAU_LABEL_S, 10.0),
}

AUDIT_CONTRACT: Dict[str, Any] = {
    "notebook": NOTEBOOK_NAME,
    "run_timestamp_utc": RUN_TIMESTAMP_UTC,
    "git_head": GIT_HEAD,

    "source_type": SOURCE_TYPE_MJ1,
    "cell_id": CELL_ID,
    "chemistry_family": CHEMISTRY_FAMILY,

    "strict_net_convention": {
        "charge_positive_current": True,
        "signed_current": True,
        "rectification": False,
        "cummax": False,
        "first_passage": True,
    },

    "locked_constants": {
        "Q_nom_Ah": Q_NOM_AH,
        "Q_nom_origin": (
            "standardized_1C_0p5C_pre_DCAC_capacity_characterization_same_MJ1_cell"
        ),
        "ONE_C_A": ONE_C_A,
        "Vmax_V": VMAX_V,
        "I_cutoff_A": I_CUTOFF_A,
        "I_cutoff_definition": I_CUTOFF_DEFINITION,
        "I_charge_onset_threshold_A": I_CHARGE_ONSET_THRESHOLD_A,
        "I_charge_onset_threshold_semantics": (
            "trim_rest_preconditioning_only_independent_from_CV_cutoff"
        ),
        "phase_convention": PHASE_CONVENTION,
        "ambient_temperature_C": AMBIENT_TEMPERATURE_C,
        "temperature_control_type": TEMPERATURE_CONTROL_TYPE,
        "temperature_sensor_type": TEMPERATURE_SENSOR_TYPE,
        "temperature_sensor_placement": TEMPERATURE_SENSOR_PLACEMENT,
        "temperature_data_source": TEMPERATURE_DATA_SOURCE,
        "temperature_alignment_method": TEMPERATURE_ALIGNMENT_METHOD,
        "temperature_to_NGU201_alignment_required": TEMPERATURE_TO_NGU201_ALIGNMENT_REQUIRED,
        "Q80_nominal_Ah": Q80_NOMINAL_AH,
        "Q90_nominal_Ah": Q90_NOMINAL_AH,
        "Q80_nominal_fraction_of_Q_nom": Q80_NOMINAL_FRACTION_OF_Q_NOM,
        "Q90_nominal_fraction_of_Q_nom": Q90_NOMINAL_FRACTION_OF_Q_NOM,
    },

    "measurement_source": {
        "default_voltage_source": DEFAULT_VOLTAGE_SOURCE,
        "default_current_source": DEFAULT_CURRENT_SOURCE,
        "default_timebase_source": DEFAULT_TIMEBASE_SOURCE,
        "default_time_alignment_method": DEFAULT_TIME_ALIGNMENT_METHOD,
        "default_voltage_current_alignment_status": DEFAULT_VOLTAGE_CURRENT_ALIGNMENT_STATUS,
        "note": "Inventory stores these per record for traceability.",
    },

    "temperature_measurement": {
        "ambient_temperature_C": AMBIENT_TEMPERATURE_C,
        "temperature_control_type": TEMPERATURE_CONTROL_TYPE,
        "temperature_sensor_type": TEMPERATURE_SENSOR_TYPE,
        "temperature_sensor_placement": TEMPERATURE_SENSOR_PLACEMENT,
        "temperature_data_source": TEMPERATURE_DATA_SOURCE,
        "temperature_alignment_method": TEMPERATURE_ALIGNMENT_METHOD,
        "temperature_to_NGU201_alignment_required": TEMPERATURE_TO_NGU201_ALIGNMENT_REQUIRED,
        "summary_fields": ["T_surface_max_C", "T_surface_mean_C"],
        "note": (
            "Temperature is used only as per-protocol summary metadata in Day21A; "
            "no sample-by-sample alignment to NGU201 voltage/current is performed."
        ),
    },

    "frequency_period_convention": {
        "tau_label_s": TAU_LABEL_S,
        "frequency_variable": "frequency_Hz",
        "tau_eff_definition": "tau_eff_s = m_tau * tau_label_s",
        "frequency_definition": "frequency_Hz = 1 / (2*pi*tau_eff_s)",
        "period_definition": "T_AC_s = 1 / frequency_Hz = 2*pi*tau_eff_s",
        "example_tau_label_11p1s": FREQ_EXAMPLES,
        "label_mapping": {
            "0.1τ": "0.1tau",
            "1τ": "1tau",
            "10τ": "10tau",
        },
        "bare_f_variable_allowed": False,
    },

    "charge_phase_time_zero": {
        "I_charge_onset_threshold_A": I_CHARGE_ONSET_THRESHOLD_A,
        "I_charge_onset_min_consecutive_samples": I_CHARGE_ONSET_MIN_CONSECUTIVE_SAMPLES,
        "purpose": "trim_rest_preconditioning_only_not_phase_alignment",
    },

    "geometry_phase_reference": {
        "allowed_status": [
            GEOM_PHASE_VERIFIED,
            GEOM_PHASE_ESTIMATED,
            GEOM_PHASE_UNRESOLVED,
        ],
        "required_fields": [
            "geometry_phase_offset_s",
            "geometry_phase_offset_rad",
            "geometry_phase_reference_status",
        ],
        "estimation_method": {
            "method": "least_squares_phase_fit",
            "model": "I_Q(t)=I_DC+I_AC*sin(2*pi*f_Hz*t+phi)",
            "fixed_parameters": ["I_DC", "I_AC", "f_Hz"],
            "estimated_parameter": "phi",
            "fit_window": "min(5*T_AC, t_ACoff_candidate)",
            "minimum_required_cycles": GEOMETRY_PHASE_MIN_REQUIRED_CYCLES,
        },
        "unresolved_verdict": "not_applicable_without_geometry_phase_reference",
    },

    "q_grid": {
        "segment_A_Q_lo_Ah": SEGMENT_A_Q_LO_AH,
        "Q_grid_step_Ah": Q_GRID_STEP_AH,
        "Q_grid_definition_method": Q_GRID_DEFINITION_METHOD,
        "Q_grid_min_count_segmentA": Q_GRID_MIN_COUNT_SEGMENT_A,
    },

    "segment_definitions": {
        "Segment_A": SEGMENT_A,
        "Segment_B": SEGMENT_B,
        "Segment_D": SEGMENT_D,
        "Segment_OUTSIDE": SEGMENT_OUTSIDE,
        "segment_C_handling": "absorbed_into_segment_B_start_no_separate_segment",
        "dt_resid_scope": "Segment_A_only_B_and_D_use_raw_dt",
        "segment_A_Q_hi_definition": "Q_segmentB_start_Ah",
        "Q_segmentB_degenerate_tolerance_Ah": Q_SEGMENTB_DEGENERATE_TOLERANCE_AH,
    },

    "vmax_and_acoff_detection": {
        "Vmax_detection_priority": VMAX_DETECTION_PRIORITY,
        "deglitch_applies_to_priorities": DEGLITCH_APPLIES_TO_PRIORITIES,
        "deglitch_min_consecutive_samples": DEGLITCH_MIN_CONSECUTIVE_SAMPLES,
        "deglitch_near_vmax_tol_V": DEGLITCH_NEAR_VMAX_TOL_V,
        "AC_off_envelope_tolerance_coefficient": AC_OFF_ENVELOPE_TOLERANCE_COEFFICIENT,
        "AC_off_persistence_n_T_AC": AC_OFF_PERSISTENCE_N_T_AC,
        "AC_off_low_frequency_voltage_window_V": AC_OFF_LOW_FREQUENCY_VOLTAGE_WINDOW_V,
        "AC_off_low_frequency_min_persistence_s": AC_OFF_LOW_FREQUENCY_MIN_PERSISTENCE_S,
        "AC_off_low_frequency_envelope_tolerance_coefficient": (
            AC_OFF_LOW_FREQUENCY_ENVELOPE_TOLERANCE_COEFFICIENT
        ),
        "AC_off_low_frequency_required_trend": AC_OFF_LOW_FREQUENCY_REQUIRED_TREND,
        "low_frequency_exception_type": "fallback_after_default_persistence_unavailable",
    },

    "final_q_consistency": {
        "final_Q_diff_threshold_Ah": FINAL_Q_DIFF_THRESHOLD_AH,
        "mismatch_status": FINAL_Q_MISMATCH_WARNING,
        "mismatch_caveat": CAVEAT_ASYMMETRIC_FINAL_Q,
        "mismatch_handling": "conservative_downgrade_by_one_evidence_level_no_exception",
    },

    "MJ1_experimental_residual_floor": {
        "floor_type": MJ1_FLOOR_TYPE,
        "max_abs_s": MJ1_FLOOR_MAX_ABS_S,
        "mean_s": MJ1_FLOOR_MEAN_S,
        "n": MJ1_FLOOR_N,
        "floor_compatible_threshold_s": SEG_A_FLOOR_COMPATIBLE_THRESHOLD_S,
        "reopen_threshold_s": SEG_A_REOPEN_THRESHOLD_S,
        "floor_uncertainty_propagation": {
            "method": "single_trial_max_inflation_by_factor_N",
            "no_replicate_SD_available": True,
            "threshold_inflation_is_heuristic_not_statistical": True,
            "known_conservative_bias": True,
        },
    },

    "late_CV_preservation": {
        "late_CV_preservation_threshold_s": LATE_CV_PRESERVATION_THRESHOLD_S,
        "unresolved_status": LATE_CV_UNRESOLVED,
        "unresolved_caveat": CAVEAT_LATE_CV_PRESERVATION_UNRESOLVED,
        "note": (
            "Numerically matched to Segment-A floor-compatible threshold for "
            "cross-section consistency, but logically independent."
        ),
    },

    "PyBaMM_reference_numerical_null_floor_s": {
        "max_abs_range_s": PYBAMM_NUMERICAL_NULL_MAX_ABS_RANGE_S,
        "note": "Reference only; not used as MJ1 experimental floor.",
    },

    "schemas": {
        "inventory_schema": INVENTORY_SCHEMA,
        "inventory_numeric_columns": INVENTORY_NUMERIC_COLUMNS,
        "unified_verdict_schema": UNIFIED_VERDICT_SCHEMA,
        "unified_verdict_numeric_columns": UNIFIED_VERDICT_NUMERIC_COLUMNS,
    },

    "output_paths": {
        "file_inventory_csv": str(OUT_FILE_INVENTORY),
        "unified_verdict_csv": str(OUT_UNIFIED_VERDICT),
        "audit_contract_json": str(OUT_AUDIT_CONTRACT_JSON),
    },
}

with open(OUT_AUDIT_CONTRACT_JSON, "w", encoding="utf-8") as f:
    json.dump(AUDIT_CONTRACT, f, indent=2, ensure_ascii=False)


# =============================================================================
# 16. Cell 1 closeout summary
# =============================================================================

print("[OK] Day21A Cell 1 v4.1 contract frozen.")
print(f"[OK] Notebook: {NOTEBOOK_NAME}")
print(f"[OK] Repo: {REPO}")
print(f"[OK] Git HEAD: {GIT_HEAD}")
print(f"[OK] Audit contract JSON: {OUT_AUDIT_CONTRACT_JSON}")
print(f"[OK] Inventory schema columns: {len(INVENTORY_SCHEMA)}")
print(f"[OK] Unified verdict schema columns: {len(UNIFIED_VERDICT_SCHEMA)}")
print(f"[OK] Q80_nominal_Ah = {Q80_NOMINAL_AH:.4f}")
print(f"[OK] Q90_nominal_Ah = {Q90_NOMINAL_AH:.4f}")
print(f"[OK] Q80_nominal_fraction_of_Q_nom = {Q80_NOMINAL_FRACTION_OF_Q_NOM:.2f}")
print(f"[OK] Q90_nominal_fraction_of_Q_nom = {Q90_NOMINAL_FRACTION_OF_Q_NOM:.2f}")
print(f"[OK] Segment-A residual Q lower bound = {SEGMENT_A_Q_LO_AH:.3f} Ah")
print(f"[OK] Segment-B degeneracy tolerance = {Q_SEGMENTB_DEGENERATE_TOLERANCE_AH:.3f} Ah")
print(f"[OK] Segment-A floor-compatible threshold = {SEG_A_FLOOR_COMPATIBLE_THRESHOLD_S:.2f} s")
print(f"[OK] Segment-A reopen threshold = {SEG_A_REOPEN_THRESHOLD_S:.2f} s")
print(f"[OK] Late-CV preservation threshold = {LATE_CV_PRESERVATION_THRESHOLD_S:.2f} s")
print(f"[OK] Q-grid step = {Q_GRID_STEP_AH:.3f} Ah")
print(f"[OK] Q-grid minimum count for p95 = {Q_GRID_MIN_COUNT_SEGMENT_A}")
print("[OK] No MJ1 data files scanned in Cell 1.")

[OK] Day21A Cell 1 v4.1 contract frozen.
[OK] Notebook: 25_day21A_MJ1_experimental_segment_audit.ipynb
[OK] Repo: /Users/louislu/pybamm-dcac-superimposed
[OK] Git HEAD: bf7514db35b9f4e407fb5f6a2942e519a27b0953
[OK] Audit contract JSON: /Users/louislu/pybamm-dcac-superimposed/data/day21A_audit_contract_schema_thresholds.json
[OK] Inventory schema columns: 45
[OK] Unified verdict schema columns: 93
[OK] Q80_nominal_Ah = 2.7200
[OK] Q90_nominal_Ah = 3.0600
[OK] Q80_nominal_fraction_of_Q_nom = 0.80
[OK] Q90_nominal_fraction_of_Q_nom = 0.90
[OK] Segment-A residual Q lower bound = 0.050 Ah
[OK] Segment-B degeneracy tolerance = 0.001 Ah
[OK] Segment-A floor-compatible threshold = 2.70 s
[OK] Segment-A reopen threshold = 6.75 s
[OK] Late-CV preservation threshold = 2.70 s
[OK] Q-grid step = 0.010 Ah
[OK] Q-grid minimum count for p95 = 30
[OK] No MJ1 data files scanned in Cell 1.


In [2]:
# Cell 2 — MJ1 file inventory
# Purpose:
# - Scan data/raw_mj1_ngu201_day21A/
# - Inspect file-level provenance and column availability
# - Infer protocol metadata from locked Day21A filenames
# - Optionally merge manual metadata if data/metadata/day21A_MJ1_manual_metadata.csv exists
# - Write data/day21A_step0_MJ1_file_inventory.csv
#
# Explicitly NOT done here:
# - No Q(t) integration
# - No Vmax / AC-off detection
# - No segment assignment
# - No Δt_raw / Δt_geom / Δt_resid
# - No mechanism verdict

from pathlib import Path
import csv
import re

RAW_DIR = DATA_DIR / "raw_mj1_ngu201_day21A"
METADATA_DIR = DATA_DIR / "metadata"
MANUAL_METADATA_CSV = METADATA_DIR / "day21A_MJ1_manual_metadata.csv"

METADATA_DIR.mkdir(parents=True, exist_ok=True)

EXPECTED_RAW_FILES = [
    "MJ1_0p3C_DC_NGU201_raw.csv",
    "MJ1_0p3C_0p7C_0p1tau_NGU201_raw.csv",
    "MJ1_0p3C_0p7C_1tau_NGU201_raw.csv",
    "MJ1_0p3C_0p7C_10tau_NGU201_raw.csv",
]


# -----------------------------------------------------------------------------
# 2.1 Small file-inspection helpers
# -----------------------------------------------------------------------------

def sniff_delimiter(path: Path, sample_bytes: int = 8192) -> str:
    """Best-effort delimiter detection for CSV header inspection."""
    try:
        sample = path.read_text(errors="ignore")[:sample_bytes]
        dialect = csv.Sniffer().sniff(sample, delimiters=[",", ";", "\t"])
        return dialect.delimiter
    except Exception:
        # NGU201-generated ARB / export CSVs are usually comma-separated.
        return ","


def read_header_columns(path: Path, delimiter: str) -> list[str]:
    """
    Read only enough of a CSV to identify columns.
    This is not trajectory loading.
    """
    try:
        # Try normal header first.
        df_head = pd.read_csv(path, sep=delimiter, nrows=5, engine="python")
        cols = [str(c).strip() for c in df_head.columns]
        if len(cols) > 1:
            return cols

        # If a file contains metadata/comment lines before the table,
        # scan for the first plausible tabular header.
        lines = path.read_text(errors="ignore").splitlines()
        for line in lines[:80]:
            parts = [p.strip() for p in line.split(delimiter)]
            lowered = [p.lower() for p in parts]
            looks_like_header = (
                any(("time" in p or "timestamp" in p) for p in lowered)
                and any(("u" in p or "volt" in p or "[v]" in p) for p in lowered)
                and any(("i" in p or "curr" in p or "[a]" in p) for p in lowered)
            )
            if len(parts) > 1 and looks_like_header:
                return parts

        return cols
    except Exception:
        return []


def column_flags(columns: list[str]) -> dict[str, bool]:
    """Infer whether key data types are present from column names only."""
    lowered = [c.lower().strip() for c in columns]

    has_time = any(
        ("time" in c) or ("timestamp" in c) or c in {"t", "zeit"}
        for c in lowered
    )

    has_voltage = any(
        ("u1" in c)
        or ("u[" in c)
        or ("u_" in c)
        or ("volt" in c)
        or ("voltage" in c)
        or ("[v]" in c and "i" not in c)
        for c in lowered
    )

    has_current = any(
        ("i1" in c)
        or ("i[" in c)
        or ("i_" in c)
        or ("curr" in c)
        or ("current" in c)
        or ("[a]" in c)
        for c in lowered
    )

    has_temperature = any(
        ("temp" in c)
        or ("temperature" in c)
        or ("pt100" in c)
        or ("°c" in c)
        or ("[c]" in c)
        for c in lowered
    )

    has_stage_marker = any(
        ("stage" in c)
        or ("mode" in c)
        or ("state" in c)
        or ("step" in c)
        or ("cv" == c)
        or ("cc" == c)
        for c in lowered
    )

    has_acoff_marker = any(
        ("ac_off" in c)
        or ("acoff" in c)
        or ("arb" in c)
        or ("arbitrary" in c)
        for c in lowered
    )

    return {
        "has_time": has_time,
        "has_voltage": has_voltage,
        "has_current": has_current,
        "has_temperature": has_temperature,
        "has_stage_marker": has_stage_marker,
        "has_AC_off_marker": has_acoff_marker,
    }


def infer_protocol_from_filename(file_name: str) -> dict[str, object]:
    """
    Locked filename-based inference for the four Day21A files.
    No trajectory data are used.
    """
    if file_name == "MJ1_0p3C_DC_NGU201_raw.csv":
        return {
            "protocol_label": "0.3C DC",
            "protocol_role": "DC_reference",
            "DC_C": 0.3,
            "AC_C": 0.0,
            "m_tau": np.nan,
            "frequency_Hz": 0.0,
            "tau_eff_s": np.nan,
            "candidate_for_DC_reference": True,
            "candidate_for_DCAC": False,
        }

    patterns = {
        "MJ1_0p3C_0p7C_0p1tau_NGU201_raw.csv": ("0.3C+0.7C 0.1tau", 0.1),
        "MJ1_0p3C_0p7C_1tau_NGU201_raw.csv": ("0.3C+0.7C 1tau", 1.0),
        "MJ1_0p3C_0p7C_10tau_NGU201_raw.csv": ("0.3C+0.7C 10tau", 10.0),
    }

    if file_name in patterns:
        protocol_label, m_tau = patterns[file_name]
        tau_eff_s = compute_tau_eff_s(TAU_LABEL_S, m_tau)
        frequency_hz = compute_frequency_hz_from_tau_eff(tau_eff_s)
        return {
            "protocol_label": protocol_label,
            "protocol_role": "DCAC",
            "DC_C": 0.3,
            "AC_C": 0.7,
            "m_tau": m_tau,
            "frequency_Hz": frequency_hz,
            "tau_eff_s": tau_eff_s,
            "candidate_for_DC_reference": False,
            "candidate_for_DCAC": True,
        }

    return {
        "protocol_label": UNKNOWN,
        "protocol_role": UNKNOWN,
        "DC_C": np.nan,
        "AC_C": np.nan,
        "m_tau": np.nan,
        "frequency_Hz": np.nan,
        "tau_eff_s": np.nan,
        "candidate_for_DC_reference": False,
        "candidate_for_DCAC": False,
    }


def make_inventory_row(path: Path) -> dict[str, object]:
    stat = path.stat()
    delimiter = sniff_delimiter(path)
    columns = read_header_columns(path, delimiter)
    flags = column_flags(columns)

    proto = infer_protocol_from_filename(path.name)

    row = {
        "file_path": str(path),
        "file_name": path.name,
        "file_mtime": datetime.fromtimestamp(stat.st_mtime).isoformat(),
        "file_size_kb": stat.st_size / 1024.0,
        "read_ok": len(columns) > 0,

        "source_type": SOURCE_TYPE_MJ1,
        "cell_id": CELL_ID,
        "source_session_date": UNKNOWN,

        "protocol_label": proto["protocol_label"],
        "protocol_role": proto["protocol_role"],
        "DC_C": proto["DC_C"],
        "AC_C": proto["AC_C"],
        "frequency_Hz": proto["frequency_Hz"],
        "m_tau": proto["m_tau"],
        "tau_label_s": TAU_LABEL_S,
        "tau_eff_s": proto["tau_eff_s"],
        "phase_convention": PHASE_CONVENTION,

        "Vmax_V": VMAX_V,
        "I_cutoff_A": I_CUTOFF_A,
        "I_charge_onset_threshold_A": I_CHARGE_ONSET_THRESHOLD_A,
        "Q_nom_Ah": Q_NOM_AH,

        "voltage_source": DEFAULT_VOLTAGE_SOURCE,
        "current_source": DEFAULT_CURRENT_SOURCE,
        # NGU201 export sampling rate is intentionally not guessed in Cell 2.
        # It can be filled from manual metadata if known.
        "sampling_rate_Hz": np.nan,

        "ambient_temperature_C": AMBIENT_TEMPERATURE_C,
        "temperature_control_type": TEMPERATURE_CONTROL_TYPE,
        "temperature_sensor_type": TEMPERATURE_SENSOR_TYPE,
        "temperature_sensor_placement": TEMPERATURE_SENSOR_PLACEMENT,
        "temperature_data_source": TEMPERATURE_DATA_SOURCE,
        "temperature_alignment_method": TEMPERATURE_ALIGNMENT_METHOD,
        "temperature_to_NGU201_alignment_required": TEMPERATURE_TO_NGU201_ALIGNMENT_REQUIRED,
        # Temperature is protocol-level summary metadata.
        # It is not computed from NGU201 V/I data in Cell 2.
        "T_surface_max_C": np.nan,
        "T_surface_mean_C": np.nan,

        "timebase_source": DEFAULT_TIMEBASE_SOURCE,
        "time_alignment_method": DEFAULT_TIME_ALIGNMENT_METHOD,
        "voltage_current_alignment_status": DEFAULT_VOLTAGE_CURRENT_ALIGNMENT_STATUS,

        **flags,

        "candidate_for_DC_reference": proto["candidate_for_DC_reference"],
        "candidate_for_DCAC": proto["candidate_for_DCAC"],
        "notes": f"delimiter={repr(delimiter)}; columns={columns}",
    }

    return row


# -----------------------------------------------------------------------------
# 2.2 Scan files
# -----------------------------------------------------------------------------

if not RAW_DIR.exists():
    raise FileNotFoundError(f"Raw data directory not found: {RAW_DIR}")

csv_paths = sorted(RAW_DIR.glob("*.csv"))

found_names = [p.name for p in csv_paths]
missing_expected = [name for name in EXPECTED_RAW_FILES if name not in found_names]
unexpected_files = [name for name in found_names if name not in EXPECTED_RAW_FILES]

print(f"[scan] RAW_DIR = {RAW_DIR}")
print(f"[scan] found CSV files = {len(csv_paths)}")
print(f"[scan] expected files missing = {missing_expected}")
print(f"[scan] unexpected CSV files = {unexpected_files}")

if missing_expected:
    raise FileNotFoundError(
        "Missing expected Day21A raw NGU201 files:\n"
        + "\n".join(missing_expected)
    )

# Unexpected files are not fatal, but they are not marked as candidates by filename inference.
rows = [make_inventory_row(path) for path in csv_paths]
df_inventory = pd.DataFrame(rows)


# -----------------------------------------------------------------------------
# 2.3 Optional manual metadata merge
# -----------------------------------------------------------------------------

MANUAL_METADATA_COLUMNS = [
    "file_name",
    "source_session_date",
    "protocol_label",
    "protocol_role",
    "DC_C",
    "AC_C",
    "frequency_Hz",
    "m_tau",
    "phase_convention",
    "sampling_rate_Hz",
    "ambient_temperature_C",
    "temperature_control_type",
    "temperature_sensor_type",
    "temperature_sensor_placement",
    "temperature_data_source",
    "temperature_alignment_method",
    "temperature_to_NGU201_alignment_required",
    "T_surface_max_C",
    "T_surface_mean_C",
    "notes",
]

if not MANUAL_METADATA_CSV.exists():
    # Create a template for user-editable metadata.
    df_template = df_inventory[[
        "file_name",
        "source_session_date",
        "protocol_label",
        "protocol_role",
        "DC_C",
        "AC_C",
        "frequency_Hz",
        "m_tau",
        "phase_convention",
        "sampling_rate_Hz",
        "ambient_temperature_C",
        "temperature_control_type",
        "temperature_sensor_type",
        "temperature_sensor_placement",
        "temperature_data_source",
        "temperature_alignment_method",
        "temperature_to_NGU201_alignment_required",
        "T_surface_max_C",
        "T_surface_mean_C",
        "notes",
    ]].copy()
    df_template.to_csv(MANUAL_METADATA_CSV, index=False)
    print(f"[metadata] Created manual metadata template: {MANUAL_METADATA_CSV}")
else:
    df_manual = pd.read_csv(MANUAL_METADATA_CSV)

    missing_manual_cols = [c for c in MANUAL_METADATA_COLUMNS if c not in df_manual.columns]
    if missing_manual_cols:
        raise ValueError(
            f"Manual metadata CSV missing columns: {missing_manual_cols}\n"
            f"File: {MANUAL_METADATA_CSV}"
        )

    # Merge manual fields by file_name. Manual metadata overrides inferred metadata
    # only for the listed editable fields.
    editable_cols = [c for c in MANUAL_METADATA_COLUMNS if c != "file_name"]
    df_inventory = df_inventory.merge(
        df_manual[["file_name", *editable_cols]],
        on="file_name",
        how="left",
        suffixes=("", "_manual"),
    )

    for col in editable_cols:
        manual_col = f"{col}_manual"
        if manual_col in df_inventory.columns:
            df_inventory[col] = df_inventory[manual_col].where(
                df_inventory[manual_col].notna(),
                df_inventory[col],
            )
            df_inventory = df_inventory.drop(columns=[manual_col])

    print(f"[metadata] Merged manual metadata: {MANUAL_METADATA_CSV}")


# -----------------------------------------------------------------------------
# 2.4 Enforce schema and contract guards
# -----------------------------------------------------------------------------

# Ensure all schema columns exist.
for col in INVENTORY_SCHEMA:
    if col not in df_inventory.columns:
        df_inventory[col] = np.nan if col in INVENTORY_NUMERIC_COLUMNS else UNKNOWN

# Drop any non-schema columns and enforce order.
df_inventory = df_inventory[INVENTORY_SCHEMA].copy()

# Fill unknown strings and numeric NaN according to contract.
df_inventory = fill_unknown_strings_and_nan_numeric(
    df_inventory,
    numeric_columns=INVENTORY_NUMERIC_COLUMNS,
    unknown=UNKNOWN,
)

# Make role flags strict bool again after fill.
df_inventory["candidate_for_DC_reference"] = df_inventory["candidate_for_DC_reference"].map(parse_bool_strict)
df_inventory["candidate_for_DCAC"] = df_inventory["candidate_for_DCAC"].map(parse_bool_strict)

# Schema checks.
assert_exact_schema(df_inventory, INVENTORY_SCHEMA, "INVENTORY_SCHEMA")
assert_candidate_roles_mutually_exclusive(df_inventory)

# Required candidate count checks for this Day21A startup batch.
n_dc_ref = int(df_inventory["candidate_for_DC_reference"].sum())
n_dcac = int(df_inventory["candidate_for_DCAC"].sum())

if n_dc_ref != 1:
    raise ValueError(f"Expected exactly 1 DC reference candidate, found {n_dc_ref}.")

if n_dcac != 3:
    raise ValueError(f"Expected exactly 3 DCAC candidates, found {n_dcac}.")

# Basic column availability checks.
missing_core_signal = df_inventory[
    ~(df_inventory["has_time"].astype(bool)
      & df_inventory["has_voltage"].astype(bool)
      & df_inventory["has_current"].astype(bool))
]

if len(missing_core_signal) > 0:
    print("[warning] Some files may be missing time/voltage/current columns by header inspection:")
    display_cols = ["file_name", "has_time", "has_voltage", "has_current", "notes"]
    print(missing_core_signal[display_cols].to_string(index=False))

# Write inventory.
df_inventory.to_csv(OUT_FILE_INVENTORY, index=False)

print(f"[OK] Wrote inventory: {OUT_FILE_INVENTORY}")
print(f"[OK] inventory shape = {df_inventory.shape}")
print(f"[OK] DC reference candidates = {n_dc_ref}")
print(f"[OK] DCAC candidates = {n_dcac}")

display_cols = [
    "file_name",
    "protocol_label",
    "protocol_role",
    "DC_C",
    "AC_C",
    "m_tau",
    "frequency_Hz",
    "candidate_for_DC_reference",
    "candidate_for_DCAC",
    "has_time",
    "has_voltage",
    "has_current",
    "sampling_rate_Hz",
    "T_surface_max_C",
    "T_surface_mean_C",
]
print(df_inventory[display_cols].to_string(index=False))

[scan] RAW_DIR = /Users/louislu/pybamm-dcac-superimposed/data/raw_mj1_ngu201_day21A
[scan] found CSV files = 4
[scan] expected files missing = []
[scan] unexpected CSV files = []
[metadata] Created manual metadata template: /Users/louislu/pybamm-dcac-superimposed/data/metadata/day21A_MJ1_manual_metadata.csv
[warning] Some files may be missing time/voltage/current columns by header inspection:
                          file_name  has_time  has_voltage  has_current                                                                                  notes
MJ1_0p3C_0p7C_0p1tau_NGU201_raw.csv     False        False        False delimiter=','; columns=['#Device', 'NGU201', 'Unnamed: 2', 'Unnamed: 3', 'Unnamed: 4']
 MJ1_0p3C_0p7C_10tau_NGU201_raw.csv     False        False        False                                           delimiter=','; columns=['#Device', 'NGU201']
  MJ1_0p3C_0p7C_1tau_NGU201_raw.csv     False        False        False                                           delimiter=','

In [3]:
# Cell 2A — debug NGU201 CSV header structure
# Purpose:
# - Print first lines of each NGU201 raw CSV with line numbers
# - Identify where the actual data header starts
# - Do not compute Q, Vmax, AC-off, or segments

RAW_DIR = DATA_DIR / "raw_mj1_ngu201_day21A"

for path in sorted(RAW_DIR.glob("*.csv")):
    print("\n" + "=" * 120)
    print(f"[FILE] {path.name}")
    print("=" * 120)

    try:
        lines = path.read_text(errors="ignore").splitlines()
    except Exception as exc:
        print(f"[ERROR] Could not read file: {exc}")
        continue

    n_show = min(80, len(lines))
    for i in range(n_show):
        print(f"{i:03d}: {lines[i]}")


[FILE] MJ1_0p3C_0p7C_0p1tau_NGU201_raw.csv
000: #Device,NGU201,,,
001: #Device Name,RS-NGU201-00000.local,,,
002: #Format,LOG,,,
003: #Date,26/11/2025,,,
004: #Version,04.013 0092C027A93,,,
005: #Serial No.,101322,,,
006: #Logging Interval[s],1,,,
007: #Mode,Unlimited,,,
008: #Start Time,10:27:16,,,
009: #Stop Time,XX:XX:XX,,,
010: #Actual Count,0000000000,,,
011: #Calibration Ch1,factory,,,
012: #Calibration Date Ch1,none,,,
013: Timestamp,U1[V],I1[A],P1[W],DVM1[V]
014: 27:16.9,nan,nan,nan,nan
015: 27:17.9,nan,nan,nan,nan
016: 27:18.9,nan,nan,nan,nan
017: 27:19.9,nan,nan,nan,nan
018: 27:20.9,nan,nan,nan,nan
019: 27:21.9,nan,nan,nan,nan
020: 27:22.9,nan,nan,nan,nan
021: 27:23.9,nan,nan,nan,nan
022: 27:24.9,nan,nan,nan,nan
023: 27:25.9,nan,nan,nan,nan
024: 27:26.9,nan,nan,nan,nan
025: 27:27.9,nan,nan,nan,nan
026: 27:28.9,nan,nan,nan,nan
027: 27:29.9,nan,nan,nan,nan
028: 27:30.9,nan,nan,nan,nan
029: 27:31.9,nan,nan,nan,nan
030: 27:32.9,nan,nan,nan,nan
031: 27:33.9,nan,nan,nan,nan
032: 2

In [11]:
# Cell 2B — rerun MJ1 file inventory with NGU201 LOG header detection fixed
# Fix:
# - NGU201 LOG files contain metadata lines before the actual table header.
# - The real table header is usually:
#   Timestamp,U1[V],I1[A],P1[W],DVM1[V]
#
# Explicitly NOT done here:
# - No Q(t) integration
# - No Vmax / AC-off detection
# - No segment assignment
# - No Δt_raw / Δt_geom / Δt_resid
# - No mechanism verdict

from pathlib import Path
import csv
import re

RAW_DIR = DATA_DIR / "raw_mj1_ngu201_day21A"
METADATA_DIR = DATA_DIR / "metadata"
MANUAL_METADATA_CSV = METADATA_DIR / "day21A_MJ1_manual_metadata.csv"

METADATA_DIR.mkdir(parents=True, exist_ok=True)

EXPECTED_RAW_FILES = [
    "MJ1_0p3C_DC_NGU201_raw.csv",
    "MJ1_0p3C_0p7C_0p1tau_NGU201_raw.csv",
    "MJ1_0p3C_0p7C_1tau_NGU201_raw.csv",
    "MJ1_0p3C_0p7C_10tau_NGU201_raw.csv",
]


# -----------------------------------------------------------------------------
# 2B.1 Header / metadata helpers
# -----------------------------------------------------------------------------

def sniff_delimiter(path: Path, sample_bytes: int = 8192) -> str:
    """Best-effort delimiter detection for CSV header inspection."""
    try:
        sample = path.read_text(errors="ignore")[:sample_bytes]
        dialect = csv.Sniffer().sniff(sample, delimiters=[",", ";", "\t"])
        return dialect.delimiter
    except Exception:
        return ","


def split_line(line: str, delimiter: str) -> list[str]:
    return [p.strip() for p in line.rstrip("\n").split(delimiter)]


def is_ngu201_data_header(parts: list[str]) -> bool:
    lowered = [p.lower().strip() for p in parts]
    return (
        "timestamp" in lowered
        and "u1[v]" in lowered
        and "i1[a]" in lowered
    )


def detect_ngu201_header(path: Path, delimiter: str, max_scan_lines: int = 200) -> dict[str, object]:
    """
    Detect NGU201 LOG metadata + actual table header.
    Returns:
      header_line_idx, columns, logging_interval_s, date, start_time
    """
    lines = path.read_text(errors="ignore").splitlines()

    header_line_idx = None
    columns: list[str] = []

    logging_interval_s = np.nan
    source_date = UNKNOWN
    start_time = UNKNOWN

    for i, line in enumerate(lines[:max_scan_lines]):
        parts = split_line(line, delimiter)
        if not parts:
            continue

        key = parts[0].strip()

        if key == "#Logging Interval[s]" and len(parts) >= 2:
            try:
                logging_interval_s = float(parts[1])
            except Exception:
                logging_interval_s = np.nan

        if key == "#Date" and len(parts) >= 2:
            source_date = str(parts[1]).strip() if str(parts[1]).strip() else UNKNOWN

        if key == "#Start Time" and len(parts) >= 2:
            start_time = str(parts[1]).strip() if str(parts[1]).strip() else UNKNOWN

        if is_ngu201_data_header(parts):
            header_line_idx = i
            columns = parts
            break

    return {
        "header_line_idx": header_line_idx,
        "columns": columns,
        "logging_interval_s": logging_interval_s,
        "source_date": source_date,
        "start_time": start_time,
    }


def column_flags(columns: list[str]) -> dict[str, bool]:
    lowered = [c.lower().strip() for c in columns]

    has_time = any(c == "timestamp" or "time" in c for c in lowered)
    has_voltage = any(c == "u1[v]" or "volt" in c or c.endswith("[v]") for c in lowered)
    has_current = any(c == "i1[a]" or "curr" in c or c.endswith("[a]") for c in lowered)

    # Temperature is not expected in NGU201 V/I file; Day21A carries temperature as protocol-level summary metadata.
    has_temperature = any(
        ("temp" in c)
        or ("temperature" in c)
        or ("pt100" in c)
        or ("°c" in c)
        or ("[c]" in c)
        for c in lowered
    )

    has_stage_marker = any(
        ("stage" in c)
        or ("mode" in c)
        or ("state" in c)
        or ("step" in c)
        or ("cv" == c)
        or ("cc" == c)
        for c in lowered
    )

    has_acoff_marker = any(
        ("ac_off" in c)
        or ("acoff" in c)
        or ("arb" in c)
        or ("arbitrary" in c)
        for c in lowered
    )

    return {
        "has_time": has_time,
        "has_voltage": has_voltage,
        "has_current": has_current,
        "has_temperature": has_temperature,
        "has_stage_marker": has_stage_marker,
        "has_AC_off_marker": has_acoff_marker,
    }


def infer_protocol_from_filename(file_name: str) -> dict[str, object]:
    """Locked filename-based inference for the four Day21A files. No trajectory data are used."""
    if file_name == "MJ1_0p3C_DC_NGU201_raw.csv":
        return {
            "protocol_label": "0.3C DC",
            "protocol_role": "DC_reference",
            "DC_C": 0.3,
            "AC_C": 0.0,
            "m_tau": np.nan,
            "frequency_Hz": 0.0,
            "tau_eff_s": np.nan,
            "candidate_for_DC_reference": True,
            "candidate_for_DCAC": False,
        }

    patterns = {
        "MJ1_0p3C_0p7C_0p1tau_NGU201_raw.csv": ("0.3C+0.7C 0.1tau", 0.1),
        "MJ1_0p3C_0p7C_1tau_NGU201_raw.csv": ("0.3C+0.7C 1tau", 1.0),
        "MJ1_0p3C_0p7C_10tau_NGU201_raw.csv": ("0.3C+0.7C 10tau", 10.0),
    }

    if file_name in patterns:
        protocol_label, m_tau = patterns[file_name]
        tau_eff_s = compute_tau_eff_s(TAU_LABEL_S, m_tau)
        frequency_hz = compute_frequency_hz_from_tau_eff(tau_eff_s)
        return {
            "protocol_label": protocol_label,
            "protocol_role": "DCAC",
            "DC_C": 0.3,
            "AC_C": 0.7,
            "m_tau": m_tau,
            "frequency_Hz": frequency_hz,
            "tau_eff_s": tau_eff_s,
            "candidate_for_DC_reference": False,
            "candidate_for_DCAC": True,
        }

    return {
        "protocol_label": UNKNOWN,
        "protocol_role": UNKNOWN,
        "DC_C": np.nan,
        "AC_C": np.nan,
        "m_tau": np.nan,
        "frequency_Hz": np.nan,
        "tau_eff_s": np.nan,
        "candidate_for_DC_reference": False,
        "candidate_for_DCAC": False,
    }


def make_inventory_row(path: Path) -> dict[str, object]:
    stat = path.stat()
    delimiter = sniff_delimiter(path)
    header_info = detect_ngu201_header(path, delimiter)
    columns = header_info["columns"]
    flags = column_flags(columns)

    proto = infer_protocol_from_filename(path.name)

    logging_interval_s = header_info["logging_interval_s"]
    sampling_rate_hz = np.nan
    if is_finite_number(logging_interval_s) and float(logging_interval_s) > 0:
        sampling_rate_hz = 1.0 / float(logging_interval_s)

    source_date = header_info["source_date"]
    start_time = header_info["start_time"]

    row = {
        "file_path": str(path),
        "file_name": path.name,
        "file_mtime": datetime.fromtimestamp(stat.st_mtime).isoformat(),
        "file_size_kb": stat.st_size / 1024.0,
        "read_ok": header_info["header_line_idx"] is not None,

        "source_type": SOURCE_TYPE_MJ1,
        "cell_id": CELL_ID,
        "source_session_date": source_date,

        "protocol_label": proto["protocol_label"],
        "protocol_role": proto["protocol_role"],
        "DC_C": proto["DC_C"],
        "AC_C": proto["AC_C"],
        "frequency_Hz": proto["frequency_Hz"],
        "m_tau": proto["m_tau"],
        "tau_label_s": TAU_LABEL_S,
        "tau_eff_s": proto["tau_eff_s"],
        "phase_convention": PHASE_CONVENTION,

        "Vmax_V": VMAX_V,
        "I_cutoff_A": I_CUTOFF_A,
        "I_charge_onset_threshold_A": I_CHARGE_ONSET_THRESHOLD_A,
        "Q_nom_Ah": Q_NOM_AH,

        "voltage_source": DEFAULT_VOLTAGE_SOURCE,
        "current_source": DEFAULT_CURRENT_SOURCE,
        "sampling_rate_Hz": sampling_rate_hz,

        "ambient_temperature_C": AMBIENT_TEMPERATURE_C,
        "temperature_control_type": TEMPERATURE_CONTROL_TYPE,
        "temperature_sensor_type": TEMPERATURE_SENSOR_TYPE,
        "temperature_sensor_placement": TEMPERATURE_SENSOR_PLACEMENT,
        "temperature_data_source": TEMPERATURE_DATA_SOURCE,
        "temperature_alignment_method": TEMPERATURE_ALIGNMENT_METHOD,
        "temperature_to_NGU201_alignment_required": TEMPERATURE_TO_NGU201_ALIGNMENT_REQUIRED,
        "T_surface_max_C": np.nan,
        "T_surface_mean_C": np.nan,

        "timebase_source": DEFAULT_TIMEBASE_SOURCE,
        "time_alignment_method": DEFAULT_TIME_ALIGNMENT_METHOD,
        "voltage_current_alignment_status": DEFAULT_VOLTAGE_CURRENT_ALIGNMENT_STATUS,

        **flags,

        "candidate_for_DC_reference": proto["candidate_for_DC_reference"],
        "candidate_for_DCAC": proto["candidate_for_DCAC"],
        "notes": (
            f"delimiter={repr(delimiter)}; "
            f"header_line_idx={header_info['header_line_idx']}; "
            f"columns={columns}; "
            f"start_time={start_time}"
        ),
    }

    return row


# -----------------------------------------------------------------------------
# 2B.2 Scan files
# -----------------------------------------------------------------------------

if not RAW_DIR.exists():
    raise FileNotFoundError(f"Raw data directory not found: {RAW_DIR}")

csv_paths = sorted(RAW_DIR.glob("*.csv"))

found_names = [p.name for p in csv_paths]
missing_expected = [name for name in EXPECTED_RAW_FILES if name not in found_names]
unexpected_files = [name for name in found_names if name not in EXPECTED_RAW_FILES]

print(f"[scan] RAW_DIR = {RAW_DIR}")
print(f"[scan] found CSV files = {len(csv_paths)}")
print(f"[scan] expected files missing = {missing_expected}")
print(f"[scan] unexpected CSV files = {unexpected_files}")

if missing_expected:
    raise FileNotFoundError(
        "Missing expected Day21A raw NGU201 files:\n"
        + "\n".join(missing_expected)
    )

rows = [make_inventory_row(path) for path in csv_paths]
df_inventory = pd.DataFrame(rows)


# -----------------------------------------------------------------------------
# 2B.3 Optional manual metadata merge
# -----------------------------------------------------------------------------

MANUAL_METADATA_COLUMNS = [
    "file_name",
    "source_session_date",
    "protocol_label",
    "protocol_role",
    "DC_C",
    "AC_C",
    "frequency_Hz",
    "m_tau",
    "phase_convention",
    "sampling_rate_Hz",
    "ambient_temperature_C",
    "temperature_control_type",
    "temperature_sensor_type",
    "temperature_sensor_placement",
    "temperature_data_source",
    "temperature_alignment_method",
    "temperature_to_NGU201_alignment_required",
    "T_surface_max_C",
    "T_surface_mean_C",
    "notes",
]

# If template already exists from previous Cell 2 run, merge it.
# It may contain blanks; blanks are not allowed to override inferred values.
if not MANUAL_METADATA_CSV.exists():
    df_template = df_inventory[[
        "file_name",
        "source_session_date",
        "protocol_label",
        "protocol_role",
        "DC_C",
        "AC_C",
        "frequency_Hz",
        "m_tau",
        "phase_convention",
        "sampling_rate_Hz",
        "ambient_temperature_C",
        "temperature_control_type",
        "temperature_sensor_type",
        "temperature_sensor_placement",
        "temperature_data_source",
        "temperature_alignment_method",
        "temperature_to_NGU201_alignment_required",
        "T_surface_max_C",
        "T_surface_mean_C",
        "notes",
    ]].copy()
    df_template.to_csv(MANUAL_METADATA_CSV, index=False)
    print(f"[metadata] Created manual metadata template: {MANUAL_METADATA_CSV}")
else:
    df_manual = pd.read_csv(MANUAL_METADATA_CSV)

    missing_manual_cols = [c for c in MANUAL_METADATA_COLUMNS if c not in df_manual.columns]
    if missing_manual_cols:
        raise ValueError(
            f"Manual metadata CSV missing columns: {missing_manual_cols}\n"
            f"File: {MANUAL_METADATA_CSV}"
        )

    editable_cols = [c for c in MANUAL_METADATA_COLUMNS if c != "file_name"]
    df_inventory = df_inventory.merge(
        df_manual[["file_name", *editable_cols]],
        on="file_name",
        how="left",
        suffixes=("", "_manual"),
    )

    for col in editable_cols:
        manual_col = f"{col}_manual"
        if manual_col in df_inventory.columns:
            # Blank strings in manual metadata must not override inferred values.
            manual_values = df_inventory[manual_col]
            if manual_values.dtype == object:
                manual_values = manual_values.replace("", np.nan)
            df_inventory[col] = manual_values.where(
                manual_values.notna(),
                df_inventory[col],
            )
            df_inventory = df_inventory.drop(columns=[manual_col])

    print(f"[metadata] Merged manual metadata: {MANUAL_METADATA_CSV}")


# -----------------------------------------------------------------------------
# 2B.4 Enforce schema and contract guards
# -----------------------------------------------------------------------------

for col in INVENTORY_SCHEMA:
    if col not in df_inventory.columns:
        df_inventory[col] = np.nan if col in INVENTORY_NUMERIC_COLUMNS else UNKNOWN

df_inventory = df_inventory[INVENTORY_SCHEMA].copy()

df_inventory = fill_unknown_strings_and_nan_numeric(
    df_inventory,
    numeric_columns=INVENTORY_NUMERIC_COLUMNS,
    unknown=UNKNOWN,
)

df_inventory["candidate_for_DC_reference"] = df_inventory["candidate_for_DC_reference"].map(parse_bool_strict)
df_inventory["candidate_for_DCAC"] = df_inventory["candidate_for_DCAC"].map(parse_bool_strict)

assert_exact_schema(df_inventory, INVENTORY_SCHEMA, "INVENTORY_SCHEMA")
assert_candidate_roles_mutually_exclusive(df_inventory)

n_dc_ref = int(df_inventory["candidate_for_DC_reference"].sum())
n_dcac = int(df_inventory["candidate_for_DCAC"].sum())

if n_dc_ref != 1:
    raise ValueError(f"Expected exactly 1 DC reference candidate, found {n_dc_ref}.")

if n_dcac != 3:
    raise ValueError(f"Expected exactly 3 DCAC candidates, found {n_dcac}.")

missing_core_signal = df_inventory[
    ~(df_inventory["has_time"].astype(bool)
      & df_inventory["has_voltage"].astype(bool)
      & df_inventory["has_current"].astype(bool))
]

if len(missing_core_signal) > 0:
    print("[warning] Some files may be missing time/voltage/current columns by header inspection:")
    display_cols = ["file_name", "has_time", "has_voltage", "has_current", "notes"]
    print(missing_core_signal[display_cols].to_string(index=False))
    raise ValueError("Header inspection failed for at least one required signal column.")

df_inventory.to_csv(OUT_FILE_INVENTORY, index=False)

print(f"[OK] Wrote inventory: {OUT_FILE_INVENTORY}")
print(f"[OK] inventory shape = {df_inventory.shape}")
print(f"[OK] DC reference candidates = {n_dc_ref}")
print(f"[OK] DCAC candidates = {n_dcac}")

display_cols = [
    "file_name",
    "source_session_date",
    "protocol_label",
    "protocol_role",
    "DC_C",
    "AC_C",
    "m_tau",
    "frequency_Hz",
    "sampling_rate_Hz",
    "candidate_for_DC_reference",
    "candidate_for_DCAC",
    "has_time",
    "has_voltage",
    "has_current",
    "T_surface_max_C",
    "T_surface_mean_C",
    "notes",
]
print(df_inventory[display_cols].to_string(index=False))

[scan] RAW_DIR = /Users/louislu/pybamm-dcac-superimposed/data/raw_mj1_ngu201_day21A
[scan] found CSV files = 4
[scan] expected files missing = []
[scan] unexpected CSV files = []
[metadata] Merged manual metadata: /Users/louislu/pybamm-dcac-superimposed/data/metadata/day21A_MJ1_manual_metadata.csv
[OK] Wrote inventory: /Users/louislu/pybamm-dcac-superimposed/data/day21A_step0_MJ1_file_inventory.csv
[OK] inventory shape = (4, 45)
[OK] DC reference candidates = 1
[OK] DCAC candidates = 3
                          file_name source_session_date   protocol_label protocol_role  DC_C  AC_C  m_tau  frequency_Hz  sampling_rate_Hz  candidate_for_DC_reference  candidate_for_DCAC  has_time  has_voltage  has_current  T_surface_max_C  T_surface_mean_C                                                                                                               notes
MJ1_0p3C_0p7C_0p1tau_NGU201_raw.csv          26/11/2025 0.3C+0.7C 0.1tau          DCAC   0.3   0.7    0.1      0.143383               1.

In [8]:
# Cell 2C — reset stale manual metadata template before rerunning Cell 2B

if MANUAL_METADATA_CSV.exists():
    MANUAL_METADATA_CSV.unlink()
    print(f"[OK] Deleted stale manual metadata template: {MANUAL_METADATA_CSV}")
else:
    print(f"[OK] Manual metadata template does not exist: {MANUAL_METADATA_CSV}")

[OK] Deleted stale manual metadata template: /Users/louislu/pybamm-dcac-superimposed/data/metadata/day21A_MJ1_manual_metadata.csv


In [10]:
# Cell 2D — patch manual metadata with protocol-level temperature summaries
# Do NOT run Cell 2C after this.

MANUAL_METADATA_CSV = DATA_DIR / "metadata" / "day21A_MJ1_manual_metadata.csv"

if not MANUAL_METADATA_CSV.exists():
    raise FileNotFoundError(
        f"Manual metadata template not found: {MANUAL_METADATA_CSV}\n"
        "Run Cell 2B once to create the template, then run this cell."
    )

df_meta = pd.read_csv(MANUAL_METADATA_CSV)

temperature_map = {
    "MJ1_0p3C_DC_NGU201_raw.csv": {
        "T_surface_max_C": 23.553,
        "T_surface_mean_C": 22.610,
    },
    "MJ1_0p3C_0p7C_0p1tau_NGU201_raw.csv": {
        "T_surface_max_C": 25.864,
        "T_surface_mean_C": 24.215,
    },
    "MJ1_0p3C_0p7C_1tau_NGU201_raw.csv": {
        "T_surface_max_C": 25.671,
        "T_surface_mean_C": 24.185,
    },
    "MJ1_0p3C_0p7C_10tau_NGU201_raw.csv": {
        "T_surface_max_C": 28.751,
        "T_surface_mean_C": 25.939,
    },
}

for file_name, vals in temperature_map.items():
    mask = df_meta["file_name"] == file_name
    if not mask.any():
        raise ValueError(f"File not found in manual metadata: {file_name}")

    df_meta.loc[mask, "T_surface_max_C"] = vals["T_surface_max_C"]
    df_meta.loc[mask, "T_surface_mean_C"] = vals["T_surface_mean_C"]

    df_meta.loc[mask, "temperature_sensor_type"] = "Pt100"
    df_meta.loc[mask, "temperature_sensor_placement"] = "axial_cell_surface"
    df_meta.loc[mask, "temperature_data_source"] = "measured_surface_temperature"
    df_meta.loc[mask, "temperature_alignment_method"] = "segment_level_summary_only"
    df_meta.loc[mask, "temperature_to_NGU201_alignment_required"] = False
    df_meta.loc[mask, "temperature_control_type"] = "ambient_lab_no_chamber"
    df_meta.loc[mask, "ambient_temperature_C"] = 20.0

df_meta.to_csv(MANUAL_METADATA_CSV, index=False)

print(f"[OK] Patched manual metadata with temperature summaries: {MANUAL_METADATA_CSV}")
print(df_meta[[
    "file_name",
    "T_surface_max_C",
    "T_surface_mean_C",
    "temperature_sensor_type",
    "temperature_alignment_method",
]].to_string(index=False))

[OK] Patched manual metadata with temperature summaries: /Users/louislu/pybamm-dcac-superimposed/data/metadata/day21A_MJ1_manual_metadata.csv
                          file_name  T_surface_max_C  T_surface_mean_C temperature_sensor_type temperature_alignment_method
MJ1_0p3C_0p7C_0p1tau_NGU201_raw.csv           25.864            24.215                   Pt100   segment_level_summary_only
 MJ1_0p3C_0p7C_10tau_NGU201_raw.csv           28.751            25.939                   Pt100   segment_level_summary_only
  MJ1_0p3C_0p7C_1tau_NGU201_raw.csv           25.671            24.185                   Pt100   segment_level_summary_only
         MJ1_0p3C_DC_NGU201_raw.csv           23.553            22.610                   Pt100   segment_level_summary_only


In [12]:
# Cell 3 — load candidate NGU201 trajectories and standardize retained charge interval
#
# Purpose:
# - Load the four NGU201 raw CSV files listed in the frozen inventory
# - Skip NGU201 metadata rows and use the detected LOG header
# - Parse Timestamp into a monotonic time axis
# - Convert measured current into charge-positive analysis convention I_Q
# - Trim pre-output / NaN / rest region using I_charge_onset_threshold_A
# - Produce basic trajectory sanity table
#
# Explicitly NOT done here:
# - No Q(t) integration for Δt analysis
# - No Vmax / AC-off detection
# - No Segment A/B/D assignment
# - No Δt_raw / Δt_geom / Δt_resid
# - No mechanism verdict

import re
from pathlib import Path

# -----------------------------------------------------------------------------
# 3.1 Load frozen file inventory
# -----------------------------------------------------------------------------

if not OUT_FILE_INVENTORY.exists():
    raise FileNotFoundError(
        f"Inventory CSV not found: {OUT_FILE_INVENTORY}\n"
        "Run Cell 2B first."
    )

df_inventory = pd.read_csv(OUT_FILE_INVENTORY)

assert_exact_schema(df_inventory, INVENTORY_SCHEMA, "INVENTORY_SCHEMA")

# Restore bool columns after CSV roundtrip
df_inventory["candidate_for_DC_reference"] = df_inventory["candidate_for_DC_reference"].map(parse_bool_strict)
df_inventory["candidate_for_DCAC"] = df_inventory["candidate_for_DCAC"].map(parse_bool_strict)

assert_candidate_roles_mutually_exclusive(df_inventory)

required_signal_ok = (
    df_inventory["has_time"].map(parse_bool_strict)
    & df_inventory["has_voltage"].map(parse_bool_strict)
    & df_inventory["has_current"].map(parse_bool_strict)
)

if not required_signal_ok.all():
    bad = df_inventory.loc[~required_signal_ok, ["file_name", "has_time", "has_voltage", "has_current", "notes"]]
    raise ValueError(
        "Cannot load trajectories: at least one file failed signal-column inventory.\n"
        + bad.to_string(index=False)
    )

print(f"[OK] Loaded inventory: {OUT_FILE_INVENTORY}")
print(f"[OK] inventory rows = {len(df_inventory)}")


# -----------------------------------------------------------------------------
# 3.2 Timestamp parsing helpers
# -----------------------------------------------------------------------------

def parse_ngu201_timestamp_to_seconds(ts: object) -> float:
    """
    Parse NGU201 Timestamp strings.

    Supported examples:
    - '27:16.9'       -> MM:SS.s
    - '44:26.0'       -> MM:SS.s
    - '10:47:26.7'    -> HH:MM:SS.s

    Returns seconds in the local timestamp coordinate before monotonic unwrapping.
    """
    if pd.isna(ts):
        return np.nan

    s = str(ts).strip()
    if not s:
        return np.nan

    parts = s.split(":")
    try:
        if len(parts) == 2:
            minutes = float(parts[0])
            seconds = float(parts[1])
            return minutes * 60.0 + seconds

        if len(parts) == 3:
            hours = float(parts[0])
            minutes = float(parts[1])
            seconds = float(parts[2])
            return hours * 3600.0 + minutes * 60.0 + seconds

        # Fallback: already numeric seconds
        return float(s)
    except Exception:
        return np.nan


def unwrap_monotonic_time_seconds(raw_seconds: np.ndarray, timestamp_strings: Sequence[object]) -> np.ndarray:
    """
    Make parsed timestamp seconds monotonic if NGU201 timestamp wraps.

    If all timestamps are MM:SS-like, use 3600 s rollover.
    If HH:MM:SS-like timestamps are present, use 24 h rollover.
    """
    out = np.asarray(raw_seconds, dtype=float).copy()

    colon_counts = []
    for x in timestamp_strings:
        if pd.isna(x):
            colon_counts.append(0)
        else:
            colon_counts.append(str(x).count(":"))

    rollover_period_s = 86400.0 if max(colon_counts, default=0) >= 2 else 3600.0

    offset = 0.0
    prev = np.nan

    for i, val in enumerate(out):
        if not np.isfinite(val):
            out[i] = np.nan
            continue

        candidate = val + offset

        if np.isfinite(prev) and candidate < prev - 1e-6:
            offset += rollover_period_s
            candidate = val + offset

        out[i] = candidate
        prev = candidate

    return out


def find_charge_onset_index(
    current_q: np.ndarray,
    voltage: np.ndarray,
    threshold_A: float = I_CHARGE_ONSET_THRESHOLD_A,
    min_consecutive: int = I_CHARGE_ONSET_MIN_CONSECUTIVE_SAMPLES,
) -> int:
    """
    Find first sustained charge-positive current onset.

    This is only for trimming pre-output/rest/nan rows.
    It does not establish ARB phase alignment.
    """
    finite = np.isfinite(current_q) & np.isfinite(voltage)
    above = finite & (current_q >= threshold_A)

    n = len(current_q)
    if n < min_consecutive:
        raise ValueError("Trajectory too short for charge-onset detection.")

    for i in range(0, n - min_consecutive + 1):
        if np.all(above[i : i + min_consecutive]):
            return i

    raise ValueError(
        f"No charge onset found with I_Q >= {threshold_A} A "
        f"for {min_consecutive} consecutive samples."
    )


def load_ngu201_log_trajectory(path: Path) -> tuple[pd.DataFrame, dict[str, object]]:
    """
    Load one NGU201 LOG CSV into a standardized retained charge trajectory.

    Returned DataFrame columns:
    - timestamp_raw
    - t_abs_s
    - t_s
    - U_V
    - I_meas_A
    - I_Q_A
    - P_W
    - DVM_V

    No Q integration is performed here.
    """
    delimiter = sniff_delimiter(path)
    header_info = detect_ngu201_header(path, delimiter)

    header_line_idx = header_info["header_line_idx"]
    if header_line_idx is None:
        raise ValueError(f"Could not detect NGU201 data header in {path.name}")

    df_raw = pd.read_csv(
        path,
        sep=delimiter,
        skiprows=header_line_idx,
        engine="python",
    )

    # Normalize expected columns
    required_cols = ["Timestamp", "U1[V]", "I1[A]"]
    missing = [c for c in required_cols if c not in df_raw.columns]
    if missing:
        raise ValueError(
            f"{path.name}: missing required columns after header parse: {missing}. "
            f"Detected columns: {df_raw.columns.tolist()}"
        )

    timestamp_raw = df_raw["Timestamp"].astype(str)

    t_parsed = np.array(
        [parse_ngu201_timestamp_to_seconds(x) for x in timestamp_raw],
        dtype=float,
    )
    t_abs = unwrap_monotonic_time_seconds(t_parsed, timestamp_raw)

    U_V = pd.to_numeric(df_raw["U1[V]"], errors="coerce").to_numpy(dtype=float)
    I_meas_A = pd.to_numeric(df_raw["I1[A]"], errors="coerce").to_numpy(dtype=float)

    # Day21A experimental NGU201 convention:
    # charging current is positive in the raw NGU201 record.
    I_Q_A = I_meas_A.copy()

    P_W = (
        pd.to_numeric(df_raw["P1[W]"], errors="coerce").to_numpy(dtype=float)
        if "P1[W]" in df_raw.columns
        else np.full(len(df_raw), np.nan)
    )

    DVM_V = (
        pd.to_numeric(df_raw["DVM1[V]"], errors="coerce").to_numpy(dtype=float)
        if "DVM1[V]" in df_raw.columns
        else np.full(len(df_raw), np.nan)
    )

    onset_idx = find_charge_onset_index(
        current_q=I_Q_A,
        voltage=U_V,
        threshold_A=I_CHARGE_ONSET_THRESHOLD_A,
        min_consecutive=I_CHARGE_ONSET_MIN_CONSECUTIVE_SAMPLES,
    )

    t0 = t_abs[onset_idx]
    if not np.isfinite(t0):
        raise ValueError(f"{path.name}: charge onset time is not finite.")

    df = pd.DataFrame(
        {
            "timestamp_raw": timestamp_raw,
            "t_abs_s": t_abs,
            "t_s": t_abs - t0,
            "U_V": U_V,
            "I_meas_A": I_meas_A,
            "I_Q_A": I_Q_A,
            "P_W": P_W,
            "DVM_V": DVM_V,
        }
    )

    # Retain from charge onset onward.
    df_retained = df.iloc[onset_idx:].reset_index(drop=True)

    # Drop rows after onset only if time is invalid. Keep signed current waveform unchanged.
    df_retained = df_retained[np.isfinite(df_retained["t_s"])].reset_index(drop=True)

    # Basic sanity metrics
    dt = np.diff(df_retained["t_s"].to_numpy(dtype=float))
    dt_finite = dt[np.isfinite(dt)]

    finite_u = np.isfinite(df_retained["U_V"].to_numpy(dtype=float))
    finite_i = np.isfinite(df_retained["I_Q_A"].to_numpy(dtype=float))

    meta = {
        "delimiter": delimiter,
        "header_line_idx": header_line_idx,
        "logging_interval_s": header_info.get("logging_interval_s", np.nan),
        "source_date": header_info.get("source_date", UNKNOWN),
        "start_time": header_info.get("start_time", UNKNOWN),
        "n_raw_rows": len(df_raw),
        "n_retained_rows": len(df_retained),
        "charge_onset_idx_raw": onset_idx,
        "charge_onset_timestamp_raw": str(timestamp_raw.iloc[onset_idx]),
        "retained_duration_s": (
            float(df_retained["t_s"].iloc[-1])
            if len(df_retained) > 0
            else np.nan
        ),
        "dt_median_s": float(np.nanmedian(dt_finite)) if len(dt_finite) else np.nan,
        "dt_min_s": float(np.nanmin(dt_finite)) if len(dt_finite) else np.nan,
        "dt_max_s": float(np.nanmax(dt_finite)) if len(dt_finite) else np.nan,
        "finite_voltage_fraction": float(np.mean(finite_u)) if len(df_retained) else np.nan,
        "finite_current_fraction": float(np.mean(finite_i)) if len(df_retained) else np.nan,
        "U_min_V": float(np.nanmin(df_retained["U_V"])) if finite_u.any() else np.nan,
        "U_max_V": float(np.nanmax(df_retained["U_V"])) if finite_u.any() else np.nan,
        "I_Q_min_A": float(np.nanmin(df_retained["I_Q_A"])) if finite_i.any() else np.nan,
        "I_Q_max_A": float(np.nanmax(df_retained["I_Q_A"])) if finite_i.any() else np.nan,
        "I_Q_mean_A": float(np.nanmean(df_retained["I_Q_A"])) if finite_i.any() else np.nan,
    }

    return df_retained, meta


# -----------------------------------------------------------------------------
# 3.3 Load all candidate trajectories
# -----------------------------------------------------------------------------

TRAJ: dict[str, pd.DataFrame] = {}
trajectory_summary_rows: list[dict[str, object]] = []

for _, inv_row in df_inventory.iterrows():
    path = Path(inv_row["file_path"])
    file_name = inv_row["file_name"]

    df_traj, meta = load_ngu201_log_trajectory(path)

    # Attach protocol metadata to each retained trajectory as constant columns.
    df_traj["file_name"] = file_name
    df_traj["protocol_label"] = inv_row["protocol_label"]
    df_traj["protocol_role"] = inv_row["protocol_role"]
    df_traj["DC_C"] = inv_row["DC_C"]
    df_traj["AC_C"] = inv_row["AC_C"]
    df_traj["m_tau"] = inv_row["m_tau"]
    df_traj["frequency_Hz"] = inv_row["frequency_Hz"]
    df_traj["phase_convention"] = inv_row["phase_convention"]

    TRAJ[file_name] = df_traj

    row = {
        "file_name": file_name,
        "protocol_label": inv_row["protocol_label"],
        "protocol_role": inv_row["protocol_role"],
        "DC_C": inv_row["DC_C"],
        "AC_C": inv_row["AC_C"],
        "m_tau": inv_row["m_tau"],
        "frequency_Hz": inv_row["frequency_Hz"],
        "sampling_rate_Hz_inventory": inv_row["sampling_rate_Hz"],
        **meta,
    }
    trajectory_summary_rows.append(row)


df_load_summary = pd.DataFrame(trajectory_summary_rows)

OUT_LOAD_SUMMARY = DATA_DIR / "day21A_step1_MJ1_loaded_trajectory_sanity.csv"
df_load_summary.to_csv(OUT_LOAD_SUMMARY, index=False)

print(f"[OK] Loaded retained trajectories: {len(TRAJ)}")
print(f"[OK] Wrote load sanity summary: {OUT_LOAD_SUMMARY}")

display_cols = [
    "file_name",
    "protocol_label",
    "protocol_role",
    "n_raw_rows",
    "n_retained_rows",
    "charge_onset_idx_raw",
    "charge_onset_timestamp_raw",
    "retained_duration_s",
    "dt_median_s",
    "dt_min_s",
    "dt_max_s",
    "finite_voltage_fraction",
    "finite_current_fraction",
    "U_min_V",
    "U_max_V",
    "I_Q_min_A",
    "I_Q_max_A",
    "I_Q_mean_A",
]
print(df_load_summary[display_cols].to_string(index=False))


# -----------------------------------------------------------------------------
# 3.4 Hard sanity guards
# -----------------------------------------------------------------------------

# No Q integration below; these guards only check data usability.

for file_name, df_traj in TRAJ.items():
    if len(df_traj) < 10:
        raise ValueError(f"{file_name}: retained trajectory has fewer than 10 rows.")

    if not df_traj["t_s"].is_monotonic_increasing:
        raise ValueError(f"{file_name}: retained t_s is not monotonic increasing.")

    if df_traj["U_V"].notna().mean() < 0.95:
        raise ValueError(f"{file_name}: voltage finite fraction below 0.95.")

    if df_traj["I_Q_A"].notna().mean() < 0.95:
        raise ValueError(f"{file_name}: current finite fraction below 0.95.")

print("[OK] Cell 3 trajectory loading and basic sanity checks passed.")
print("[OK] No Q integration, event detection, segmentation, or verdict performed in Cell 3.")

[OK] Loaded inventory: /Users/louislu/pybamm-dcac-superimposed/data/day21A_step0_MJ1_file_inventory.csv
[OK] inventory rows = 4
[OK] Loaded retained trajectories: 4
[OK] Wrote load sanity summary: /Users/louislu/pybamm-dcac-superimposed/data/day21A_step1_MJ1_loaded_trajectory_sanity.csv
                          file_name   protocol_label protocol_role  n_raw_rows  n_retained_rows  charge_onset_idx_raw charge_onset_timestamp_raw  retained_duration_s  dt_median_s  dt_min_s  dt_max_s  finite_voltage_fraction  finite_current_fraction  U_min_V  U_max_V  I_Q_min_A  I_Q_max_A  I_Q_mean_A
MJ1_0p3C_0p7C_0p1tau_NGU201_raw.csv 0.3C+0.7C 0.1tau          DCAC       13456            13417                    39                    27:56.0              13430.0          1.0       0.8       8.2                 0.996422                 0.996422 2.740865 4.202711  -1.360000   3.400048    0.897529
 MJ1_0p3C_0p7C_10tau_NGU201_raw.csv  0.3C+0.7C 10tau          DCAC       12796            12735               

In [13]:
# Cell 4 — Event / AC-off detection audit
#
# Purpose:
# - Detect Vmax event time from NGU201 voltage
# - Detect AC-off / transition support from current waveform
# - Persist event-detection method and lag audit
#
# Explicitly NOT done here:
# - No Q(t) integration
# - No Segment A/B/D assignment
# - No Δt_raw / Δt_geom / Δt_resid
# - No mechanism verdict

OUT_EVENT_AUDIT = DATA_DIR / "day21A_step2_MJ1_event_acoff_audit.csv"


# -----------------------------------------------------------------------------
# 4.1 Vmax detection
# -----------------------------------------------------------------------------

def detect_vmax_event(
    df: pd.DataFrame,
    vmax_v: float = VMAX_V,
    near_tol_v: float = DEGLITCH_NEAR_VMAX_TOL_V,
    min_consecutive: int = DEGLITCH_MIN_CONSECUTIVE_SAMPLES,
) -> dict[str, object]:
    """
    Priority 3 fallback detection:
    first V >= Vmax with deglitch validation.

    Deglitch:
    - first sample V >= Vmax
    - next min_consecutive samples are near/above Vmax - near_tol_v
    """
    U = df["U_V"].to_numpy(dtype=float)
    t = df["t_s"].to_numpy(dtype=float)

    finite = np.isfinite(U) & np.isfinite(t)
    candidate_idxs = np.where(finite & (U >= vmax_v))[0]

    for idx in candidate_idxs:
        end = min(idx + min_consecutive, len(df))
        if end - idx < min_consecutive:
            continue

        window_U = U[idx:end]
        if np.all(np.isfinite(window_U)) and np.all(window_U >= vmax_v - near_tol_v):
            return {
                "idx_Vmax": int(idx),
                "t_Vmax_detected_s": float(t[idx]),
                "U_at_Vmax_detected_V": float(U[idx]),
                "Vmax_detection_method_used": (
                    "priority3_first_V_ge_4p2V_with_deglitch"
                ),
                "Vmax_detection_status": "detected",
            }

    if len(candidate_idxs) > 0:
        idx = int(candidate_idxs[0])
        return {
            "idx_Vmax": idx,
            "t_Vmax_detected_s": float(t[idx]),
            "U_at_Vmax_detected_V": float(U[idx]),
            "Vmax_detection_method_used": (
                "priority3_first_V_ge_4p2V_without_deglitch_warning"
            ),
            "Vmax_detection_status": "detected_without_deglitch",
        }

    return {
        "idx_Vmax": np.nan,
        "t_Vmax_detected_s": np.nan,
        "U_at_Vmax_detected_V": np.nan,
        "Vmax_detection_method_used": "unresolved_no_V_ge_Vmax",
        "Vmax_detection_status": "unresolved",
    }


# -----------------------------------------------------------------------------
# 4.2 AC-off detection support from current waveform
# -----------------------------------------------------------------------------

def ac_off_window_supports_no_sinusoid(
    df: pd.DataFrame,
    idx_candidate: int,
    persistence_s: float,
    ac_amp_A: float,
    voltage_window: Optional[Sequence[float]] = None,
) -> dict[str, object]:
    """
    Check whether the post-candidate current window supports AC disappearance.

    Contract-compatible operational checks:
    - post-candidate record covers required persistence window
    - if voltage_window is provided, candidate voltage lies inside it
    - current does not show negative AC half-wave after candidate:
      min(I_Q) >= -0.05 * I_AC
    """
    t = df["t_s"].to_numpy(dtype=float)
    U = df["U_V"].to_numpy(dtype=float)
    I = df["I_Q_A"].to_numpy(dtype=float)

    if idx_candidate < 0 or idx_candidate >= len(df):
        return {"ok": False, "reason": "candidate_index_out_of_range"}

    t0 = t[idx_candidate]
    if not np.isfinite(t0):
        return {"ok": False, "reason": "candidate_time_not_finite"}

    if not np.isfinite(persistence_s) or persistence_s <= 0:
        return {"ok": False, "reason": "invalid_persistence_s"}

    if not np.isfinite(ac_amp_A) or ac_amp_A <= 0:
        return {"ok": False, "reason": "invalid_ac_amp_A"}

    if voltage_window is not None:
        lo, hi = float(voltage_window[0]), float(voltage_window[1])
        U0 = U[idx_candidate]
        if not np.isfinite(U0) or not (lo <= U0 <= hi):
            return {
                "ok": False,
                "reason": "candidate_not_in_voltage_window",
                "U_candidate_V": float(U0) if np.isfinite(U0) else np.nan,
            }

    mask = (t >= t0) & (t <= t0 + persistence_s)
    n_window = int(np.sum(mask))

    if n_window < 3:
        return {"ok": False, "reason": "insufficient_samples_in_window"}

    t_end_available = np.nanmax(t)
    if t_end_available < t0 + persistence_s:
        return {
            "ok": False,
            "reason": "post_transition_record_shorter_than_persistence",
            "post_available_s": float(t_end_available - t0),
            "required_persistence_s": float(persistence_s),
        }

    Iw = I[mask]
    finite_i = Iw[np.isfinite(Iw)]

    if len(finite_i) < 3:
        return {"ok": False, "reason": "insufficient_finite_current_samples"}

    tol_A = AC_OFF_ENVELOPE_TOLERANCE_COEFFICIENT * float(ac_amp_A)
    min_i = float(np.nanmin(finite_i))
    max_i = float(np.nanmax(finite_i))
    mean_i = float(np.nanmean(finite_i))

    no_negative_ac_halfwave = min_i >= -tol_A

    return {
        "ok": bool(no_negative_ac_halfwave),
        "reason": "ok" if no_negative_ac_halfwave else "negative_ac_halfwave_detected",
        "n_window": n_window,
        "required_persistence_s": float(persistence_s),
        "tol_A": float(tol_A),
        "I_window_min_A": min_i,
        "I_window_max_A": max_i,
        "I_window_mean_A": mean_i,
        "post_available_s": float(t_end_available - t0),
    }


def detect_acoff_event(
    df: pd.DataFrame,
    inv_row: pd.Series,
    vmax_result: Mapping[str, object],
) -> dict[str, object]:
    """
    Detect AC-off support for DCAC protocols.

    For pure DC:
    - t_AC_off_detected_s = NaN
    - method = not_applicable_DC_reference

    For DCAC:
    - use Vmax candidate as transition candidate
    - first try default persistence = 3 * T_AC
    - if default persistence cannot be satisfied, try low-frequency fallback
    """
    protocol_role = str(inv_row["protocol_role"])
    if protocol_role == "DC_reference":
        return {
            "idx_AC_off": np.nan,
            "t_AC_off_detected_s": np.nan,
            "AC_off_detection_method_used": "not_applicable_DC_reference",
            "AC_off_detection_status": "not_applicable",
            "AC_off_detection_reason": "pure_DC_reference_has_no_AC_off",
            "AC_off_post_available_s": np.nan,
            "AC_off_required_persistence_s": np.nan,
            "AC_off_window_min_I_A": np.nan,
            "AC_off_window_max_I_A": np.nan,
        }

    idx_vmax = vmax_result.get("idx_Vmax", np.nan)
    if not is_finite_number(idx_vmax):
        return {
            "idx_AC_off": np.nan,
            "t_AC_off_detected_s": np.nan,
            "AC_off_detection_method_used": "unresolved_no_Vmax_candidate",
            "AC_off_detection_status": "unresolved",
            "AC_off_detection_reason": "Vmax_not_detected",
            "AC_off_post_available_s": np.nan,
            "AC_off_required_persistence_s": np.nan,
            "AC_off_window_min_I_A": np.nan,
            "AC_off_window_max_I_A": np.nan,
        }

    idx_vmax = int(idx_vmax)

    ac_amp_A = float(inv_row["AC_C"]) * ONE_C_A
    frequency_hz = float(inv_row["frequency_Hz"])
    T_AC_s = compute_t_ac_s(frequency_hz)

    if not np.isfinite(T_AC_s):
        return {
            "idx_AC_off": np.nan,
            "t_AC_off_detected_s": np.nan,
            "AC_off_detection_method_used": "unresolved_invalid_frequency",
            "AC_off_detection_status": "unresolved",
            "AC_off_detection_reason": "invalid_frequency_Hz",
            "AC_off_post_available_s": np.nan,
            "AC_off_required_persistence_s": np.nan,
            "AC_off_window_min_I_A": np.nan,
            "AC_off_window_max_I_A": np.nan,
        }

    # Candidate search starts at Vmax index and allows a short post-Vmax delay.
    # No negative-lag candidate is used here.
    search_last = min(idx_vmax + 60, len(df) - 1)
    candidate_indices = range(idx_vmax, search_last + 1)

    # Default rule: 3 * T_AC
    default_persistence_s = AC_OFF_PERSISTENCE_N_T_AC * T_AC_s

    for idx_candidate in candidate_indices:
        support = ac_off_window_supports_no_sinusoid(
            df=df,
            idx_candidate=idx_candidate,
            persistence_s=default_persistence_s,
            ac_amp_A=ac_amp_A,
            voltage_window=None,
        )
        if support["ok"]:
            t_candidate = float(df["t_s"].iloc[idx_candidate])
            return {
                "idx_AC_off": int(idx_candidate),
                "t_AC_off_detected_s": t_candidate,
                "AC_off_detection_method_used": (
                    "priority2_current_envelope_default_3TAC_after_Vmax"
                ),
                "AC_off_detection_status": "detected",
                "AC_off_detection_reason": support["reason"],
                "AC_off_post_available_s": support.get("post_available_s", np.nan),
                "AC_off_required_persistence_s": support.get("required_persistence_s", np.nan),
                "AC_off_window_min_I_A": support.get("I_window_min_A", np.nan),
                "AC_off_window_max_I_A": support.get("I_window_max_A", np.nan),
            }

    # Fallback: low-frequency exception
    low_freq_persistence_s = AC_OFF_LOW_FREQUENCY_MIN_PERSISTENCE_S

    for idx_candidate in candidate_indices:
        support = ac_off_window_supports_no_sinusoid(
            df=df,
            idx_candidate=idx_candidate,
            persistence_s=low_freq_persistence_s,
            ac_amp_A=ac_amp_A,
            voltage_window=AC_OFF_LOW_FREQUENCY_VOLTAGE_WINDOW_V,
        )
        if support["ok"]:
            t_candidate = float(df["t_s"].iloc[idx_candidate])
            return {
                "idx_AC_off": int(idx_candidate),
                "t_AC_off_detected_s": t_candidate,
                "AC_off_detection_method_used": (
                    "priority2_low_frequency_exception_120s_voltage_window"
                ),
                "AC_off_detection_status": "detected",
                "AC_off_detection_reason": support["reason"],
                "AC_off_post_available_s": support.get("post_available_s", np.nan),
                "AC_off_required_persistence_s": support.get("required_persistence_s", np.nan),
                "AC_off_window_min_I_A": support.get("I_window_min_A", np.nan),
                "AC_off_window_max_I_A": support.get("I_window_max_A", np.nan),
            }

    # Final fallback: unresolved, not Vmax substitution.
    return {
        "idx_AC_off": np.nan,
        "t_AC_off_detected_s": np.nan,
        "AC_off_detection_method_used": "unresolved_current_envelope_detection_failed",
        "AC_off_detection_status": "unresolved",
        "AC_off_detection_reason": "no_candidate_satisfied_default_or_low_frequency_exception",
        "AC_off_post_available_s": np.nan,
        "AC_off_required_persistence_s": default_persistence_s,
        "AC_off_window_min_I_A": np.nan,
        "AC_off_window_max_I_A": np.nan,
    }


# -----------------------------------------------------------------------------
# 4.3 Run event audit
# -----------------------------------------------------------------------------

event_rows = []

for _, inv_row in df_inventory.iterrows():
    file_name = inv_row["file_name"]
    df_traj = TRAJ[file_name]

    vmax_result = detect_vmax_event(df_traj)
    acoff_result = detect_acoff_event(df_traj, inv_row, vmax_result)

    t_vmax = vmax_result["t_Vmax_detected_s"]
    t_acoff = acoff_result["t_AC_off_detected_s"]

    if is_finite_number(t_vmax) and is_finite_number(t_acoff):
        ac_off_lag_s = float(t_acoff) - float(t_vmax)
    else:
        ac_off_lag_s = np.nan

    row = {
        "file_name": file_name,
        "protocol_label": inv_row["protocol_label"],
        "protocol_role": inv_row["protocol_role"],
        "DC_C": inv_row["DC_C"],
        "AC_C": inv_row["AC_C"],
        "m_tau": inv_row["m_tau"],
        "frequency_Hz": inv_row["frequency_Hz"],

        **vmax_result,
        **acoff_result,

        "AC_off_lag_s": ac_off_lag_s,
    }

    event_rows.append(row)

df_event_audit = pd.DataFrame(event_rows)

# Add framework flags, but no Q-based segment assignment yet.
def event_framework_status(row: pd.Series) -> str:
    if row["protocol_role"] == "DC_reference":
        return "event_audit_ok_DC_reference"

    if row["Vmax_detection_status"] not in ["detected", "detected_without_deglitch"]:
        return "Vmax_unresolved"

    if row["AC_off_detection_status"] != "detected":
        return "AC_off_unresolved"

    if is_finite_number(row["AC_off_lag_s"]) and float(row["AC_off_lag_s"]) < 0:
        return SEGMENT_FRAMEWORK_AC_OFF_PRECEDES_VMAX

    return "event_audit_ok_DCAC"

df_event_audit["event_framework_status"] = df_event_audit.apply(event_framework_status, axis=1)

df_event_audit.to_csv(OUT_EVENT_AUDIT, index=False)

print(f"[OK] Wrote event / AC-off audit: {OUT_EVENT_AUDIT}")

display_cols = [
    "file_name",
    "protocol_label",
    "protocol_role",
    "t_Vmax_detected_s",
    "U_at_Vmax_detected_V",
    "Vmax_detection_method_used",
    "t_AC_off_detected_s",
    "AC_off_detection_method_used",
    "AC_off_lag_s",
    "AC_off_required_persistence_s",
    "AC_off_post_available_s",
    "AC_off_window_min_I_A",
    "AC_off_window_max_I_A",
    "event_framework_status",
]
print(df_event_audit[display_cols].to_string(index=False))


# -----------------------------------------------------------------------------
# 4.4 Hard guards
# -----------------------------------------------------------------------------

bad_event = df_event_audit[
    ~df_event_audit["event_framework_status"].isin([
        "event_audit_ok_DC_reference",
        "event_audit_ok_DCAC",
    ])
]

if len(bad_event) > 0:
    print("[warning] Event audit has unresolved / non-OK rows:")
    print(bad_event[display_cols].to_string(index=False))
    raise ValueError("Event / AC-off audit did not pass for all required trajectories.")

print("[OK] Cell 4 event / AC-off audit passed.")
print("[OK] No Q integration, segment assignment, Δt computation, or verdict performed in Cell 4.")

[OK] Wrote event / AC-off audit: /Users/louislu/pybamm-dcac-superimposed/data/day21A_step2_MJ1_event_acoff_audit.csv
                          file_name   protocol_label protocol_role  t_Vmax_detected_s  U_at_Vmax_detected_V              Vmax_detection_method_used  t_AC_off_detected_s                       AC_off_detection_method_used  AC_off_lag_s  AC_off_required_persistence_s  AC_off_post_available_s  AC_off_window_min_I_A  AC_off_window_max_I_A      event_framework_status
MJ1_0p3C_0p7C_0p1tau_NGU201_raw.csv 0.3C+0.7C 0.1tau          DCAC             9419.0              4.202711 priority3_first_V_ge_4p2V_with_deglitch               9419.0 priority2_current_envelope_default_3TAC_after_Vmax           0.0                      20.923007                   4011.0               2.566196               3.400047         event_audit_ok_DCAC
 MJ1_0p3C_0p7C_10tau_NGU201_raw.csv  0.3C+0.7C 10tau          DCAC             8556.5              4.200239 priority3_first_V_ge_4p2V_with_deglitch        

In [14]:
# Cell 5 — Strict-net Q integration and final-Q consistency audit
#
# Purpose:
# - Compute strict-net Q_net(t) for each retained trajectory
# - Preserve signed current, no rectification, no cummax
# - Audit local non-monotonicity caused by DCAC negative-current intervals
# - Compute Q_final for DC and each DCAC protocol
# - Compute final-Q consistency status
#
# Explicitly NOT done here:
# - No Vmax charge extraction yet
# - No Segment A/B/D assignment
# - No Δt_raw / Δt_geom / Δt_resid
# - No mechanism verdict

OUT_Q_SUMMARY = DATA_DIR / "day21A_step3_MJ1_Q_integration_summary.csv"
OUT_FINALQ_PAIR_AUDIT = DATA_DIR / "day21A_step3_MJ1_finalQ_pair_audit.csv"


# -----------------------------------------------------------------------------
# 5.1 Strict-net signed trapezoidal integration
# -----------------------------------------------------------------------------

def integrate_strict_net_Q_Ah(
    t_s: np.ndarray,
    i_q_A: np.ndarray,
) -> np.ndarray:
    """
    Strict-net signed trapezoidal integration.

    Rules:
    - Use signed charge-positive current I_Q_A.
    - Do not rectify.
    - Do not apply cummax.
    - Preserve local decreases in Q caused by negative-current intervals.
    - Q_net(t=0) = 0.
    """
    t = np.asarray(t_s, dtype=float)
    i = np.asarray(i_q_A, dtype=float)

    if len(t) != len(i):
        raise ValueError("time and current arrays must have the same length")

    q = np.zeros(len(t), dtype=float)
    if len(t) == 0:
        return q

    for k in range(1, len(t)):
        if (
            np.isfinite(t[k])
            and np.isfinite(t[k - 1])
            and np.isfinite(i[k])
            and np.isfinite(i[k - 1])
        ):
            dt_h = (t[k] - t[k - 1]) / 3600.0
            if dt_h > 0:
                q[k] = q[k - 1] + 0.5 * (i[k] + i[k - 1]) * dt_h
            else:
                q[k] = q[k - 1]
        else:
            q[k] = q[k - 1]

    return q


# -----------------------------------------------------------------------------
# 5.2 Apply integration to all loaded trajectories
# -----------------------------------------------------------------------------

q_summary_rows = []

for file_name, df_traj in TRAJ.items():
    t_s = df_traj["t_s"].to_numpy(dtype=float)
    i_q = df_traj["I_Q_A"].to_numpy(dtype=float)

    q_net_Ah = integrate_strict_net_Q_Ah(t_s, i_q)

    # Attach Q to trajectory object for later cells.
    # This is still only Q integration, not Δt analysis.
    TRAJ[file_name] = df_traj.copy()
    TRAJ[file_name]["Q_net_Ah"] = q_net_Ah

    dq = np.diff(q_net_Ah)
    finite_dq = dq[np.isfinite(dq)]

    n_q_decrease = int(np.sum(finite_dq < -1e-12)) if len(finite_dq) else 0
    q_decrease_fraction = (
        float(n_q_decrease / len(finite_dq)) if len(finite_dq) else np.nan
    )

    q_final = float(q_net_Ah[-1]) if len(q_net_Ah) else np.nan
    q_max = float(np.nanmax(q_net_Ah)) if len(q_net_Ah) else np.nan
    q_min = float(np.nanmin(q_net_Ah)) if len(q_net_Ah) else np.nan

    inv_row = df_inventory.loc[df_inventory["file_name"] == file_name].iloc[0]

    q_summary_rows.append({
        "file_name": file_name,
        "protocol_label": inv_row["protocol_label"],
        "protocol_role": inv_row["protocol_role"],
        "DC_C": inv_row["DC_C"],
        "AC_C": inv_row["AC_C"],
        "m_tau": inv_row["m_tau"],
        "frequency_Hz": inv_row["frequency_Hz"],
        "n_retained_rows": len(TRAJ[file_name]),
        "t_final_s": float(TRAJ[file_name]["t_s"].iloc[-1]),
        "Q_final_Ah": q_final,
        "Q_final_mAh": q_final * 1000.0 if np.isfinite(q_final) else np.nan,
        "Q_max_Ah": q_max,
        "Q_min_Ah": q_min,
        "Q_decrease_count": n_q_decrease,
        "Q_decrease_fraction": q_decrease_fraction,
        "I_Q_min_A": float(np.nanmin(i_q)),
        "I_Q_max_A": float(np.nanmax(i_q)),
        "I_Q_mean_A": float(np.nanmean(i_q)),
    })

df_q_summary = pd.DataFrame(q_summary_rows)

df_q_summary.to_csv(OUT_Q_SUMMARY, index=False)

print(f"[OK] Wrote Q integration summary: {OUT_Q_SUMMARY}")

display_cols = [
    "file_name",
    "protocol_label",
    "protocol_role",
    "t_final_s",
    "Q_final_Ah",
    "Q_final_mAh",
    "Q_max_Ah",
    "Q_decrease_count",
    "Q_decrease_fraction",
    "I_Q_min_A",
    "I_Q_max_A",
    "I_Q_mean_A",
]
print(df_q_summary[display_cols].to_string(index=False))


# -----------------------------------------------------------------------------
# 5.3 Final-Q consistency audit against DC reference
# -----------------------------------------------------------------------------

dc_rows = df_q_summary[df_q_summary["protocol_role"] == "DC_reference"]
if len(dc_rows) != 1:
    raise ValueError(f"Expected exactly one DC reference in Q summary, found {len(dc_rows)}")

dc_ref = dc_rows.iloc[0]
q_final_dc = float(dc_ref["Q_final_Ah"])

pair_rows = []

for _, row in df_q_summary.iterrows():
    if row["protocol_role"] != "DCAC":
        continue

    q_final_dcac = float(row["Q_final_Ah"])
    diff = q_final_dc - q_final_dcac
    abs_diff = abs(diff)

    common = compute_common_anchors(
        q_final_dc_ah=q_final_dc,
        q_final_dcac_ah=q_final_dcac,
        q_nom_ah=Q_NOM_AH,
    )

    pair_rows.append({
        "protocol_pair": f"{dc_ref['protocol_label']} vs {row['protocol_label']}",
        "protocol_label_DC": dc_ref["protocol_label"],
        "protocol_label_DCAC": row["protocol_label"],
        "Q_nom_Ah": Q_NOM_AH,
        "Q_final_DC_Ah": q_final_dc,
        "Q_final_DCAC_Ah": q_final_dcac,
        "Q_final_diff_Ah": diff,
        "Q_final_abs_diff_Ah": abs_diff,
        "Q_final_diff_mAh": diff * 1000.0,
        "Q_final_abs_diff_mAh": abs_diff * 1000.0,
        "Q_final_diff_status": final_q_status(q_final_dc, q_final_dcac),
        "Q80_nominal_Ah": Q80_NOMINAL_AH,
        "Q90_nominal_Ah": Q90_NOMINAL_AH,
        "Q80_nominal_fraction_of_Q_nom": Q80_NOMINAL_FRACTION_OF_Q_NOM,
        "Q90_nominal_fraction_of_Q_nom": Q90_NOMINAL_FRACTION_OF_Q_NOM,
        "Q80_common_Ah": common["Q80_common_Ah"],
        "Q90_common_Ah": common["Q90_common_Ah"],
        "Q80_common_fraction_of_Q_nom": common["Q80_common_fraction_of_Q_nom"],
        "Q90_common_fraction_of_Q_nom": common["Q90_common_fraction_of_Q_nom"],
        "Q_common_final_Ah_internal": common["Q_common_final_Ah"],
    })

df_finalq_pairs = pd.DataFrame(pair_rows)

df_finalq_pairs.to_csv(OUT_FINALQ_PAIR_AUDIT, index=False)

print(f"[OK] Wrote final-Q pair audit: {OUT_FINALQ_PAIR_AUDIT}")

display_cols = [
    "protocol_pair",
    "Q_final_DC_Ah",
    "Q_final_DCAC_Ah",
    "Q_final_diff_mAh",
    "Q_final_abs_diff_mAh",
    "Q_final_diff_status",
    "Q80_common_Ah",
    "Q90_common_Ah",
    "Q80_common_fraction_of_Q_nom",
    "Q90_common_fraction_of_Q_nom",
]
print(df_finalq_pairs[display_cols].to_string(index=False))


# -----------------------------------------------------------------------------
# 5.4 Hard guards
# -----------------------------------------------------------------------------

if not np.isfinite(q_final_dc) or q_final_dc <= 0:
    raise ValueError("DC reference final Q is invalid.")

for _, row in df_q_summary.iterrows():
    if not np.isfinite(row["Q_final_Ah"]) or row["Q_final_Ah"] <= 0:
        raise ValueError(f"{row['file_name']}: invalid Q_final_Ah.")

# This is an audit, not a failure condition.
n_mismatch = int((df_finalq_pairs["Q_final_diff_status"] == FINAL_Q_MISMATCH_WARNING).sum())
if n_mismatch > 0:
    print(f"[warning] Final-Q mismatch warning in {n_mismatch} DCAC pair(s).")
    print("          Common anchors remain valid as shared reachable absolute-Q targets.")
    print("          Later verdicts using common anchors must carry asymmetric_final_Q caveat.")

print("[OK] Cell 5 strict-net Q integration and final-Q audit completed.")
print("[OK] No event charge extraction, segment assignment, Δt computation, or verdict performed in Cell 5.")

[OK] Wrote Q integration summary: /Users/louislu/pybamm-dcac-superimposed/data/day21A_step3_MJ1_Q_integration_summary.csv
                          file_name   protocol_label protocol_role  t_final_s  Q_final_Ah  Q_final_mAh  Q_max_Ah  Q_decrease_count  Q_decrease_fraction  I_Q_min_A  I_Q_max_A  I_Q_mean_A
MJ1_0p3C_0p7C_0p1tau_NGU201_raw.csv 0.3C+0.7C 0.1tau          DCAC    13430.0    3.339662  3339.661615  3.339662              3216             0.239714  -1.360000   3.400048    0.897529
 MJ1_0p3C_0p7C_10tau_NGU201_raw.csv  0.3C+0.7C 10tau          DCAC    12737.3    3.312244  3312.244358  3.312244              3013             0.236611  -1.360020   3.399960    0.942567
  MJ1_0p3C_0p7C_1tau_NGU201_raw.csv   0.3C+0.7C 1tau          DCAC    12982.1    3.254487  3254.487347  3.254487              3213             0.247554  -1.359972   3.400057    0.907264
         MJ1_0p3C_DC_NGU201_raw.csv          0.3C DC  DC_reference    13426.0    3.259324  3259.324059  3.259324                 0    

In [15]:
# Cell 6 — Event charge extraction and Q-anchor segment assignment
#
# Purpose:
# - Extract Q_net at Vmax and Segment-B start events
# - Compute Q_Vmax_DC_Ah, Q_Vmax_DCAC_Ah, Q_segmentB_start_Ah
# - Assign Q80/Q90 nominal and common anchors to A/B/D/outside
# - Audit ordering status and segment framework status
#
# Explicitly NOT done here:
# - No Δt_raw computation
# - No Δt_geom computation
# - No Δt_resid computation
# - No mechanism verdict

OUT_SEGMENT_ASSIGNMENT = DATA_DIR / "day21A_step4_MJ1_segment_assignment.csv"

# -----------------------------------------------------------------------------
# 6.1 Load event audit and final-Q pair audit
# -----------------------------------------------------------------------------

if not OUT_EVENT_AUDIT.exists():
    raise FileNotFoundError(
        f"Event audit file not found: {OUT_EVENT_AUDIT}\n"
        "Run Cell 4 first."
    )

if not OUT_FINALQ_PAIR_AUDIT.exists():
    raise FileNotFoundError(
        f"Final-Q pair audit file not found: {OUT_FINALQ_PAIR_AUDIT}\n"
        "Run Cell 5 first."
    )

df_event_audit = pd.read_csv(OUT_EVENT_AUDIT)
df_finalq_pairs = pd.read_csv(OUT_FINALQ_PAIR_AUDIT)

print(f"[OK] Loaded event audit: {OUT_EVENT_AUDIT}")
print(f"[OK] Loaded final-Q pair audit: {OUT_FINALQ_PAIR_AUDIT}")


# -----------------------------------------------------------------------------
# 6.2 Helpers
# -----------------------------------------------------------------------------

def interp_q_at_time(df_traj: pd.DataFrame, t_target_s: float) -> float:
    """
    Interpolate Q_net_Ah at a target time.

    This is not first-passage Δt analysis.
    It only maps already-detected event times to Q coordinates.
    """
    if not is_finite_number(t_target_s):
        return np.nan

    if "Q_net_Ah" not in df_traj.columns:
        raise ValueError("Trajectory does not contain Q_net_Ah. Run Cell 5 first.")

    t = df_traj["t_s"].to_numpy(dtype=float)
    q = df_traj["Q_net_Ah"].to_numpy(dtype=float)

    finite = np.isfinite(t) & np.isfinite(q)
    if finite.sum() < 2:
        return np.nan

    t_f = t[finite]
    q_f = q[finite]

    if t_target_s < t_f[0] or t_target_s > t_f[-1]:
        return np.nan

    return float(np.interp(float(t_target_s), t_f, q_f))


def get_single_event_row(file_name: str) -> pd.Series:
    rows = df_event_audit[df_event_audit["file_name"] == file_name]
    if len(rows) != 1:
        raise ValueError(f"Expected exactly one event row for {file_name}, found {len(rows)}")
    return rows.iloc[0]


def get_single_inventory_row(file_name: str) -> pd.Series:
    rows = df_inventory[df_inventory["file_name"] == file_name]
    if len(rows) != 1:
        raise ValueError(f"Expected exactly one inventory row for {file_name}, found {len(rows)}")
    return rows.iloc[0]


def get_dc_reference_file_name() -> str:
    rows = df_inventory[df_inventory["candidate_for_DC_reference"].map(parse_bool_strict)]
    if len(rows) != 1:
        raise ValueError(f"Expected exactly one DC reference, found {len(rows)}")
    return str(rows.iloc[0]["file_name"])


# -----------------------------------------------------------------------------
# 6.3 Extract DC reference Vmax charge
# -----------------------------------------------------------------------------

dc_file = get_dc_reference_file_name()
dc_inv = get_single_inventory_row(dc_file)
dc_event = get_single_event_row(dc_file)
dc_traj = TRAJ[dc_file]

t_vmax_dc_s = float(dc_event["t_Vmax_detected_s"])
q_vmax_dc_ah = interp_q_at_time(dc_traj, t_vmax_dc_s)

if not is_finite_number(q_vmax_dc_ah):
    raise ValueError("Could not extract Q_Vmax_DC_Ah.")

print(f"[OK] DC reference file = {dc_file}")
print(f"[OK] t_Vmax_DC_s = {t_vmax_dc_s:.3f}")
print(f"[OK] Q_Vmax_DC_Ah = {q_vmax_dc_ah:.6f}")


# -----------------------------------------------------------------------------
# 6.4 Pairwise DC-vs-DCAC event charge and anchor segment assignment
# -----------------------------------------------------------------------------

segment_rows = []

dc_q_summary = df_q_summary[df_q_summary["file_name"] == dc_file].iloc[0]
q_final_dc = float(dc_q_summary["Q_final_Ah"])

for _, inv_row in df_inventory.iterrows():
    if not parse_bool_strict(inv_row["candidate_for_DCAC"]):
        continue

    dcac_file = str(inv_row["file_name"])
    dcac_event = get_single_event_row(dcac_file)
    dcac_traj = TRAJ[dcac_file]

    q_summary_dcac = df_q_summary[df_q_summary["file_name"] == dcac_file].iloc[0]
    q_final_dcac = float(q_summary_dcac["Q_final_Ah"])

    t_vmax_dcac_s = float(dcac_event["t_Vmax_detected_s"])
    q_vmax_dcac_ah = interp_q_at_time(dcac_traj, t_vmax_dcac_s)

    # Segment-B start = min(t_Vmax_DCAC, t_ACoff_DCAC) if both finite.
    # Current event audit gives lag = 0 for all current DCAC cases.
    t_acoff_dcac_s = dcac_event["t_AC_off_detected_s"]

    if is_finite_number(t_acoff_dcac_s):
        t_segmentB_start_s = min(float(t_vmax_dcac_s), float(t_acoff_dcac_s))
    else:
        t_segmentB_start_s = float(t_vmax_dcac_s)

    q_segmentB_start_ah = interp_q_at_time(dcac_traj, t_segmentB_start_s)

    q_ordering_status = q_boundary_ordering_status(
        q_segmentB_start_ah=q_segmentB_start_ah,
        q_vmax_dc_ah=q_vmax_dc_ah,
    )
    segment_framework_status = segment_framework_status_from_ordering(q_ordering_status)

    if is_finite_number(dcac_event["AC_off_lag_s"]) and float(dcac_event["AC_off_lag_s"]) < 0:
        segment_framework_status = SEGMENT_FRAMEWORK_AC_OFF_PRECEDES_VMAX

    final_status = final_q_status(q_final_dc, q_final_dcac)

    common = compute_common_anchors(
        q_final_dc_ah=q_final_dc,
        q_final_dcac_ah=q_final_dcac,
        q_nom_ah=Q_NOM_AH,
    )

    q80_nominal_ah = Q80_NOMINAL_AH
    q90_nominal_ah = Q90_NOMINAL_AH
    q80_common_ah = common["Q80_common_Ah"]
    q90_common_ah = common["Q90_common_Ah"]

    q80_nominal_segment = assign_segment_by_q(
        q80_nominal_ah,
        q_segmentB_start_ah,
        q_vmax_dc_ah,
        q_final_dc,
        q_final_dcac,
    )
    q90_nominal_segment = assign_segment_by_q(
        q90_nominal_ah,
        q_segmentB_start_ah,
        q_vmax_dc_ah,
        q_final_dc,
        q_final_dcac,
    )
    q80_common_segment = assign_segment_by_q(
        q80_common_ah,
        q_segmentB_start_ah,
        q_vmax_dc_ah,
        q_final_dc,
        q_final_dcac,
    )
    q90_common_segment = assign_segment_by_q(
        q90_common_ah,
        q_segmentB_start_ah,
        q_vmax_dc_ah,
        q_final_dc,
        q_final_dcac,
    )

    row = {
        "protocol_pair": f"{dc_inv['protocol_label']} vs {inv_row['protocol_label']}",
        "protocol_label_DC": dc_inv["protocol_label"],
        "protocol_label_DCAC": inv_row["protocol_label"],
        "file_name_DC": dc_file,
        "file_name_DCAC": dcac_file,

        "Q_nom_Ah": Q_NOM_AH,

        "Q_final_DC_Ah": q_final_dc,
        "Q_final_DCAC_Ah": q_final_dcac,
        "Q_final_diff_Ah": q_final_dc - q_final_dcac,
        "Q_final_diff_status": final_status,

        "t_Vmax_DC_s": t_vmax_dc_s,
        "t_Vmax_DCAC_s": t_vmax_dcac_s,
        "t_AC_off_DCAC_s": float(t_acoff_dcac_s) if is_finite_number(t_acoff_dcac_s) else np.nan,
        "AC_off_lag_s": float(dcac_event["AC_off_lag_s"]) if is_finite_number(dcac_event["AC_off_lag_s"]) else np.nan,
        "t_segmentB_start_s": t_segmentB_start_s,

        "Q_Vmax_DC_Ah": q_vmax_dc_ah,
        "Q_Vmax_DCAC_Ah": q_vmax_dcac_ah,
        "Q_segmentB_start_Ah": q_segmentB_start_ah,
        "segment_A_Q_hi_definition": "Q_segmentB_start_Ah",

        "Q_Vmax_shift_Ah": q_vmax_dc_ah - q_vmax_dcac_ah,
        "Q_Vmax_ordering_status": q_ordering_status,
        "segment_framework_status": segment_framework_status,

        "Q80_nominal_Ah": q80_nominal_ah,
        "Q90_nominal_Ah": q90_nominal_ah,
        "Q80_common_Ah": q80_common_ah,
        "Q90_common_Ah": q90_common_ah,

        "Q80_nominal_fraction_of_Q_nom": Q80_NOMINAL_FRACTION_OF_Q_NOM,
        "Q90_nominal_fraction_of_Q_nom": Q90_NOMINAL_FRACTION_OF_Q_NOM,
        "Q80_common_fraction_of_Q_nom": common["Q80_common_fraction_of_Q_nom"],
        "Q90_common_fraction_of_Q_nom": common["Q90_common_fraction_of_Q_nom"],

        "Q80_nominal_segment": q80_nominal_segment,
        "Q90_nominal_segment": q90_nominal_segment,
        "Q80_common_segment": q80_common_segment,
        "Q90_common_segment": q90_common_segment,

        # Residual-grid lower bound is global fixed contract value.
        "segment_A_Q_lo_Ah": SEGMENT_A_Q_LO_AH,
        "segment_A_Q_hi_Ah": q_segmentB_start_ah,
        "segment_A_Q_grid_count": segment_A_grid_count(
            segment_A_Q_lo_Ah=SEGMENT_A_Q_LO_AH,
            segment_A_Q_hi_Ah=q_segmentB_start_ah,
        ),
    }

    segment_rows.append(row)

df_segment_assignment = pd.DataFrame(segment_rows)

df_segment_assignment.to_csv(OUT_SEGMENT_ASSIGNMENT, index=False)

print(f"[OK] Wrote segment assignment audit: {OUT_SEGMENT_ASSIGNMENT}")

display_cols = [
    "protocol_pair",
    "Q_final_diff_status",
    "t_Vmax_DC_s",
    "t_Vmax_DCAC_s",
    "Q_Vmax_DC_Ah",
    "Q_Vmax_DCAC_Ah",
    "Q_Vmax_shift_Ah",
    "Q_segmentB_start_Ah",
    "Q_Vmax_ordering_status",
    "segment_framework_status",
    "Q80_nominal_Ah",
    "Q80_nominal_segment",
    "Q90_nominal_Ah",
    "Q90_nominal_segment",
    "Q80_common_Ah",
    "Q80_common_segment",
    "Q90_common_Ah",
    "Q90_common_segment",
    "segment_A_Q_grid_count",
]
print(df_segment_assignment[display_cols].to_string(index=False))


# -----------------------------------------------------------------------------
# 6.5 Hard guards
# -----------------------------------------------------------------------------

bad_framework = df_segment_assignment[
    df_segment_assignment["segment_framework_status"].isin([
        SEGMENT_FRAMEWORK_ORDERING_VIOLATED,
        SEGMENT_FRAMEWORK_AC_OFF_PRECEDES_VMAX,
        SEGMENT_FRAMEWORK_UNRESOLVED,
    ])
]

if len(bad_framework) > 0:
    print("[warning] Segment framework has non-OK rows:")
    print(bad_framework[display_cols].to_string(index=False))
    raise ValueError("Segment framework status is not valid for all pairs.")

if (df_segment_assignment["segment_A_Q_grid_count"] < Q_GRID_MIN_COUNT_SEGMENT_A).any():
    print("[warning] At least one pair has low Segment-A grid count.")
    print("          This does not stop Cell 6, but p95 residual guard may be unavailable later.")

print("[OK] Cell 6 event-charge extraction and anchor segment assignment completed.")
print("[OK] No Δt_raw, Δt_geom, Δt_resid, or verdict performed in Cell 6.")

[OK] Loaded event audit: /Users/louislu/pybamm-dcac-superimposed/data/day21A_step2_MJ1_event_acoff_audit.csv
[OK] Loaded final-Q pair audit: /Users/louislu/pybamm-dcac-superimposed/data/day21A_step3_MJ1_finalQ_pair_audit.csv
[OK] DC reference file = MJ1_0p3C_DC_NGU201_raw.csv
[OK] t_Vmax_DC_s = 10458.000
[OK] Q_Vmax_DC_Ah = 2.962785
[OK] Wrote segment assignment audit: /Users/louislu/pybamm-dcac-superimposed/data/day21A_step4_MJ1_segment_assignment.csv
              protocol_pair      Q_final_diff_status  t_Vmax_DC_s  t_Vmax_DCAC_s  Q_Vmax_DC_Ah  Q_Vmax_DCAC_Ah  Q_Vmax_shift_Ah  Q_segmentB_start_Ah                 Q_Vmax_ordering_status segment_framework_status  Q80_nominal_Ah                    Q80_nominal_segment  Q90_nominal_Ah Q90_nominal_segment  Q80_common_Ah                     Q80_common_segment  Q90_common_Ah                     Q90_common_segment  segment_A_Q_grid_count
0.3C DC vs 0.3C+0.7C 0.1tau final_Q_mismatch_warning      10458.0         9419.0      2.962785        2.674

In [16]:
# Cell 7 — First-passage Δt and Segment-A residual audit
#
# Purpose:
# - Compute first-passage raw Δt(Q) on anchors and segment grids
# - Compute prescribed-current geometry Δt_geom(Q)
# - Compute Δt_resid(Q) only in Segment A
# - Audit Segment-A residual against the pre-registered MJ1 floor
# - Compute Segment B/D raw Δt statistics
#
# Explicitly NOT done here:
# - No final mechanism verdict
# - No unified MJ1–PyBaMM verdict table

OUT_DTQ_AUDIT_LONG = DATA_DIR / "day21A_step5_MJ1_dtQ_segment_audit_long.csv"
OUT_DTQ_SUMMARY = DATA_DIR / "day21A_step5_MJ1_dtQ_segment_summary.csv"


# -----------------------------------------------------------------------------
# 7.1 Load segment assignment
# -----------------------------------------------------------------------------

if not OUT_SEGMENT_ASSIGNMENT.exists():
    raise FileNotFoundError(
        f"Segment assignment file not found: {OUT_SEGMENT_ASSIGNMENT}\n"
        "Run Cell 6 first."
    )

df_segment_assignment = pd.read_csv(OUT_SEGMENT_ASSIGNMENT)

print(f"[OK] Loaded segment assignment: {OUT_SEGMENT_ASSIGNMENT}")


# -----------------------------------------------------------------------------
# 7.2 First-passage helpers
# -----------------------------------------------------------------------------

def first_passage_time_from_Q(
    q_target_Ah: float,
    t_s: np.ndarray,
    q_Ah: np.ndarray,
) -> float:
    """
    First-passage time: first t where Q(t) >= q_target.
    No monotonic enforcement is applied.
    """
    if not is_finite_number(q_target_Ah):
        return np.nan

    t = np.asarray(t_s, dtype=float)
    q = np.asarray(q_Ah, dtype=float)

    finite = np.isfinite(t) & np.isfinite(q)
    if finite.sum() < 2:
        return np.nan

    t_f = t[finite]
    q_f = q[finite]

    hit = np.where(q_f >= float(q_target_Ah))[0]
    if len(hit) == 0:
        return np.nan

    idx = int(hit[0])

    # If first sample already hits, return its time.
    if idx == 0:
        return float(t_f[idx])

    # Linear interpolation between previous and hit sample.
    q0, q1 = q_f[idx - 1], q_f[idx]
    t0, t1 = t_f[idx - 1], t_f[idx]

    if not np.isfinite(q0) or not np.isfinite(q1) or q1 == q0:
        return float(t1)

    frac = (float(q_target_Ah) - q0) / (q1 - q0)
    frac = float(np.clip(frac, 0.0, 1.0))
    return float(t0 + frac * (t1 - t0))


def prescribed_geometry_Q_Ah(
    t_s: np.ndarray,
    I_DC_A: float,
    I_AC_A: float,
    frequency_Hz: float,
    phase_rad: float = 0.0,
) -> np.ndarray:
    """
    Compute prescribed geometry charge Q_geom(t) for Segment-A residual.

    Charge-positive convention:
    I_geom(t) = I_DC + I_AC * sin(2π f_Hz t + phase_rad)

    Strict signed integration is used.
    """
    t = np.asarray(t_s, dtype=float)
    if len(t) == 0:
        return np.array([], dtype=float)

    omega = 2.0 * np.pi * float(frequency_Hz)

    if float(frequency_Hz) == 0.0 or float(I_AC_A) == 0.0:
        i_geom = np.full(len(t), float(I_DC_A), dtype=float)
    else:
        i_geom = float(I_DC_A) + float(I_AC_A) * np.sin(omega * t + float(phase_rad))

    return integrate_strict_net_Q_Ah(t, i_geom)


def estimate_geometry_phase_rad(
    df_traj: pd.DataFrame,
    I_DC_A: float,
    I_AC_A: float,
    frequency_Hz: float,
    t_acoff_candidate_s: float,
) -> dict[str, object]:
    """
    Estimate geometry phase by least-squares grid search over phi.

    Contract model:
    I_Q(t) = I_DC + I_AC sin(2π f_Hz t + phi)

    Fixed parameters:
    I_DC, I_AC, f_Hz

    Fit window:
    min(5*T_AC, t_ACoff_candidate)

    If the fit window contains < 1 full AC period, phase is unresolved.
    """
    if float(I_AC_A) <= 0 or float(frequency_Hz) <= 0:
        return {
            "geometry_phase_offset_rad": 0.0,
            "geometry_phase_offset_s": 0.0,
            "geometry_phase_reference_status": GEOM_PHASE_VERIFIED,
            "geometry_phase_fit_rmse_A": 0.0,
            "geometry_phase_fit_window_s": np.nan,
        }

    T_AC_s = compute_t_ac_s(frequency_Hz)
    fit_window_s = min(GEOMETRY_PHASE_FIT_N_T_AC * T_AC_s, float(t_acoff_candidate_s))

    if fit_window_s < GEOMETRY_PHASE_MIN_REQUIRED_CYCLES * T_AC_s:
        return {
            "geometry_phase_offset_rad": np.nan,
            "geometry_phase_offset_s": np.nan,
            "geometry_phase_reference_status": GEOM_PHASE_UNRESOLVED,
            "geometry_phase_fit_rmse_A": np.nan,
            "geometry_phase_fit_window_s": fit_window_s,
        }

    t = df_traj["t_s"].to_numpy(dtype=float)
    i = df_traj["I_Q_A"].to_numpy(dtype=float)

    mask = (
        np.isfinite(t)
        & np.isfinite(i)
        & (t >= 0)
        & (t <= fit_window_s)
    )

    if mask.sum() < 10:
        return {
            "geometry_phase_offset_rad": np.nan,
            "geometry_phase_offset_s": np.nan,
            "geometry_phase_reference_status": GEOM_PHASE_UNRESOLVED,
            "geometry_phase_fit_rmse_A": np.nan,
            "geometry_phase_fit_window_s": fit_window_s,
        }

    tt = t[mask]
    ii = i[mask]

    # Grid-search phi is sufficient and deterministic.
    phi_grid = np.linspace(-np.pi, np.pi, 2001)
    omega = 2.0 * np.pi * float(frequency_Hz)

    best_phi = np.nan
    best_rmse = np.inf

    for phi in phi_grid:
        pred = float(I_DC_A) + float(I_AC_A) * np.sin(omega * tt + phi)
        rmse = float(np.sqrt(np.mean((ii - pred) ** 2)))
        if rmse < best_rmse:
            best_rmse = rmse
            best_phi = float(phi)

    phase_offset_s = best_phi / omega if omega > 0 else np.nan

    # If phase is approximately zero, classify as verified.
    if abs(best_phi) <= 0.05:
        status = GEOM_PHASE_VERIFIED
    else:
        status = GEOM_PHASE_ESTIMATED

    return {
        "geometry_phase_offset_rad": best_phi,
        "geometry_phase_offset_s": phase_offset_s,
        "geometry_phase_reference_status": status,
        "geometry_phase_fit_rmse_A": best_rmse,
        "geometry_phase_fit_window_s": fit_window_s,
    }


def make_fixed_Q_grid(q_lo_Ah: float, q_hi_Ah: float, step_Ah: float) -> np.ndarray:
    """
    Fixed Q-grid from first grid value >= q_lo to last grid value <= q_hi.
    """
    if not all(is_finite_number(x) for x in [q_lo_Ah, q_hi_Ah, step_Ah]):
        return np.array([], dtype=float)

    if q_hi_Ah < q_lo_Ah or step_Ah <= 0:
        return np.array([], dtype=float)

    start = np.ceil(float(q_lo_Ah) / step_Ah) * step_Ah
    stop = np.floor(float(q_hi_Ah) / step_Ah) * step_Ah

    if stop < start:
        return np.array([], dtype=float)

    n = int(round((stop - start) / step_Ah)) + 1
    return start + step_Ah * np.arange(n)


# -----------------------------------------------------------------------------
# 7.3 Build pairwise Δt(Q) audits
# -----------------------------------------------------------------------------

dtq_rows = []
summary_rows = []

dc_file = get_dc_reference_file_name()
dc_traj = TRAJ[dc_file]

t_dc = dc_traj["t_s"].to_numpy(dtype=float)
q_dc = dc_traj["Q_net_Ah"].to_numpy(dtype=float)

for _, seg_row in df_segment_assignment.iterrows():
    dcac_file = seg_row["file_name_DCAC"]
    dcac_traj = TRAJ[dcac_file]

    inv_dcac = get_single_inventory_row(dcac_file)

    t_dcac = dcac_traj["t_s"].to_numpy(dtype=float)
    q_dcac = dcac_traj["Q_net_Ah"].to_numpy(dtype=float)

    I_DC_A = float(inv_dcac["DC_C"]) * ONE_C_A
    I_AC_A = float(inv_dcac["AC_C"]) * ONE_C_A
    frequency_hz = float(inv_dcac["frequency_Hz"])

    phase_result = estimate_geometry_phase_rad(
        df_traj=dcac_traj,
        I_DC_A=I_DC_A,
        I_AC_A=I_AC_A,
        frequency_Hz=frequency_hz,
        t_acoff_candidate_s=float(seg_row["t_segmentB_start_s"]),
    )

    phase_rad = phase_result["geometry_phase_offset_rad"]
    if not is_finite_number(phase_rad):
        phase_rad = 0.0

    # Geometry reference trajectories on the measured time grids.
    q_geom_dc = prescribed_geometry_Q_Ah(
        t_s=t_dc,
        I_DC_A=I_DC_A,
        I_AC_A=0.0,
        frequency_Hz=0.0,
        phase_rad=0.0,
    )

    q_geom_dcac = prescribed_geometry_Q_Ah(
        t_s=t_dcac,
        I_DC_A=I_DC_A,
        I_AC_A=I_AC_A,
        frequency_Hz=frequency_hz,
        phase_rad=phase_rad,
    )

    q_seg_A_lo = SEGMENT_A_Q_LO_AH
    q_seg_A_hi = float(seg_row["segment_A_Q_hi_Ah"])
    q_seg_B_lo = q_seg_A_hi
    q_seg_B_hi = float(seg_row["Q_Vmax_DC_Ah"])
    q_seg_D_lo = q_seg_B_hi
    q_seg_D_hi = min(float(seg_row["Q_final_DC_Ah"]), float(seg_row["Q_final_DCAC_Ah"]))

    # Segment grids
    grid_A = make_fixed_Q_grid(q_seg_A_lo, q_seg_A_hi, Q_GRID_STEP_AH)
    grid_B = make_fixed_Q_grid(q_seg_B_lo + Q_GRID_STEP_AH, q_seg_B_hi, Q_GRID_STEP_AH)
    grid_D = make_fixed_Q_grid(q_seg_D_lo + Q_GRID_STEP_AH, q_seg_D_hi, Q_GRID_STEP_AH)

    # Anchor targets
    anchors = {
        "Q80_nominal": float(seg_row["Q80_nominal_Ah"]),
        "Q90_nominal": float(seg_row["Q90_nominal_Ah"]),
        "Q80_common": float(seg_row["Q80_common_Ah"]),
        "Q90_common": float(seg_row["Q90_common_Ah"]),
    }

    pair_id = seg_row["protocol_pair"]

    def add_dtq_row(q_target, q_label, segment_label, is_anchor):
        t_dc_q = first_passage_time_from_Q(q_target, t_dc, q_dc)
        t_dcac_q = first_passage_time_from_Q(q_target, t_dcac, q_dcac)
        dt_raw = t_dc_q - t_dcac_q if is_finite_number(t_dc_q) and is_finite_number(t_dcac_q) else np.nan

        if segment_label == SEGMENT_A:
            t_geom_dc_q = first_passage_time_from_Q(q_target, t_dc, q_geom_dc)
            t_geom_dcac_q = first_passage_time_from_Q(q_target, t_dcac, q_geom_dcac)
            dt_geom = (
                t_geom_dc_q - t_geom_dcac_q
                if is_finite_number(t_geom_dc_q) and is_finite_number(t_geom_dcac_q)
                else np.nan
            )
            dt_resid = (
                dt_raw - dt_geom
                if is_finite_number(dt_raw) and is_finite_number(dt_geom)
                else np.nan
            )
        else:
            t_geom_dc_q = np.nan
            t_geom_dcac_q = np.nan
            dt_geom = np.nan
            dt_resid = np.nan

        dtq_rows.append({
            "protocol_pair": pair_id,
            "protocol_label_DCAC": seg_row["protocol_label_DCAC"],
            "file_name_DCAC": dcac_file,
            "Q_label": q_label,
            "Q_Ah": float(q_target),
            "is_anchor": bool(is_anchor),
            "segment_label": segment_label,
            "t_DC_s": t_dc_q,
            "t_DCAC_s": t_dcac_q,
            "dt_raw_s": dt_raw,
            "t_geom_DC_s": t_geom_dc_q,
            "t_geom_DCAC_s": t_geom_dcac_q,
            "dt_geom_s": dt_geom,
            "dt_resid_s": dt_resid,
            "geometry_phase_offset_rad": phase_result["geometry_phase_offset_rad"],
            "geometry_phase_offset_s": phase_result["geometry_phase_offset_s"],
            "geometry_phase_reference_status": phase_result["geometry_phase_reference_status"],
            "geometry_phase_fit_rmse_A": phase_result["geometry_phase_fit_rmse_A"],
            "geometry_phase_fit_window_s": phase_result["geometry_phase_fit_window_s"],
        })

    # Add grids
    for q in grid_A:
        add_dtq_row(q, "SegmentA_grid", SEGMENT_A, False)

    for q in grid_B:
        add_dtq_row(q, "SegmentB_grid", SEGMENT_B, False)

    for q in grid_D:
        add_dtq_row(q, "SegmentD_grid", SEGMENT_D, False)

    # Add anchors with pre-assigned segments
    anchor_segment_map = {
        "Q80_nominal": seg_row["Q80_nominal_segment"],
        "Q90_nominal": seg_row["Q90_nominal_segment"],
        "Q80_common": seg_row["Q80_common_segment"],
        "Q90_common": seg_row["Q90_common_segment"],
    }

    for label, q in anchors.items():
        add_dtq_row(q, label, anchor_segment_map[label], True)

    # Collect summary statistics from rows just added for this pair
    pair_rows = [r for r in dtq_rows if r["protocol_pair"] == pair_id]
    df_pair = pd.DataFrame(pair_rows)

    segA = df_pair[(df_pair["segment_label"] == SEGMENT_A) & (~df_pair["is_anchor"])]
    segB = df_pair[(df_pair["segment_label"] == SEGMENT_B) & (~df_pair["is_anchor"])]
    segD = df_pair[(df_pair["segment_label"] == SEGMENT_D) & (~df_pair["is_anchor"])]

    segA_resid = segA["dt_resid_s"].dropna().to_numpy(dtype=float)
    segA_raw = segA["dt_raw_s"].dropna().to_numpy(dtype=float)
    segB_raw = segB["dt_raw_s"].dropna().to_numpy(dtype=float)
    segD_raw = segD["dt_raw_s"].dropna().to_numpy(dtype=float)

    segA_grid_count = int(len(segA))

    if len(segA_resid) > 0:
        segA_resid_mean = float(np.mean(segA_resid))
        segA_resid_max_abs = float(np.max(np.abs(segA_resid)))
        if segA_grid_count >= Q_GRID_MIN_COUNT_SEGMENT_A:
            segA_resid_p95_abs = float(np.percentile(np.abs(segA_resid), 95))
        else:
            segA_resid_p95_abs = np.nan
    else:
        segA_resid_mean = np.nan
        segA_resid_max_abs = np.nan
        segA_resid_p95_abs = np.nan

    segA_above_status = segment_A_above_floor_status(
        max_abs_s=segA_resid_max_abs,
        p95_abs_s=segA_resid_p95_abs,
        q_grid_count=segA_grid_count,
    )

    anchor_rows = df_pair[df_pair["is_anchor"]].copy()
    anchor_dt = {
        row["Q_label"]: row["dt_raw_s"]
        for _, row in anchor_rows.iterrows()
    }

    # Segment-D anchors for late-CV preservation check
    segmentD_anchor_dt = anchor_rows.loc[
        anchor_rows["segment_label"] == SEGMENT_D,
        "dt_raw_s",
    ].dropna().to_list()

    late_cv_status = late_cv_preservation_status(
        segment_D_anchor_dt_raw_s=segmentD_anchor_dt,
        segment_D_dt_raw_median_s=float(np.median(segD_raw)) if len(segD_raw) else np.nan,
    )

    summary_rows.append({
        "protocol_pair": pair_id,
        "protocol_label_DCAC": seg_row["protocol_label_DCAC"],

        "geometry_phase_offset_rad": phase_result["geometry_phase_offset_rad"],
        "geometry_phase_offset_s": phase_result["geometry_phase_offset_s"],
        "geometry_phase_reference_status": phase_result["geometry_phase_reference_status"],
        "geometry_phase_fit_rmse_A": phase_result["geometry_phase_fit_rmse_A"],
        "geometry_phase_fit_window_s": phase_result["geometry_phase_fit_window_s"],

        "dt_Q80_nominal_raw_s": anchor_dt.get("Q80_nominal", np.nan),
        "dt_Q90_nominal_raw_s": anchor_dt.get("Q90_nominal", np.nan),
        "dt_Q80_common_raw_s": anchor_dt.get("Q80_common", np.nan),
        "dt_Q90_common_raw_s": anchor_dt.get("Q90_common", np.nan),

        "segment_A_Q_lo_Ah": q_seg_A_lo,
        "segment_A_Q_hi_Ah": q_seg_A_hi,
        "segment_A_Q_grid_count": segA_grid_count,
        "segment_A_dt_raw_median_s": float(np.median(segA_raw)) if len(segA_raw) else np.nan,
        "segment_A_dt_raw_max_s": float(np.max(segA_raw)) if len(segA_raw) else np.nan,
        "segment_A_dt_resid_mean_s": segA_resid_mean,
        "segment_A_dt_resid_max_abs_s": segA_resid_max_abs,
        "segment_A_dt_resid_p95_abs_s": segA_resid_p95_abs,
        "segment_A_resid_floor_s": MJ1_FLOOR_MAX_ABS_S,
        "segment_A_resid_floor_type": MJ1_FLOOR_TYPE,
        "segment_A_resid_floor_n": MJ1_FLOOR_N,
        "segment_A_above_floor_status": segA_above_status,

        "segment_B_Q_lo_Ah": q_seg_B_lo,
        "segment_B_Q_hi_Ah": q_seg_B_hi,
        "segment_B_dt_raw_median_s": float(np.median(segB_raw)) if len(segB_raw) else np.nan,
        "segment_B_dt_raw_max_s": float(np.max(segB_raw)) if len(segB_raw) else np.nan,

        "segment_D_Q_lo_Ah": q_seg_D_lo,
        "segment_D_Q_hi_Ah": q_seg_D_hi,
        "segment_D_dt_raw_min_s": float(np.min(segD_raw)) if len(segD_raw) else np.nan,
        "segment_D_dt_raw_median_s": float(np.median(segD_raw)) if len(segD_raw) else np.nan,
        "segment_D_dt_raw_max_s": float(np.max(segD_raw)) if len(segD_raw) else np.nan,
        "late_CV_preservation_threshold_s": LATE_CV_PRESERVATION_THRESHOLD_S,
        "late_CV_preservation_satisfied": late_cv_status,
    })


df_dtq_long = pd.DataFrame(dtq_rows)
df_dtq_summary = pd.DataFrame(summary_rows)

df_dtq_long.to_csv(OUT_DTQ_AUDIT_LONG, index=False)
df_dtq_summary.to_csv(OUT_DTQ_SUMMARY, index=False)

print(f"[OK] Wrote Δt(Q) segment audit long table: {OUT_DTQ_AUDIT_LONG}")
print(f"[OK] Wrote Δt(Q) segment summary: {OUT_DTQ_SUMMARY}")

display_cols = [
    "protocol_pair",
    "geometry_phase_reference_status",
    "geometry_phase_offset_rad",
    "geometry_phase_fit_rmse_A",
    "dt_Q80_nominal_raw_s",
    "dt_Q90_nominal_raw_s",
    "dt_Q80_common_raw_s",
    "dt_Q90_common_raw_s",
    "segment_A_Q_grid_count",
    "segment_A_dt_resid_mean_s",
    "segment_A_dt_resid_max_abs_s",
    "segment_A_dt_resid_p95_abs_s",
    "segment_A_above_floor_status",
    "segment_B_dt_raw_median_s",
    "segment_D_dt_raw_median_s",
    "late_CV_preservation_satisfied",
]
print(df_dtq_summary[display_cols].to_string(index=False))


# -----------------------------------------------------------------------------
# 7.4 Hard guards
# -----------------------------------------------------------------------------

if (df_dtq_summary["geometry_phase_reference_status"] == GEOM_PHASE_UNRESOLVED).any():
    print("[warning] Geometry phase unresolved for at least one pair.")
    print(df_dtq_summary[display_cols].to_string(index=False))
    raise ValueError("Segment-A residual is not interpretable for at least one pair.")

if df_dtq_summary["segment_A_Q_grid_count"].min() < Q_GRID_MIN_COUNT_SEGMENT_A:
    print("[warning] Segment-A Q-grid count below p95 requirement for at least one pair.")

print("[OK] Cell 7 first-passage Δt and Segment-A residual audit completed.")
print("[OK] No final mechanism verdict performed in Cell 7.")

[OK] Loaded segment assignment: /Users/louislu/pybamm-dcac-superimposed/data/day21A_step4_MJ1_segment_assignment.csv
[OK] Wrote Δt(Q) segment audit long table: /Users/louislu/pybamm-dcac-superimposed/data/day21A_step5_MJ1_dtQ_segment_audit_long.csv
[OK] Wrote Δt(Q) segment summary: /Users/louislu/pybamm-dcac-superimposed/data/day21A_step5_MJ1_dtQ_segment_summary.csv
              protocol_pair geometry_phase_reference_status  geometry_phase_offset_rad  geometry_phase_fit_rmse_A  dt_Q80_nominal_raw_s  dt_Q90_nominal_raw_s  dt_Q80_common_raw_s  dt_Q90_common_raw_s  segment_A_Q_grid_count  segment_A_dt_resid_mean_s  segment_A_dt_resid_max_abs_s  segment_A_dt_resid_p95_abs_s segment_A_above_floor_status  segment_B_dt_raw_median_s  segment_D_dt_raw_median_s late_CV_preservation_satisfied
0.3C DC vs 0.3C+0.7C 0.1tau estimated_from_current_waveform                   0.116239                   0.092546            114.070280            421.990458             0.196020           351.715593       

In [17]:
# Cell 7A — Diagnostic audit of Segment-A residual shape
#
# Purpose:
# - Locate where above-floor Segment-A residuals occur
# - Distinguish distributed residual from boundary spike / first-passage artifact
# - Inspect raw Δt, geometry Δt, residual vs Q
#
# Explicitly NOT done here:
# - No verdict
# - No schema changes
# - No redefinition of thresholds

if not OUT_DTQ_AUDIT_LONG.exists():
    raise FileNotFoundError(f"Missing long dtQ audit file: {OUT_DTQ_AUDIT_LONG}")

if not OUT_DTQ_SUMMARY.exists():
    raise FileNotFoundError(f"Missing dtQ summary file: {OUT_DTQ_SUMMARY}")

df_dtq_long = pd.read_csv(OUT_DTQ_AUDIT_LONG)
df_dtq_summary = pd.read_csv(OUT_DTQ_SUMMARY)

SEG_A_GRID = df_dtq_long[
    (df_dtq_long["segment_label"] == SEGMENT_A)
    & (df_dtq_long["is_anchor"] == False)
].copy()

diagnostic_rows = []

for pair, g in SEG_A_GRID.groupby("protocol_pair"):
    g = g.sort_values("Q_Ah").reset_index(drop=True)
    g["abs_resid_s"] = g["dt_resid_s"].abs()

    finite = g[np.isfinite(g["dt_resid_s"])].copy()

    if len(finite) == 0:
        diagnostic_rows.append({
            "protocol_pair": pair,
            "n": 0,
            "resid_mean_s": np.nan,
            "resid_median_s": np.nan,
            "resid_p95_abs_s": np.nan,
            "resid_max_abs_s": np.nan,
            "Q_at_max_abs_Ah": np.nan,
            "dt_raw_at_max_s": np.nan,
            "dt_geom_at_max_s": np.nan,
            "dt_resid_at_max_s": np.nan,
            "fraction_abs_gt_2p70": np.nan,
            "fraction_abs_gt_6p75": np.nan,
            "near_upper_10pct_max_abs_s": np.nan,
            "early_10pct_max_abs_s": np.nan,
        })
        continue

    idx_max = finite["abs_resid_s"].idxmax()
    row_max = finite.loc[idx_max]

    q_min = finite["Q_Ah"].min()
    q_max = finite["Q_Ah"].max()
    q_span = q_max - q_min

    early = finite[finite["Q_Ah"] <= q_min + 0.10 * q_span]
    late = finite[finite["Q_Ah"] >= q_max - 0.10 * q_span]

    diagnostic_rows.append({
        "protocol_pair": pair,
        "n": len(finite),
        "Q_lo_Ah": q_min,
        "Q_hi_Ah": q_max,
        "resid_mean_s": float(finite["dt_resid_s"].mean()),
        "resid_median_s": float(finite["dt_resid_s"].median()),
        "resid_p95_abs_s": float(np.percentile(finite["abs_resid_s"], 95)),
        "resid_max_abs_s": float(finite["abs_resid_s"].max()),
        "Q_at_max_abs_Ah": float(row_max["Q_Ah"]),
        "dt_raw_at_max_s": float(row_max["dt_raw_s"]),
        "dt_geom_at_max_s": float(row_max["dt_geom_s"]),
        "dt_resid_at_max_s": float(row_max["dt_resid_s"]),
        "fraction_abs_gt_2p70": float((finite["abs_resid_s"] > 2.70).mean()),
        "fraction_abs_gt_6p75": float((finite["abs_resid_s"] > 6.75).mean()),
        "early_10pct_max_abs_s": float(early["abs_resid_s"].max()) if len(early) else np.nan,
        "near_upper_10pct_max_abs_s": float(late["abs_resid_s"].max()) if len(late) else np.nan,
    })

df_resid_diag = pd.DataFrame(diagnostic_rows)

OUT_RESID_DIAG = DATA_DIR / "day21A_step5A_MJ1_segmentA_residual_diagnostics.csv"
df_resid_diag.to_csv(OUT_RESID_DIAG, index=False)

print(f"[OK] Wrote Segment-A residual diagnostics: {OUT_RESID_DIAG}")
print(df_resid_diag.to_string(index=False))


# Show top residual points per protocol
for pair, g in SEG_A_GRID.groupby("protocol_pair"):
    g = g.copy()
    g["abs_resid_s"] = g["dt_resid_s"].abs()
    top = (
        g.sort_values("abs_resid_s", ascending=False)
        .head(10)
        [[
            "protocol_pair",
            "Q_Ah",
            "dt_raw_s",
            "dt_geom_s",
            "dt_resid_s",
            "abs_resid_s",
            "t_DC_s",
            "t_DCAC_s",
            "t_geom_DC_s",
            "t_geom_DCAC_s",
        ]]
    )
    print("\n" + "=" * 120)
    print(f"[Top residual points] {pair}")
    print("=" * 120)
    print(top.to_string(index=False))

[OK] Wrote Segment-A residual diagnostics: /Users/louislu/pybamm-dcac-superimposed/data/day21A_step5A_MJ1_segmentA_residual_diagnostics.csv
              protocol_pair   n  Q_lo_Ah  Q_hi_Ah  resid_mean_s  resid_median_s  resid_p95_abs_s  resid_max_abs_s  Q_at_max_abs_Ah  dt_raw_at_max_s  dt_geom_at_max_s  dt_resid_at_max_s  fraction_abs_gt_2p70  fraction_abs_gt_6p75  early_10pct_max_abs_s  near_upper_10pct_max_abs_s
0.3C DC vs 0.3C+0.7C 0.1tau 263     0.05     2.67     -0.757979       -1.135797         3.696359        28.685838             2.67        14.681684        -14.004154          28.685838              0.186312              0.003802               3.814212                   28.685838
 0.3C DC vs 0.3C+0.7C 10tau 245     0.05     2.49      5.631817       -2.848807        10.835425       426.430441             2.23       521.308930         94.878489         426.430441              0.697959              0.281633               4.297857                  419.458443
  0.3C DC vs 0.3C+0.

In [18]:
# Cell 7B — Geometry near-miss / first-passage branch-jump audit
#
# Purpose:
# - Diagnose whether large Segment-A residuals arise from first-passage branch jumps
# - Detect geometry near-miss cases where Q_geom almost reaches target Q
#   before the reported first-passage crossing
# - Quantify target deficit to the previous local maximum of Q_geom_DCAC
#
# Explicitly NOT done here:
# - No mechanism verdict
# - No threshold redefinition
# - No Segment-A residual recomputation

OUT_BRANCH_JUMP_AUDIT = DATA_DIR / "day21A_step5B_MJ1_geometry_branch_jump_audit.csv"

if not OUT_DTQ_AUDIT_LONG.exists():
    raise FileNotFoundError(f"Missing long dtQ audit file: {OUT_DTQ_AUDIT_LONG}")

df_dtq_long = pd.read_csv(OUT_DTQ_AUDIT_LONG)

SEG_A_GRID = df_dtq_long[
    (df_dtq_long["segment_label"] == SEGMENT_A)
    & (df_dtq_long["is_anchor"] == False)
].copy()


def reconstruct_geometry_for_pair(protocol_pair: str):
    """
    Reconstruct measured and geometry Q trajectories for one protocol pair.
    Uses the same definitions as Cell 7.
    """
    seg_row = df_segment_assignment[df_segment_assignment["protocol_pair"] == protocol_pair]
    if len(seg_row) != 1:
        raise ValueError(f"Expected one segment row for {protocol_pair}, found {len(seg_row)}")
    seg_row = seg_row.iloc[0]

    dcac_file = seg_row["file_name_DCAC"]
    dc_file = seg_row["file_name_DC"]

    dc_traj = TRAJ[dc_file]
    dcac_traj = TRAJ[dcac_file]
    inv_dcac = get_single_inventory_row(dcac_file)

    I_DC_A = float(inv_dcac["DC_C"]) * ONE_C_A
    I_AC_A = float(inv_dcac["AC_C"]) * ONE_C_A
    frequency_hz = float(inv_dcac["frequency_Hz"])

    # Pull phase used in Cell 7 from dtQ long table.
    pair_rows = df_dtq_long[df_dtq_long["protocol_pair"] == protocol_pair]
    phase_vals = pair_rows["geometry_phase_offset_rad"].dropna().unique()
    if len(phase_vals) == 0:
        phase_rad = 0.0
    else:
        phase_rad = float(phase_vals[0])

    t_dc = dc_traj["t_s"].to_numpy(dtype=float)
    q_dc = dc_traj["Q_net_Ah"].to_numpy(dtype=float)

    t_dcac = dcac_traj["t_s"].to_numpy(dtype=float)
    q_dcac = dcac_traj["Q_net_Ah"].to_numpy(dtype=float)

    q_geom_dc = prescribed_geometry_Q_Ah(
        t_s=t_dc,
        I_DC_A=I_DC_A,
        I_AC_A=0.0,
        frequency_Hz=0.0,
        phase_rad=0.0,
    )

    q_geom_dcac = prescribed_geometry_Q_Ah(
        t_s=t_dcac,
        I_DC_A=I_DC_A,
        I_AC_A=I_AC_A,
        frequency_Hz=frequency_hz,
        phase_rad=phase_rad,
    )

    return {
        "dc_file": dc_file,
        "dcac_file": dcac_file,
        "t_dc": t_dc,
        "q_dc": q_dc,
        "t_dcac": t_dcac,
        "q_dcac": q_dcac,
        "q_geom_dc": q_geom_dc,
        "q_geom_dcac": q_geom_dcac,
        "frequency_hz": frequency_hz,
        "T_AC_s": compute_t_ac_s(frequency_hz),
        "phase_rad": phase_rad,
    }


def previous_geometry_near_miss(
    q_target_Ah: float,
    t_geom_dcac: float,
    t_dcac: np.ndarray,
    q_geom_dcac: np.ndarray,
):
    """
    For a target Q and geometry first-passage time, inspect geometry trajectory
    before the crossing.

    We ask:
    - What was the maximum Q_geom before t_geom_DCAC?
    - How far below target was that previous maximum?
    - How long before the crossing did that maximum occur?
    """
    if not is_finite_number(q_target_Ah) or not is_finite_number(t_geom_dcac):
        return {
            "geom_prev_max_Q_Ah": np.nan,
            "geom_prev_max_time_s": np.nan,
            "geom_prev_max_deficit_mAh": np.nan,
            "geom_prev_max_lead_time_s": np.nan,
        }

    t = np.asarray(t_dcac, dtype=float)
    q = np.asarray(q_geom_dcac, dtype=float)

    finite = np.isfinite(t) & np.isfinite(q)
    before = finite & (t < float(t_geom_dcac))

    if before.sum() == 0:
        return {
            "geom_prev_max_Q_Ah": np.nan,
            "geom_prev_max_time_s": np.nan,
            "geom_prev_max_deficit_mAh": np.nan,
            "geom_prev_max_lead_time_s": np.nan,
        }

    idxs = np.where(before)[0]
    q_before = q[idxs]
    idx_local = int(np.nanargmax(q_before))
    idx = int(idxs[idx_local])

    q_prev_max = float(q[idx])
    t_prev_max = float(t[idx])
    deficit_mAh = (float(q_target_Ah) - q_prev_max) * 1000.0
    lead_time_s = float(t_geom_dcac) - t_prev_max

    return {
        "geom_prev_max_Q_Ah": q_prev_max,
        "geom_prev_max_time_s": t_prev_max,
        "geom_prev_max_deficit_mAh": deficit_mAh,
        "geom_prev_max_lead_time_s": lead_time_s,
    }


branch_rows = []

# We audit all Segment-A grid rows, but mark strongest residuals clearly.
for pair, g in SEG_A_GRID.groupby("protocol_pair"):
    recon = reconstruct_geometry_for_pair(pair)

    g = g.copy()
    g["abs_resid_s"] = g["dt_resid_s"].abs()
    g = g.sort_values("abs_resid_s", ascending=False)

    for _, row in g.iterrows():
        nm = previous_geometry_near_miss(
            q_target_Ah=float(row["Q_Ah"]),
            t_geom_dcac=float(row["t_geom_DCAC_s"]),
            t_dcac=recon["t_dcac"],
            q_geom_dcac=recon["q_geom_dcac"],
        )

        branch_rows.append({
            "protocol_pair": pair,
            "Q_Ah": float(row["Q_Ah"]),
            "dt_raw_s": float(row["dt_raw_s"]),
            "dt_geom_s": float(row["dt_geom_s"]),
            "dt_resid_s": float(row["dt_resid_s"]),
            "abs_resid_s": float(row["abs_resid_s"]),
            "t_DCAC_s": float(row["t_DCAC_s"]),
            "t_geom_DCAC_s": float(row["t_geom_DCAC_s"]),
            "T_AC_s": float(recon["T_AC_s"]),
            "geometry_phase_offset_rad": float(recon["phase_rad"]),
            **nm,
        })

df_branch = pd.DataFrame(branch_rows)

# Classification thresholds for diagnostic labeling only.
# These do not redefine the pre-registered residual floor.
NEAR_MISS_DEFICIT_MAH = 1.0
BRANCH_JUMP_TIME_FRACTION_TAC = 0.25

df_branch["near_miss_deficit_le_1mAh"] = (
    df_branch["geom_prev_max_deficit_mAh"].abs() <= NEAR_MISS_DEFICIT_MAH
)

df_branch["branch_jump_time_ge_0p25TAC"] = (
    df_branch["geom_prev_max_lead_time_s"] >= BRANCH_JUMP_TIME_FRACTION_TAC * df_branch["T_AC_s"]
)

df_branch["branch_jump_artifact_candidate"] = (
    df_branch["near_miss_deficit_le_1mAh"]
    & df_branch["branch_jump_time_ge_0p25TAC"]
)

df_branch.to_csv(OUT_BRANCH_JUMP_AUDIT, index=False)

print(f"[OK] Wrote geometry branch-jump audit: {OUT_BRANCH_JUMP_AUDIT}")

summary = (
    df_branch.groupby("protocol_pair")
    .agg(
        n=("Q_Ah", "size"),
        max_abs_resid_s=("abs_resid_s", "max"),
        n_branch_jump_candidates=("branch_jump_artifact_candidate", "sum"),
        frac_branch_jump_candidates=("branch_jump_artifact_candidate", "mean"),
        min_prev_max_deficit_mAh=("geom_prev_max_deficit_mAh", "min"),
        p05_prev_max_deficit_mAh=("geom_prev_max_deficit_mAh", lambda x: np.nanpercentile(x, 5)),
        median_prev_max_deficit_mAh=("geom_prev_max_deficit_mAh", "median"),
    )
    .reset_index()
)

print("\n[Branch-jump diagnostic summary]")
print(summary.to_string(index=False))

for pair, g in df_branch.groupby("protocol_pair"):
    top = g.sort_values("abs_resid_s", ascending=False).head(12)
    print("\n" + "=" * 120)
    print(f"[Top branch-jump diagnostics] {pair}")
    print("=" * 120)
    print(top[[
        "protocol_pair",
        "Q_Ah",
        "dt_raw_s",
        "dt_geom_s",
        "dt_resid_s",
        "abs_resid_s",
        "geom_prev_max_Q_Ah",
        "geom_prev_max_deficit_mAh",
        "geom_prev_max_lead_time_s",
        "T_AC_s",
        "branch_jump_artifact_candidate",
    ]].to_string(index=False))

print("\n[OK] Cell 7B branch-jump diagnostic audit completed.")
print("[OK] No verdict performed in Cell 7B.")

[OK] Wrote geometry branch-jump audit: /Users/louislu/pybamm-dcac-superimposed/data/day21A_step5B_MJ1_geometry_branch_jump_audit.csv

[Branch-jump diagnostic summary]
              protocol_pair   n  max_abs_resid_s  n_branch_jump_candidates  frac_branch_jump_candidates  min_prev_max_deficit_mAh  p05_prev_max_deficit_mAh  median_prev_max_deficit_mAh
0.3C DC vs 0.3C+0.7C 0.1tau 263        28.685838                        57                     0.216730                  0.002259                  0.039055                     0.336855
 0.3C DC vs 0.3C+0.7C 10tau 245       426.430441                         2                     0.008163                  0.004120                  0.035241                     0.377975
  0.3C DC vs 0.3C+0.7C 1tau 250        42.929877                         3                     0.012000                  0.000725                  0.048185                     0.398780

[Top branch-jump diagnostics] 0.3C DC vs 0.3C+0.7C 0.1tau
              protocol_pair  Q_Ah 

In [19]:
# Cell 7C — Measured-vs-prescribed current / geometry fidelity audit
#
# Purpose:
# - Diagnose whether Segment-A residual is driven by mismatch between
#   measured NGU201 current and prescribed ideal sine geometry
# - Decompose residual into DC and DCAC first-passage geometry errors:
#
#   dt_resid = (t_DC_meas - t_DC_geom) - (t_DCAC_meas - t_DCAC_geom)
#
# Explicitly NOT done here:
# - No threshold redefinition
# - No mechanism verdict
# - No schema changes

OUT_GEOM_FIDELITY_SUMMARY = DATA_DIR / "day21A_step5C_MJ1_geometry_fidelity_summary.csv"
OUT_GEOM_FIDELITY_LONG = DATA_DIR / "day21A_step5C_MJ1_geometry_fidelity_long.csv"

if not OUT_DTQ_AUDIT_LONG.exists():
    raise FileNotFoundError(f"Missing long dtQ audit file: {OUT_DTQ_AUDIT_LONG}")

if not OUT_DTQ_SUMMARY.exists():
    raise FileNotFoundError(f"Missing dtQ summary file: {OUT_DTQ_SUMMARY}")

df_dtq_long = pd.read_csv(OUT_DTQ_AUDIT_LONG)
df_dtq_summary = pd.read_csv(OUT_DTQ_SUMMARY)


def prescribed_current_A(
    t_s: np.ndarray,
    I_DC_A: float,
    I_AC_A: float,
    frequency_Hz: float,
    phase_rad: float = 0.0,
) -> np.ndarray:
    """Charge-positive prescribed current."""
    t = np.asarray(t_s, dtype=float)

    if float(I_AC_A) == 0.0 or float(frequency_Hz) == 0.0:
        return np.full(len(t), float(I_DC_A), dtype=float)

    omega = 2.0 * np.pi * float(frequency_Hz)
    return float(I_DC_A) + float(I_AC_A) * np.sin(omega * t + float(phase_rad))


def get_phase_for_pair(protocol_pair: str) -> float:
    vals = (
        df_dtq_long.loc[
            df_dtq_long["protocol_pair"] == protocol_pair,
            "geometry_phase_offset_rad",
        ]
        .dropna()
        .unique()
    )
    if len(vals) == 0:
        return 0.0
    return float(vals[0])


def get_segment_assignment_row(protocol_pair: str) -> pd.Series:
    rows = df_segment_assignment[df_segment_assignment["protocol_pair"] == protocol_pair]
    if len(rows) != 1:
        raise ValueError(f"Expected exactly one segment assignment row for {protocol_pair}, found {len(rows)}")
    return rows.iloc[0]


fidelity_summary_rows = []
fidelity_long_rows = []

for protocol_pair, segA in df_dtq_long[
    (df_dtq_long["segment_label"] == SEGMENT_A)
    & (df_dtq_long["is_anchor"] == False)
].groupby("protocol_pair"):

    seg_row = get_segment_assignment_row(protocol_pair)

    dc_file = seg_row["file_name_DC"]
    dcac_file = seg_row["file_name_DCAC"]

    dc_traj = TRAJ[dc_file]
    dcac_traj = TRAJ[dcac_file]
    inv_dcac = get_single_inventory_row(dcac_file)

    I_DC_A = float(inv_dcac["DC_C"]) * ONE_C_A
    I_AC_A = float(inv_dcac["AC_C"]) * ONE_C_A
    frequency_hz = float(inv_dcac["frequency_Hz"])
    phase_rad = get_phase_for_pair(protocol_pair)

    q_seg_hi = float(seg_row["segment_A_Q_hi_Ah"])
    t_segB_start = float(seg_row["t_segmentB_start_s"])

    # -------------------------------------------------------------------------
    # DC current fidelity over the DC time window needed to reach Segment-A upper Q
    # -------------------------------------------------------------------------
    t_dc = dc_traj["t_s"].to_numpy(dtype=float)
    i_dc_meas = dc_traj["I_Q_A"].to_numpy(dtype=float)
    q_dc_meas = dc_traj["Q_net_Ah"].to_numpy(dtype=float)

    t_dc_hi = first_passage_time_from_Q(q_seg_hi, t_dc, q_dc_meas)
    dc_mask = np.isfinite(t_dc) & np.isfinite(i_dc_meas) & (t_dc >= 0)
    if is_finite_number(t_dc_hi):
        dc_mask = dc_mask & (t_dc <= float(t_dc_hi))

    i_dc_geom = prescribed_current_A(
        t_s=t_dc,
        I_DC_A=I_DC_A,
        I_AC_A=0.0,
        frequency_Hz=0.0,
        phase_rad=0.0,
    )

    q_dc_geom = prescribed_geometry_Q_Ah(
        t_s=t_dc,
        I_DC_A=I_DC_A,
        I_AC_A=0.0,
        frequency_Hz=0.0,
        phase_rad=0.0,
    )

    dc_i_err = i_dc_meas[dc_mask] - i_dc_geom[dc_mask]
    dc_q_err = q_dc_meas[dc_mask] - q_dc_geom[dc_mask]

    # -------------------------------------------------------------------------
    # DCAC current fidelity over Segment-A time window
    # -------------------------------------------------------------------------
    t_dcac = dcac_traj["t_s"].to_numpy(dtype=float)
    i_dcac_meas = dcac_traj["I_Q_A"].to_numpy(dtype=float)
    q_dcac_meas = dcac_traj["Q_net_Ah"].to_numpy(dtype=float)

    dcac_mask = (
        np.isfinite(t_dcac)
        & np.isfinite(i_dcac_meas)
        & (t_dcac >= 0)
        & (t_dcac <= t_segB_start)
    )

    i_dcac_geom = prescribed_current_A(
        t_s=t_dcac,
        I_DC_A=I_DC_A,
        I_AC_A=I_AC_A,
        frequency_Hz=frequency_hz,
        phase_rad=phase_rad,
    )

    q_dcac_geom = prescribed_geometry_Q_Ah(
        t_s=t_dcac,
        I_DC_A=I_DC_A,
        I_AC_A=I_AC_A,
        frequency_Hz=frequency_hz,
        phase_rad=phase_rad,
    )

    dcac_i_err = i_dcac_meas[dcac_mask] - i_dcac_geom[dcac_mask]
    dcac_q_err = q_dcac_meas[dcac_mask] - q_dcac_geom[dcac_mask]

    # -------------------------------------------------------------------------
    # First-passage geometry-error decomposition over Segment-A grid
    # -------------------------------------------------------------------------
    segA = segA.sort_values("Q_Ah").copy()

    for _, row in segA.iterrows():
        q = float(row["Q_Ah"])

        t_dc_meas = first_passage_time_from_Q(q, t_dc, q_dc_meas)
        t_dc_geom = first_passage_time_from_Q(q, t_dc, q_dc_geom)

        t_dcac_meas = first_passage_time_from_Q(q, t_dcac, q_dcac_meas)
        t_dcac_geom = first_passage_time_from_Q(q, t_dcac, q_dcac_geom)

        fp_err_dc = (
            t_dc_meas - t_dc_geom
            if is_finite_number(t_dc_meas) and is_finite_number(t_dc_geom)
            else np.nan
        )

        fp_err_dcac = (
            t_dcac_meas - t_dcac_geom
            if is_finite_number(t_dcac_meas) and is_finite_number(t_dcac_geom)
            else np.nan
        )

        residual_reconstructed = (
            fp_err_dc - fp_err_dcac
            if is_finite_number(fp_err_dc) and is_finite_number(fp_err_dcac)
            else np.nan
        )

        fidelity_long_rows.append({
            "protocol_pair": protocol_pair,
            "Q_Ah": q,
            "dt_resid_s": float(row["dt_resid_s"]),
            "fp_err_DC_s": fp_err_dc,
            "fp_err_DCAC_s": fp_err_dcac,
            "residual_reconstructed_s": residual_reconstructed,
            "reconstruction_error_s": (
                float(row["dt_resid_s"]) - residual_reconstructed
                if is_finite_number(row["dt_resid_s"]) and is_finite_number(residual_reconstructed)
                else np.nan
            ),
        })

    # -------------------------------------------------------------------------
    # Summary
    # -------------------------------------------------------------------------
    df_pair_long = pd.DataFrame([
        r for r in fidelity_long_rows if r["protocol_pair"] == protocol_pair
    ])

    def safe_mean(x):
        x = np.asarray(x, dtype=float)
        x = x[np.isfinite(x)]
        return float(np.mean(x)) if len(x) else np.nan

    def safe_rmse(x):
        x = np.asarray(x, dtype=float)
        x = x[np.isfinite(x)]
        return float(np.sqrt(np.mean(x ** 2))) if len(x) else np.nan

    def safe_absmax(x):
        x = np.asarray(x, dtype=float)
        x = x[np.isfinite(x)]
        return float(np.max(np.abs(x))) if len(x) else np.nan

    def safe_p95_abs(x):
        x = np.asarray(x, dtype=float)
        x = x[np.isfinite(x)]
        return float(np.percentile(np.abs(x), 95)) if len(x) else np.nan

    fidelity_summary_rows.append({
        "protocol_pair": protocol_pair,
        "I_DC_A": I_DC_A,
        "I_AC_A": I_AC_A,
        "frequency_Hz": frequency_hz,
        "T_AC_s": compute_t_ac_s(frequency_hz),
        "geometry_phase_offset_rad": phase_rad,

        "DC_I_error_mean_A": safe_mean(dc_i_err),
        "DC_I_error_rmse_A": safe_rmse(dc_i_err),
        "DC_I_error_max_abs_A": safe_absmax(dc_i_err),
        "DC_Q_error_final_mAh": float(dc_q_err[-1] * 1000.0) if len(dc_q_err) else np.nan,
        "DC_Q_error_max_abs_mAh": safe_absmax(dc_q_err * 1000.0),

        "DCAC_I_error_mean_A": safe_mean(dcac_i_err),
        "DCAC_I_error_rmse_A": safe_rmse(dcac_i_err),
        "DCAC_I_error_max_abs_A": safe_absmax(dcac_i_err),
        "DCAC_Q_error_at_segmentB_mAh": float(dcac_q_err[-1] * 1000.0) if len(dcac_q_err) else np.nan,
        "DCAC_Q_error_max_abs_mAh": safe_absmax(dcac_q_err * 1000.0),

        "fp_err_DC_mean_s": safe_mean(df_pair_long["fp_err_DC_s"]),
        "fp_err_DC_p95_abs_s": safe_p95_abs(df_pair_long["fp_err_DC_s"]),
        "fp_err_DCAC_mean_s": safe_mean(df_pair_long["fp_err_DCAC_s"]),
        "fp_err_DCAC_p95_abs_s": safe_p95_abs(df_pair_long["fp_err_DCAC_s"]),
        "dt_resid_mean_s": safe_mean(df_pair_long["dt_resid_s"]),
        "dt_resid_p95_abs_s": safe_p95_abs(df_pair_long["dt_resid_s"]),
        "reconstruction_error_max_abs_s": safe_absmax(df_pair_long["reconstruction_error_s"]),
    })

df_fidelity_long = pd.DataFrame(fidelity_long_rows)
df_fidelity_summary = pd.DataFrame(fidelity_summary_rows)

df_fidelity_long.to_csv(OUT_GEOM_FIDELITY_LONG, index=False)
df_fidelity_summary.to_csv(OUT_GEOM_FIDELITY_SUMMARY, index=False)

print(f"[OK] Wrote geometry fidelity long table: {OUT_GEOM_FIDELITY_LONG}")
print(f"[OK] Wrote geometry fidelity summary: {OUT_GEOM_FIDELITY_SUMMARY}")

display_cols = [
    "protocol_pair",
    "DC_I_error_mean_A",
    "DC_I_error_rmse_A",
    "DC_Q_error_final_mAh",
    "DCAC_I_error_mean_A",
    "DCAC_I_error_rmse_A",
    "DCAC_Q_error_at_segmentB_mAh",
    "DCAC_Q_error_max_abs_mAh",
    "fp_err_DC_mean_s",
    "fp_err_DCAC_mean_s",
    "fp_err_DCAC_p95_abs_s",
    "dt_resid_mean_s",
    "dt_resid_p95_abs_s",
    "reconstruction_error_max_abs_s",
]
print(df_fidelity_summary[display_cols].to_string(index=False))

print("\n[Top |fp_err_DCAC| points]")
for pair, g in df_fidelity_long.groupby("protocol_pair"):
    top = g.copy()
    top["abs_fp_err_DCAC_s"] = top["fp_err_DCAC_s"].abs()
    top = top.sort_values("abs_fp_err_DCAC_s", ascending=False).head(10)
    print("\n" + "=" * 120)
    print(pair)
    print("=" * 120)
    print(top[[
        "protocol_pair",
        "Q_Ah",
        "dt_resid_s",
        "fp_err_DC_s",
        "fp_err_DCAC_s",
        "residual_reconstructed_s",
        "reconstruction_error_s",
    ]].to_string(index=False))

print("\n[OK] Cell 7C measured-vs-prescribed geometry fidelity audit completed.")
print("[OK] No verdict performed in Cell 7C.")

[OK] Wrote geometry fidelity long table: /Users/louislu/pybamm-dcac-superimposed/data/day21A_step5C_MJ1_geometry_fidelity_long.csv
[OK] Wrote geometry fidelity summary: /Users/louislu/pybamm-dcac-superimposed/data/day21A_step5C_MJ1_geometry_fidelity_summary.csv
              protocol_pair  DC_I_error_mean_A  DC_I_error_rmse_A  DC_Q_error_final_mAh  DCAC_I_error_mean_A  DCAC_I_error_rmse_A  DCAC_Q_error_at_segmentB_mAh  DCAC_Q_error_max_abs_mAh  fp_err_DC_mean_s  fp_err_DCAC_mean_s  fp_err_DCAC_p95_abs_s  dt_resid_mean_s  dt_resid_p95_abs_s  reconstruction_error_max_abs_s
0.3C DC vs 0.3C+0.7C 0.1tau          -0.000108           0.000108             -0.282308             0.001053             2.543167                      8.448144                  8.448144          0.508810            1.266789               4.312909        -0.757979            3.696359                    3.637979e-12
 0.3C DC vs 0.3C+0.7C 10tau          -0.000108           0.000108             -0.262470            -0.0051

In [20]:
# Cell 7D — Fitted waveform geometry diagnostic
#
# Purpose:
# - Fit actual measured DCAC current waveform in Segment A:
#   I_Q(t) = I0 + A * sin(2π f_fit t + phi)
# - Compare prescribed-geometry residual vs fitted-waveform-geometry residual
# - Diagnose whether above-floor residual is caused by waveform realization mismatch
#
# Explicitly NOT done here:
# - No threshold redefinition
# - No mechanism verdict
# - No schema overwrite

from scipy.optimize import least_squares

OUT_FIT_GEOM_LONG = DATA_DIR / "day21A_step5D_MJ1_fitted_geometry_residual_long.csv"
OUT_FIT_GEOM_SUMMARY = DATA_DIR / "day21A_step5D_MJ1_fitted_geometry_residual_summary.csv"


def fit_sine_current_segment_A(
    df_traj: pd.DataFrame,
    t_hi_s: float,
    I0_init_A: float,
    A_init_A: float,
    f_init_Hz: float,
    phi_init_rad: float = 0.0,
) -> dict[str, float | str]:
    """
    Fit measured current in Segment A with:
        I(t) = I0 + A * sin(2π f t + phi)

    This is diagnostic only. It does not redefine the pre-registered prescribed geometry.
    """
    t = df_traj["t_s"].to_numpy(dtype=float)
    i = df_traj["I_Q_A"].to_numpy(dtype=float)

    mask = (
        np.isfinite(t)
        & np.isfinite(i)
        & (t >= 0)
        & (t <= float(t_hi_s))
    )

    if mask.sum() < 20:
        return {
            "fit_status": "unresolved_insufficient_samples",
            "I0_fit_A": np.nan,
            "A_fit_A": np.nan,
            "f_fit_Hz": np.nan,
            "phi_fit_rad": np.nan,
            "fit_rmse_A": np.nan,
            "fit_n": int(mask.sum()),
        }

    tt = t[mask]
    ii = i[mask]

    # Use relative time for better conditioning; convert phase back implicitly by model using tt_rel.
    tt_rel = tt - tt[0]

    def model(params):
        I0, A, f, phi = params
        return I0 + A * np.sin(2.0 * np.pi * f * tt_rel + phi)

    def residual(params):
        return model(params) - ii

    if not np.isfinite(f_init_Hz) or f_init_Hz <= 0:
        return {
            "fit_status": "unresolved_invalid_initial_frequency",
            "I0_fit_A": np.nan,
            "A_fit_A": np.nan,
            "f_fit_Hz": np.nan,
            "phi_fit_rad": np.nan,
            "fit_rmse_A": np.nan,
            "fit_n": int(mask.sum()),
        }

    x0 = np.array([
        float(I0_init_A),
        max(float(A_init_A), 1e-6),
        float(f_init_Hz),
        float(phi_init_rad),
    ])

    # Conservative bounds. They prevent pathological fits but allow realistic NGU realization drift.
    lower = np.array([
        -ONE_C_A,              # I0 lower
        0.0,                   # amplitude lower
        0.5 * float(f_init_Hz),
        -np.pi,
    ])
    upper = np.array([
        2.0 * ONE_C_A,          # I0 upper
        2.0 * ONE_C_A,          # amplitude upper
        1.5 * float(f_init_Hz),
        np.pi,
    ])

    try:
        res = least_squares(
            residual,
            x0=x0,
            bounds=(lower, upper),
            max_nfev=5000,
            xtol=1e-12,
            ftol=1e-12,
            gtol=1e-12,
        )
    except Exception as exc:
        return {
            "fit_status": f"unresolved_fit_error:{type(exc).__name__}",
            "I0_fit_A": np.nan,
            "A_fit_A": np.nan,
            "f_fit_Hz": np.nan,
            "phi_fit_rad": np.nan,
            "fit_rmse_A": np.nan,
            "fit_n": int(mask.sum()),
        }

    I0_fit, A_fit, f_fit, phi_fit = res.x
    pred = model(res.x)
    rmse = float(np.sqrt(np.mean((pred - ii) ** 2)))

    return {
        "fit_status": "ok" if res.success else "fit_not_successful",
        "I0_fit_A": float(I0_fit),
        "A_fit_A": float(A_fit),
        "f_fit_Hz": float(f_fit),
        "phi_fit_rad": float(phi_fit),
        "fit_rmse_A": rmse,
        "fit_n": int(mask.sum()),
    }


def fitted_geometry_Q_Ah(
    t_s: np.ndarray,
    I0_A: float,
    A_A: float,
    f_Hz: float,
    phi_rad: float,
) -> np.ndarray:
    """
    Diagnostic fitted-waveform geometry Q.
    Uses relative time origin t[0] for the fitted model.
    """
    t = np.asarray(t_s, dtype=float)
    if len(t) == 0:
        return np.array([], dtype=float)

    t_rel = t - t[0]
    i_fit = I0_A + A_A * np.sin(2.0 * np.pi * f_Hz * t_rel + phi_rad)
    return integrate_strict_net_Q_Ah(t, i_fit)


fit_long_rows = []
fit_summary_rows = []

for _, seg_row in df_segment_assignment.iterrows():
    pair = seg_row["protocol_pair"]
    dcac_file = seg_row["file_name_DCAC"]
    dc_file = seg_row["file_name_DC"]

    dc_traj = TRAJ[dc_file]
    dcac_traj = TRAJ[dcac_file]
    inv_dcac = get_single_inventory_row(dcac_file)

    I_DC_A = float(inv_dcac["DC_C"]) * ONE_C_A
    I_AC_A = float(inv_dcac["AC_C"]) * ONE_C_A
    f_meta_Hz = float(inv_dcac["frequency_Hz"])

    t_seg_hi_s = float(seg_row["t_segmentB_start_s"])
    q_seg_hi_Ah = float(seg_row["segment_A_Q_hi_Ah"])

    # Pull previous phase estimate from Cell 7 if available.
    pair_dtq = df_dtq_long[df_dtq_long["protocol_pair"] == pair]
    phi_prev_vals = pair_dtq["geometry_phase_offset_rad"].dropna().unique()
    phi_init = float(phi_prev_vals[0]) if len(phi_prev_vals) else 0.0

    fit = fit_sine_current_segment_A(
        df_traj=dcac_traj,
        t_hi_s=t_seg_hi_s,
        I0_init_A=I_DC_A,
        A_init_A=I_AC_A,
        f_init_Hz=f_meta_Hz,
        phi_init_rad=phi_init,
    )

    # DC reference geometry remains prescribed constant; DC error was already negligible.
    t_dc = dc_traj["t_s"].to_numpy(dtype=float)
    q_dc_meas = dc_traj["Q_net_Ah"].to_numpy(dtype=float)

    q_geom_dc = prescribed_geometry_Q_Ah(
        t_s=t_dc,
        I_DC_A=I_DC_A,
        I_AC_A=0.0,
        frequency_Hz=0.0,
        phase_rad=0.0,
    )

    t_dcac = dcac_traj["t_s"].to_numpy(dtype=float)
    q_dcac_meas = dcac_traj["Q_net_Ah"].to_numpy(dtype=float)

    if fit["fit_status"] == "ok":
        q_geom_dcac_fit = fitted_geometry_Q_Ah(
            t_s=t_dcac,
            I0_A=fit["I0_fit_A"],
            A_A=fit["A_fit_A"],
            f_Hz=fit["f_fit_Hz"],
            phi_rad=fit["phi_fit_rad"],
        )
    else:
        q_geom_dcac_fit = np.full(len(t_dcac), np.nan)

    # Only Segment-A grid points.
    segA = df_dtq_long[
        (df_dtq_long["protocol_pair"] == pair)
        & (df_dtq_long["segment_label"] == SEGMENT_A)
        & (df_dtq_long["is_anchor"] == False)
    ].copy()

    for _, row in segA.iterrows():
        q = float(row["Q_Ah"])

        t_dc_meas = first_passage_time_from_Q(q, t_dc, q_dc_meas)
        t_dc_geom = first_passage_time_from_Q(q, t_dc, q_geom_dc)

        t_dcac_meas = first_passage_time_from_Q(q, t_dcac, q_dcac_meas)
        t_dcac_fit_geom = first_passage_time_from_Q(q, t_dcac, q_geom_dcac_fit)

        dt_raw = (
            t_dc_meas - t_dcac_meas
            if is_finite_number(t_dc_meas) and is_finite_number(t_dcac_meas)
            else np.nan
        )

        dt_geom_fit = (
            t_dc_geom - t_dcac_fit_geom
            if is_finite_number(t_dc_geom) and is_finite_number(t_dcac_fit_geom)
            else np.nan
        )

        dt_resid_fit = (
            dt_raw - dt_geom_fit
            if is_finite_number(dt_raw) and is_finite_number(dt_geom_fit)
            else np.nan
        )

        fit_long_rows.append({
            "protocol_pair": pair,
            "Q_Ah": q,
            "dt_raw_s": dt_raw,
            "dt_geom_prescribed_s": float(row["dt_geom_s"]),
            "dt_resid_prescribed_s": float(row["dt_resid_s"]),
            "dt_geom_fitted_s": dt_geom_fit,
            "dt_resid_fitted_s": dt_resid_fit,
            "fit_status": fit["fit_status"],
            "I0_fit_A": fit["I0_fit_A"],
            "A_fit_A": fit["A_fit_A"],
            "f_meta_Hz": f_meta_Hz,
            "f_fit_Hz": fit["f_fit_Hz"],
            "f_fit_rel_error_ppm": (
                (fit["f_fit_Hz"] - f_meta_Hz) / f_meta_Hz * 1e6
                if fit["fit_status"] == "ok" and f_meta_Hz > 0
                else np.nan
            ),
            "phi_fit_rad": fit["phi_fit_rad"],
            "fit_rmse_A": fit["fit_rmse_A"],
        })

    df_pair = pd.DataFrame([r for r in fit_long_rows if r["protocol_pair"] == pair])

    def safe_mean(x):
        x = np.asarray(x, dtype=float)
        x = x[np.isfinite(x)]
        return float(np.mean(x)) if len(x) else np.nan

    def safe_p95_abs(x):
        x = np.asarray(x, dtype=float)
        x = x[np.isfinite(x)]
        return float(np.percentile(np.abs(x), 95)) if len(x) else np.nan

    def safe_max_abs(x):
        x = np.asarray(x, dtype=float)
        x = x[np.isfinite(x)]
        return float(np.max(np.abs(x))) if len(x) else np.nan

    fit_summary_rows.append({
        "protocol_pair": pair,
        "fit_status": fit["fit_status"],
        "I0_fit_A": fit["I0_fit_A"],
        "A_fit_A": fit["A_fit_A"],
        "f_meta_Hz": f_meta_Hz,
        "f_fit_Hz": fit["f_fit_Hz"],
        "f_fit_rel_error_ppm": (
            (fit["f_fit_Hz"] - f_meta_Hz) / f_meta_Hz * 1e6
            if fit["fit_status"] == "ok" and f_meta_Hz > 0
            else np.nan
        ),
        "phi_fit_rad": fit["phi_fit_rad"],
        "fit_rmse_A": fit["fit_rmse_A"],
        "fit_n": fit["fit_n"],

        "dt_resid_prescribed_mean_s": safe_mean(df_pair["dt_resid_prescribed_s"]),
        "dt_resid_prescribed_p95_abs_s": safe_p95_abs(df_pair["dt_resid_prescribed_s"]),
        "dt_resid_prescribed_max_abs_s": safe_max_abs(df_pair["dt_resid_prescribed_s"]),

        "dt_resid_fitted_mean_s": safe_mean(df_pair["dt_resid_fitted_s"]),
        "dt_resid_fitted_p95_abs_s": safe_p95_abs(df_pair["dt_resid_fitted_s"]),
        "dt_resid_fitted_max_abs_s": safe_max_abs(df_pair["dt_resid_fitted_s"]),
    })

df_fit_long = pd.DataFrame(fit_long_rows)
df_fit_summary = pd.DataFrame(fit_summary_rows)

df_fit_long.to_csv(OUT_FIT_GEOM_LONG, index=False)
df_fit_summary.to_csv(OUT_FIT_GEOM_SUMMARY, index=False)

print(f"[OK] Wrote fitted geometry residual long table: {OUT_FIT_GEOM_LONG}")
print(f"[OK] Wrote fitted geometry residual summary: {OUT_FIT_GEOM_SUMMARY}")

display_cols = [
    "protocol_pair",
    "fit_status",
    "I0_fit_A",
    "A_fit_A",
    "f_meta_Hz",
    "f_fit_Hz",
    "f_fit_rel_error_ppm",
    "phi_fit_rad",
    "fit_rmse_A",
    "dt_resid_prescribed_p95_abs_s",
    "dt_resid_fitted_p95_abs_s",
    "dt_resid_prescribed_max_abs_s",
    "dt_resid_fitted_max_abs_s",
]
print(df_fit_summary[display_cols].to_string(index=False))

print("\n[Top fitted residual points]")
for pair, g in df_fit_long.groupby("protocol_pair"):
    g = g.copy()
    g["abs_resid_fitted_s"] = g["dt_resid_fitted_s"].abs()
    top = g.sort_values("abs_resid_fitted_s", ascending=False).head(10)
    print("\n" + "=" * 120)
    print(pair)
    print("=" * 120)
    print(top[[
        "protocol_pair",
        "Q_Ah",
        "dt_raw_s",
        "dt_geom_prescribed_s",
        "dt_resid_prescribed_s",
        "dt_geom_fitted_s",
        "dt_resid_fitted_s",
        "abs_resid_fitted_s",
    ]].to_string(index=False))

print("\n[OK] Cell 7D fitted-waveform geometry diagnostic completed.")
print("[OK] No verdict performed in Cell 7D.")

[OK] Wrote fitted geometry residual long table: /Users/louislu/pybamm-dcac-superimposed/data/day21A_step5D_MJ1_fitted_geometry_residual_long.csv
[OK] Wrote fitted geometry residual summary: /Users/louislu/pybamm-dcac-superimposed/data/day21A_step5D_MJ1_fitted_geometry_residual_summary.csv
              protocol_pair fit_status  I0_fit_A  A_fit_A  f_meta_Hz  f_fit_Hz  f_fit_rel_error_ppm  phi_fit_rad  fit_rmse_A  dt_resid_prescribed_p95_abs_s  dt_resid_fitted_p95_abs_s  dt_resid_prescribed_max_abs_s  dt_resid_fitted_max_abs_s
0.3C DC vs 0.3C+0.7C 0.1tau         ok  1.020977 0.319165   0.143383  0.143292          -632.746581     2.560950    1.667262                       3.696359                   9.331629                      28.685838                  11.080290
 0.3C DC vs 0.3C+0.7C 10tau         ok  1.020309 2.379785   0.001434  0.001430         -2971.495055    -0.002292    0.008534                      10.835425                   0.794760                     426.430441               

In [21]:
# Cell 7E — Diagnostic synthesis before formal verdict
#
# Purpose:
# - Combine residual floor result, branch-jump audit, and fitted-waveform geometry audit
# - Classify whether Segment-A above-floor residual is likely mechanism-relevant
#   or dominated by waveform/first-passage diagnostics
#
# Explicitly NOT done here:
# - No formal mechanism verdict
# - No change to pre-registered thresholds
# - No overwrite of Cell 7 formal residual outputs

OUT_DIAGNOSTIC_SYNTHESIS = DATA_DIR / "day21A_step5E_MJ1_residual_diagnostic_synthesis.csv"

if not OUT_DTQ_SUMMARY.exists():
    raise FileNotFoundError(f"Missing dtQ summary: {OUT_DTQ_SUMMARY}")

if not OUT_BRANCH_JUMP_AUDIT.exists():
    raise FileNotFoundError(f"Missing branch-jump audit: {OUT_BRANCH_JUMP_AUDIT}")

if not OUT_FIT_GEOM_SUMMARY.exists():
    raise FileNotFoundError(f"Missing fitted geometry summary: {OUT_FIT_GEOM_SUMMARY}")

df_dtq_summary = pd.read_csv(OUT_DTQ_SUMMARY)
df_branch = pd.read_csv(OUT_BRANCH_JUMP_AUDIT)
df_fit_summary = pd.read_csv(OUT_FIT_GEOM_SUMMARY)

# Branch summary
df_branch_summary = (
    df_branch.groupby("protocol_pair")
    .agg(
        branch_n=("Q_Ah", "size"),
        branch_max_abs_resid_s=("abs_resid_s", "max"),
        branch_n_candidates=("branch_jump_artifact_candidate", "sum"),
        branch_frac_candidates=("branch_jump_artifact_candidate", "mean"),
    )
    .reset_index()
)

df_diag = (
    df_dtq_summary
    .merge(df_branch_summary, on="protocol_pair", how="left")
    .merge(df_fit_summary, on="protocol_pair", how="left", suffixes=("", "_fit"))
)

# Diagnostic-only fit quality rule.
# Not part of the pre-registered residual threshold.
df_diag["fit_rmse_fraction_of_A_fit"] = (
    df_diag["fit_rmse_A"] / df_diag["A_fit_A"].abs()
)

df_diag["fit_quality_status"] = np.where(
    (df_diag["fit_status"] == "ok")
    & np.isfinite(df_diag["fit_rmse_fraction_of_A_fit"])
    & (df_diag["fit_rmse_fraction_of_A_fit"] <= 0.10),
    "fit_usable",
    "fit_unreliable",
)

df_diag["fitted_p95_floor_status"] = np.where(
    df_diag["dt_resid_fitted_p95_abs_s"] <= SEG_A_FLOOR_COMPATIBLE_THRESHOLD_S,
    "fitted_p95_floor_compatible",
    np.where(
        df_diag["dt_resid_fitted_p95_abs_s"] >= SEG_A_REOPEN_THRESHOLD_S,
        "fitted_p95_above_reopen",
        "fitted_p95_intermediate",
    ),
)

df_diag["fitted_max_localized_spike_status"] = np.where(
    (df_diag["dt_resid_fitted_p95_abs_s"] <= SEG_A_FLOOR_COMPATIBLE_THRESHOLD_S)
    & (df_diag["dt_resid_fitted_max_abs_s"] >= SEG_A_REOPEN_THRESHOLD_S),
    "localized_first_passage_spike_after_fitting",
    "no_large_localized_spike_after_fitting",
)

def diagnostic_interpretation(row):
    formal_status = row["segment_A_above_floor_status"]
    fit_quality = row["fit_quality_status"]
    fitted_p95 = row["dt_resid_fitted_p95_abs_s"]
    fitted_max = row["dt_resid_fitted_max_abs_s"]

    # Formal Cell 7 result already spike-like
    if formal_status == ABOVE_FLOOR_SPIKE:
        if fit_quality == "fit_unreliable":
            return "prescribed_spike_fit_unreliable_not_mechanism_evidence"
        return "prescribed_spike_not_distributed_mechanism_evidence"

    # Above-floor by prescribed geometry, but fitted geometry collapses p95 into floor
    if formal_status in [ABOVE_FLOOR_YES, ABOVE_FLOOR_YES_NO_P95_GUARD]:
        if (
            fit_quality == "fit_usable"
            and np.isfinite(fitted_p95)
            and fitted_p95 <= SEG_A_FLOOR_COMPATIBLE_THRESHOLD_S
        ):
            if np.isfinite(fitted_max) and fitted_max >= SEG_A_REOPEN_THRESHOLD_S:
                return "waveform_geometry_mismatch_with_localized_first_passage_spike"
            return "waveform_geometry_mismatch_supported"

        if fit_quality == "fit_unreliable":
            return "above_floor_prescribed_but_fit_unreliable_requires_manual_review"

        return "above_floor_not_resolved_by_fitted_geometry"

    if formal_status == ABOVE_FLOOR_NO:
        return "floor_compatible"

    if formal_status == ABOVE_FLOOR_INTERMEDIATE:
        return "intermediate_residual_requires_caution"

    return "diagnostic_unresolved"

df_diag["diagnostic_interpretation"] = df_diag.apply(diagnostic_interpretation, axis=1)

# Keep a compact output table
cols = [
    "protocol_pair",
    "segment_A_above_floor_status",
    "segment_A_dt_resid_mean_s",
    "segment_A_dt_resid_max_abs_s",
    "segment_A_dt_resid_p95_abs_s",
    "branch_n_candidates",
    "branch_frac_candidates",
    "fit_status",
    "I0_fit_A",
    "A_fit_A",
    "f_fit_rel_error_ppm",
    "fit_rmse_A",
    "fit_rmse_fraction_of_A_fit",
    "fit_quality_status",
    "dt_resid_prescribed_p95_abs_s",
    "dt_resid_fitted_p95_abs_s",
    "dt_resid_prescribed_max_abs_s",
    "dt_resid_fitted_max_abs_s",
    "fitted_p95_floor_status",
    "fitted_max_localized_spike_status",
    "diagnostic_interpretation",
]

df_diag_out = df_diag[cols].copy()
df_diag_out.to_csv(OUT_DIAGNOSTIC_SYNTHESIS, index=False)

print(f"[OK] Wrote residual diagnostic synthesis: {OUT_DIAGNOSTIC_SYNTHESIS}")
print(df_diag_out.to_string(index=False))

print("\n[Interpretation guide]")
print("- formal Cell 7 residual status remains preserved.")
print("- fitted-waveform diagnostics are explanatory, not threshold redefinition.")
print("- If fitted p95 collapses below 2.70 s, prescribed above-floor residual is attributed to waveform-geometry mismatch unless contradicted by further evidence.")
print("- Large fitted max with low fitted p95 is treated as localized first-passage spike, not distributed Segment-A mechanism.")

[OK] Wrote residual diagnostic synthesis: /Users/louislu/pybamm-dcac-superimposed/data/day21A_step5E_MJ1_residual_diagnostic_synthesis.csv
              protocol_pair segment_A_above_floor_status  segment_A_dt_resid_mean_s  segment_A_dt_resid_max_abs_s  segment_A_dt_resid_p95_abs_s  branch_n_candidates  branch_frac_candidates fit_status  I0_fit_A  A_fit_A  f_fit_rel_error_ppm  fit_rmse_A  fit_rmse_fraction_of_A_fit fit_quality_status  dt_resid_prescribed_p95_abs_s  dt_resid_fitted_p95_abs_s  dt_resid_prescribed_max_abs_s  dt_resid_fitted_max_abs_s     fitted_p95_floor_status           fitted_max_localized_spike_status                                     diagnostic_interpretation
0.3C DC vs 0.3C+0.7C 0.1tau spike_or_transition_artifact                  -0.757979                     28.685838                      3.696359                   57                0.216730         ok  1.020977 0.319165          -632.746581    1.667262                    5.223827     fit_unreliable              

In [22]:
# Cell 8 — MJ1 formal mechanism verdict with diagnostic caveats
#
# Purpose:
# - Apply pre-registered mechanism verdict helper to each MJ1 DC-vs-DCAC pair
# - Preserve formal Cell 7 residual classification
# - Attach Cell 7E diagnostic caveats without redefining thresholds
# - Write MJ1-only verdict table using UNIFIED_VERDICT_SCHEMA
#
# Explicitly NOT done here:
# - No PyBaMM merge
# - No threshold redefinition
# - No post-hoc change of formal residual status

OUT_MJ1_VERDICT = DATA_DIR / "day21A_step6_MJ1_mechanism_verdict.csv"
OUT_MJ1_VERDICT_DIAGNOSTIC = DATA_DIR / "day21A_step6_MJ1_mechanism_verdict_diagnostic_view.csv"

required_files = [
    OUT_FILE_INVENTORY,
    OUT_EVENT_AUDIT,
    OUT_FINALQ_PAIR_AUDIT,
    OUT_SEGMENT_ASSIGNMENT,
    OUT_DTQ_SUMMARY,
    OUT_DIAGNOSTIC_SYNTHESIS,
]

for p in required_files:
    if not p.exists():
        raise FileNotFoundError(f"Required file missing: {p}")

df_inventory = pd.read_csv(OUT_FILE_INVENTORY)
df_event_audit = pd.read_csv(OUT_EVENT_AUDIT)
df_finalq_pairs = pd.read_csv(OUT_FINALQ_PAIR_AUDIT)
df_segment_assignment = pd.read_csv(OUT_SEGMENT_ASSIGNMENT)
df_dtq_summary = pd.read_csv(OUT_DTQ_SUMMARY)
df_diag = pd.read_csv(OUT_DIAGNOSTIC_SYNTHESIS)

assert_exact_schema(df_inventory, INVENTORY_SCHEMA, "INVENTORY_SCHEMA")


def one_row(df: pd.DataFrame, mask, label: str) -> pd.Series:
    rows = df.loc[mask]
    if len(rows) != 1:
        raise ValueError(f"Expected exactly one row for {label}, found {len(rows)}")
    return rows.iloc[0]


def get_inventory_by_file(file_name: str) -> pd.Series:
    return one_row(df_inventory, df_inventory["file_name"] == file_name, f"inventory:{file_name}")


def get_event_by_file(file_name: str) -> pd.Series:
    return one_row(df_event_audit, df_event_audit["file_name"] == file_name, f"event:{file_name}")


def get_dtq_summary_by_pair(protocol_pair: str) -> pd.Series:
    return one_row(df_dtq_summary, df_dtq_summary["protocol_pair"] == protocol_pair, f"dtq:{protocol_pair}")


def get_diag_by_pair(protocol_pair: str) -> pd.Series:
    return one_row(df_diag, df_diag["protocol_pair"] == protocol_pair, f"diag:{protocol_pair}")


def diagnostic_caveat_from_interpretation(diag_interpretation: str) -> str:
    mapping = {
        "prescribed_spike_fit_unreliable_not_mechanism_evidence":
            "diagnostic_prescribed_spike_fit_unreliable_not_mechanism_evidence",
        "prescribed_spike_not_distributed_mechanism_evidence":
            "diagnostic_prescribed_spike_not_distributed_mechanism_evidence",
        "waveform_geometry_mismatch_supported":
            "diagnostic_waveform_geometry_mismatch_supported",
        "waveform_geometry_mismatch_with_localized_first_passage_spike":
            "diagnostic_waveform_geometry_mismatch_with_localized_first_passage_spike",
        "above_floor_prescribed_but_fit_unreliable_requires_manual_review":
            "diagnostic_above_floor_prescribed_fit_unreliable_manual_review",
        "above_floor_not_resolved_by_fitted_geometry":
            "diagnostic_above_floor_not_resolved_by_fitted_geometry",
        "floor_compatible":
            "diagnostic_floor_compatible",
        "intermediate_residual_requires_caution":
            "diagnostic_intermediate_residual_requires_caution",
    }
    return mapping.get(str(diag_interpretation), f"diagnostic_{str(diag_interpretation)}")


def source_id_from_pair(protocol_pair: str) -> str:
    s = str(protocol_pair)
    s = s.replace(" ", "_").replace("+", "p").replace(".", "p")
    s = s.replace("τ", "tau")
    return f"MJ1_Day21A_{s}"


verdict_rows = []
diagnostic_view_rows = []

for _, seg in df_segment_assignment.iterrows():
    protocol_pair = seg["protocol_pair"]
    dc_file = seg["file_name_DC"]
    dcac_file = seg["file_name_DCAC"]

    inv_dcac = get_inventory_by_file(dcac_file)
    event_dc = get_event_by_file(dc_file)
    event_dcac = get_event_by_file(dcac_file)
    dtq = get_dtq_summary_by_pair(protocol_pair)
    diag = get_diag_by_pair(protocol_pair)

    formal = pre_registered_mechanism_verdict(
        q80_nominal_segment=seg["Q80_nominal_segment"],
        q90_nominal_segment=seg["Q90_nominal_segment"],
        q80_common_segment=seg["Q80_common_segment"],
        q90_common_segment=seg["Q90_common_segment"],
        segment_A_dt_resid_max_abs_s=dtq["segment_A_dt_resid_max_abs_s"],
        segment_A_dt_resid_p95_abs_s=dtq["segment_A_dt_resid_p95_abs_s"],
        segment_A_Q_grid_count=int(dtq["segment_A_Q_grid_count"]),
        q_vmax_ordering_status=seg["Q_Vmax_ordering_status"],
        segment_framework_status=seg["segment_framework_status"],
        geometry_phase_reference_status=dtq["geometry_phase_reference_status"],
        final_q_diff_status=seg["Q_final_diff_status"],
        late_CV_preservation_satisfied=dtq["late_CV_preservation_satisfied"],
    )

    caveat = formal["caveat"]
    caveat = append_caveat(caveat, diagnostic_caveat_from_interpretation(diag["diagnostic_interpretation"]))

    # Additional explicit diagnostic tags for traceability
    if str(diag["fit_quality_status"]) == "fit_unreliable":
        caveat = append_caveat(caveat, "diagnostic_fit_unreliable")

    if str(diag["fitted_max_localized_spike_status"]) == "localized_first_passage_spike_after_fitting":
        caveat = append_caveat(caveat, "diagnostic_localized_first_passage_spike_after_fitting")

    # Build schema-conforming row
    row = {col: np.nan if col in UNIFIED_VERDICT_NUMERIC_COLUMNS else UNKNOWN for col in UNIFIED_VERDICT_SCHEMA}

    # Source / protocol identity
    row["source_type"] = SOURCE_TYPE_MJ1
    row["source_id"] = source_id_from_pair(protocol_pair)
    row["cell_or_param_set"] = CELL_ID
    row["chemistry_family"] = CHEMISTRY_FAMILY
    row["protocol_pair"] = protocol_pair
    row["protocol_label_DC"] = seg["protocol_label_DC"]
    row["protocol_label_DCAC"] = seg["protocol_label_DCAC"]
    row["phase_convention"] = PHASE_CONVENTION

    # Measurement / provenance fields
    row["voltage_source"] = inv_dcac["voltage_source"]
    row["current_source"] = inv_dcac["current_source"]
    row["sampling_rate_Hz"] = inv_dcac["sampling_rate_Hz"]
    row["ambient_temperature_C"] = inv_dcac["ambient_temperature_C"]
    row["temperature_control_type"] = inv_dcac["temperature_control_type"]
    row["temperature_sensor_type"] = inv_dcac["temperature_sensor_type"]
    row["temperature_sensor_placement"] = inv_dcac["temperature_sensor_placement"]
    row["temperature_data_source"] = inv_dcac["temperature_data_source"]
    row["temperature_alignment_method"] = inv_dcac["temperature_alignment_method"]
    row["temperature_to_NGU201_alignment_required"] = inv_dcac["temperature_to_NGU201_alignment_required"]
    row["T_surface_max_C"] = inv_dcac["T_surface_max_C"]
    row["T_surface_mean_C"] = inv_dcac["T_surface_mean_C"]
    row["timebase_source"] = inv_dcac["timebase_source"]
    row["time_alignment_method"] = inv_dcac["time_alignment_method"]
    row["voltage_current_alignment_status"] = inv_dcac["voltage_current_alignment_status"]

    # Capacity and final-Q consistency
    row["Q_nom_Ah"] = Q_NOM_AH
    row["Q80_nominal_fraction_of_Q_nom"] = Q80_NOMINAL_FRACTION_OF_Q_NOM
    row["Q90_nominal_fraction_of_Q_nom"] = Q90_NOMINAL_FRACTION_OF_Q_NOM
    row["Q80_common_fraction_of_Q_nom"] = seg["Q80_common_fraction_of_Q_nom"]
    row["Q90_common_fraction_of_Q_nom"] = seg["Q90_common_fraction_of_Q_nom"]
    row["Q_final_DC_Ah"] = seg["Q_final_DC_Ah"]
    row["Q_final_DCAC_Ah"] = seg["Q_final_DCAC_Ah"]
    row["Q_final_diff_Ah"] = seg["Q_final_diff_Ah"]
    row["Q_final_diff_status"] = seg["Q_final_diff_status"]

    # Protocol constants
    row["Vmax_V"] = VMAX_V
    row["I_cutoff_A"] = I_CUTOFF_A
    row["I_cutoff_definition"] = I_CUTOFF_DEFINITION
    row["I_charge_onset_threshold_A"] = I_CHARGE_ONSET_THRESHOLD_A

    # Event detection methods and times
    row["Vmax_detection_method_used"] = (
        f"DC={event_dc['Vmax_detection_method_used']};"
        f"DCAC={event_dcac['Vmax_detection_method_used']}"
    )
    row["AC_off_detection_method_used"] = event_dcac["AC_off_detection_method_used"]
    row["t_Vmax_DC_s"] = seg["t_Vmax_DC_s"]
    row["t_Vmax_DCAC_s"] = seg["t_Vmax_DCAC_s"]
    row["t_AC_off_DC_s"] = np.nan
    row["t_AC_off_DCAC_s"] = seg["t_AC_off_DCAC_s"]
    row["AC_off_lag_s"] = seg["AC_off_lag_s"]
    row["t_segmentB_start_s"] = seg["t_segmentB_start_s"]

    # Voltage-boundary charges and segment boundary
    row["Q_Vmax_DC_Ah"] = seg["Q_Vmax_DC_Ah"]
    row["Q_Vmax_DCAC_Ah"] = seg["Q_Vmax_DCAC_Ah"]
    row["Q_segmentB_start_Ah"] = seg["Q_segmentB_start_Ah"]
    row["segment_A_Q_hi_definition"] = seg["segment_A_Q_hi_definition"]
    row["Q_Vmax_shift_Ah"] = seg["Q_Vmax_shift_Ah"]
    row["Q_Vmax_ordering_status"] = seg["Q_Vmax_ordering_status"]
    row["segment_framework_status"] = seg["segment_framework_status"]

    # Geometry phase reference
    row["geometry_phase_offset_s"] = dtq["geometry_phase_offset_s"]
    row["geometry_phase_offset_rad"] = dtq["geometry_phase_offset_rad"]
    row["geometry_phase_reference_status"] = dtq["geometry_phase_reference_status"]

    # Anchor definitions
    row["Q80_nominal_Ah"] = seg["Q80_nominal_Ah"]
    row["Q90_nominal_Ah"] = seg["Q90_nominal_Ah"]
    row["Q80_common_Ah"] = seg["Q80_common_Ah"]
    row["Q90_common_Ah"] = seg["Q90_common_Ah"]

    # Anchor segment assignment
    row["Q80_nominal_segment"] = seg["Q80_nominal_segment"]
    row["Q90_nominal_segment"] = seg["Q90_nominal_segment"]
    row["Q80_common_segment"] = seg["Q80_common_segment"]
    row["Q90_common_segment"] = seg["Q90_common_segment"]

    # Anchor raw time gains
    row["dt_Q80_nominal_raw_s"] = dtq["dt_Q80_nominal_raw_s"]
    row["dt_Q90_nominal_raw_s"] = dtq["dt_Q90_nominal_raw_s"]
    row["dt_Q80_common_raw_s"] = dtq["dt_Q80_common_raw_s"]
    row["dt_Q90_common_raw_s"] = dtq["dt_Q90_common_raw_s"]

    # Segment A statistics
    row["segment_A_Q_lo_Ah"] = dtq["segment_A_Q_lo_Ah"]
    row["segment_A_Q_hi_Ah"] = dtq["segment_A_Q_hi_Ah"]
    row["segment_A_Q_grid_count"] = dtq["segment_A_Q_grid_count"]
    row["segment_A_dt_raw_median_s"] = dtq["segment_A_dt_raw_median_s"]
    row["segment_A_dt_raw_max_s"] = dtq["segment_A_dt_raw_max_s"]
    row["segment_A_dt_resid_mean_s"] = dtq["segment_A_dt_resid_mean_s"]
    row["segment_A_dt_resid_max_abs_s"] = dtq["segment_A_dt_resid_max_abs_s"]
    row["segment_A_dt_resid_p95_abs_s"] = dtq["segment_A_dt_resid_p95_abs_s"]
    row["segment_A_resid_floor_s"] = dtq["segment_A_resid_floor_s"]
    row["segment_A_resid_floor_type"] = dtq["segment_A_resid_floor_type"]
    row["segment_A_resid_floor_n"] = dtq["segment_A_resid_floor_n"]
    row["segment_A_above_floor_status"] = dtq["segment_A_above_floor_status"]

    # Segment B statistics
    row["segment_B_Q_lo_Ah"] = dtq["segment_B_Q_lo_Ah"]
    row["segment_B_Q_hi_Ah"] = dtq["segment_B_Q_hi_Ah"]
    row["segment_B_dt_raw_median_s"] = dtq["segment_B_dt_raw_median_s"]
    row["segment_B_dt_raw_max_s"] = dtq["segment_B_dt_raw_max_s"]

    # Segment D statistics and late-CV preservation
    row["segment_D_Q_lo_Ah"] = dtq["segment_D_Q_lo_Ah"]
    row["segment_D_Q_hi_Ah"] = dtq["segment_D_Q_hi_Ah"]
    row["segment_D_dt_raw_min_s"] = dtq["segment_D_dt_raw_min_s"]
    row["segment_D_dt_raw_median_s"] = dtq["segment_D_dt_raw_median_s"]
    row["segment_D_dt_raw_max_s"] = dtq["segment_D_dt_raw_max_s"]
    row["late_CV_preservation_threshold_s"] = dtq["late_CV_preservation_threshold_s"]
    row["late_CV_preservation_satisfied"] = dtq["late_CV_preservation_satisfied"]

    # Verdict
    row["evidence_status"] = formal["evidence_status"]
    row["mechanism_verdict"] = formal["mechanism_verdict"]
    row["interpretation_class"] = formal["interpretation_class"]
    row["caveat"] = caveat

    verdict_rows.append(row)

    diagnostic_view_rows.append({
        "protocol_pair": protocol_pair,
        "formal_evidence_status": formal["evidence_status"],
        "formal_mechanism_verdict": formal["mechanism_verdict"],
        "formal_interpretation_class": formal["interpretation_class"],
        "segment_A_above_floor_status": dtq["segment_A_above_floor_status"],
        "diagnostic_interpretation": diag["diagnostic_interpretation"],
        "fit_quality_status": diag["fit_quality_status"],
        "dt_resid_prescribed_p95_abs_s": diag["dt_resid_prescribed_p95_abs_s"],
        "dt_resid_fitted_p95_abs_s": diag["dt_resid_fitted_p95_abs_s"],
        "dt_resid_prescribed_max_abs_s": diag["dt_resid_prescribed_max_abs_s"],
        "dt_resid_fitted_max_abs_s": diag["dt_resid_fitted_max_abs_s"],
        "caveat": caveat,
    })


df_mj1_verdict = pd.DataFrame(verdict_rows)
df_mj1_diag_view = pd.DataFrame(diagnostic_view_rows)

# Enforce frozen unified schema for MJ1 verdict table.
df_mj1_verdict = df_mj1_verdict[UNIFIED_VERDICT_SCHEMA].copy()
df_mj1_verdict = fill_unknown_strings_and_nan_numeric(
    df_mj1_verdict,
    numeric_columns=UNIFIED_VERDICT_NUMERIC_COLUMNS,
    unknown=UNKNOWN,
)

assert_exact_schema(df_mj1_verdict, UNIFIED_VERDICT_SCHEMA, "UNIFIED_VERDICT_SCHEMA")

df_mj1_verdict.to_csv(OUT_MJ1_VERDICT, index=False)
df_mj1_diag_view.to_csv(OUT_MJ1_VERDICT_DIAGNOSTIC, index=False)

print(f"[OK] Wrote MJ1 formal verdict table: {OUT_MJ1_VERDICT}")
print(f"[OK] Wrote MJ1 diagnostic verdict view: {OUT_MJ1_VERDICT_DIAGNOSTIC}")

display_cols = [
    "protocol_pair",
    "evidence_status",
    "mechanism_verdict",
    "interpretation_class",
    "segment_A_above_floor_status",
    "dt_Q80_common_raw_s",
    "dt_Q90_common_raw_s",
    "Q80_common_segment",
    "Q90_common_segment",
    "caveat",
]
print(df_mj1_verdict[display_cols].to_string(index=False))

print("\n[Diagnostic view]")
print(df_mj1_diag_view.to_string(index=False))

print("\n[OK] Cell 8 MJ1 verdict completed.")
print("[OK] No PyBaMM merge performed in Cell 8.")

[OK] Wrote MJ1 formal verdict table: /Users/louislu/pybamm-dcac-superimposed/data/day21A_step6_MJ1_mechanism_verdict.csv
[OK] Wrote MJ1 diagnostic verdict view: /Users/louislu/pybamm-dcac-superimposed/data/day21A_step6_MJ1_mechanism_verdict_diagnostic_view.csv
              protocol_pair evidence_status                    mechanism_verdict                           interpretation_class segment_A_above_floor_status  dt_Q80_common_raw_s  dt_Q90_common_raw_s                     Q80_common_segment                     Q90_common_segment                                                                                                                                                                                             caveat
0.3C DC vs 0.3C+0.7C 0.1tau       ambiguous ambiguous_defer_mechanism_commitment                   spike_or_transition_artifact spike_or_transition_artifact             0.196020           351.715593            A_shared_prescribed_current B_voltage_boundary_control_st

In [23]:
# Cell 9A — locate Day20B PyBaMM full-protocol outputs

from pathlib import Path

candidate_patterns = [
    "*day20B*.csv",
    "*Day20B*.csv",
    "*full*protocol*.csv",
    "*CC*CV*.csv",
    "*verdict*.csv",
    "*segment*.csv",
]

matches = []
for pattern in candidate_patterns:
    matches.extend(sorted(DATA_DIR.glob(pattern)))

# Unique, sorted
matches = sorted(set(matches), key=lambda p: str(p))

print(f"[scan] DATA_DIR = {DATA_DIR}")
print(f"[scan] candidate CSV files found = {len(matches)}")

for p in matches:
    print(p.relative_to(REPO))

[scan] DATA_DIR = /Users/louislu/pybamm-dcac-superimposed/data
[scan] candidate CSV files found = 33
data/day19A_step3_legacy_lineage_verdict_summary.csv
data/day19A_step6_final_verdict_summary.csv
data/day20B_step0_full_protocol_design.csv
data/day20B_step1_Chen2020_full_protocol_smoke_pair_summary.csv
data/day20B_step1_Chen2020_full_protocol_smoke_summary.csv
data/day20B_step1_Chen2020_full_protocol_smoke_trajectories.csv
data/day20B_step2_Chen2020_Q_monotonicity_diagnostic.csv
data/day20B_step2_Chen2020_full_protocol_anchor_points.csv
data/day20B_step2_Chen2020_full_protocol_dtQ_segments.csv
data/day20B_step2_Chen2020_full_protocol_segment_summary.csv
data/day20B_step3_Chen2020_full_protocol_smoke_verdict.csv
data/day20B_step4_full_protocol_batch_errors.csv
data/day20B_step4_full_protocol_batch_pair_summary.csv
data/day20B_step4_full_protocol_batch_summary.csv
data/day20B_step4_full_protocol_batch_trajectories.csv
data/day20B_step5_full_protocol_batch_Q_monotonicity_diagnostic.csv
d

In [24]:
# Cell 9B — Inspect Day20B PyBaMM full-protocol output schemas
#
# Purpose:
# - Inspect candidate Day20B PyBaMM CSV schemas before merging with MJ1
# - Do not merge
# - Do not reinterpret
# - Do not modify files

DAY20B_CANDIDATES = {
    "batch_verdict": DATA_DIR / "day20B_step6_full_protocol_batch_verdict.csv",
    "segment_summary": DATA_DIR / "day20B_step5_full_protocol_batch_segment_summary.csv",
    "anchor_points": DATA_DIR / "day20B_step5_full_protocol_batch_anchor_points.csv",
    "pair_summary": DATA_DIR / "day20B_step4_full_protocol_batch_pair_summary.csv",
    "current_audit": DATA_DIR / "day20B_step6_full_protocol_current_audit.csv",
}

for label, path in DAY20B_CANDIDATES.items():
    print("\n" + "=" * 120)
    print(f"[{label}] {path.relative_to(REPO)}")
    print("=" * 120)

    if not path.exists():
        print("[missing]")
        continue

    df = pd.read_csv(path)
    print(f"[shape] {df.shape}")
    print("[columns]")
    for c in df.columns:
        print(f"  - {c}")

    print("\n[head]")
    print(df.head(5).to_string(index=False))


[batch_verdict] data/day20B_step6_full_protocol_batch_verdict.csv
[shape] (3, 34)
[columns]
  - param_set
  - full_protocol_ok
  - evidence_status
  - dt_total_min
  - dt_total_s
  - Q_end_DC_Ah
  - Q_end_DCAC_Ah
  - Q_end_diff_Ah
  - Q_to_Vmax_DC_Ah
  - Q_to_Vmax_DCAC_Ah
  - Q_to_Vmax_shift_Ah
  - dt_to_Vmax_s
  - segment_A_median_dt_min
  - segment_B_median_dt_min
  - segment_D_median_dt_min
  - Q80_common_dt_min
  - Q90_common_dt_min
  - Q80_nominal_5Ah_dt_min
  - Q90_nominal_5Ah_dt_min
  - anchors_Q80_Q90_common_in_segment_B
  - anchors_Q80_Q90_nominal_in_segment_B
  - DCAC_CC_charge_peak_C
  - DCAC_CC_discharge_peak_C
  - DCAC_CC_peak_charge_ok
  - DCAC_CC_peak_discharge_ok
  - DCAC_CV_charge_peak_C
  - DCAC_CV_discharge_peak_C
  - DCAC_CV_current_transient_warning
  - DCAC_CV_charge_above_CC_peak
  - DC_Q_negative_steps
  - DCAC_Q_negative_steps
  - DCAC_has_positive_discharge_current
  - mechanism_verdict
  - interpretation

[head]
 param_set  full_protocol_ok                  

In [25]:
# Cell 9C — Build unified MJ1–PyBaMM mechanism verdict table
#
# Purpose:
# - Adapt Day20B PyBaMM full-protocol verdict outputs into UNIFIED_VERDICT_SCHEMA
# - Concatenate MJ1 Day21A verdict rows with PyBaMM Day20B rows
# - Preserve source-specific caveats
#
# Explicitly NOT done here:
# - No recomputation of PyBaMM results
# - No recomputation of MJ1 results
# - No threshold redefinition
# - No new mechanism interpretation beyond existing source verdicts

OUT_UNIFIED_MJ1_PYBAMM = DATA_DIR / "day21A_step7_unified_MJ1_PyBaMM_mechanism_verdict.csv"

PYBAMM_BATCH_VERDICT = DATA_DIR / "day20B_step6_full_protocol_batch_verdict.csv"
PYBAMM_SEGMENT_SUMMARY = DATA_DIR / "day20B_step5_full_protocol_batch_segment_summary.csv"
PYBAMM_ANCHOR_POINTS = DATA_DIR / "day20B_step5_full_protocol_batch_anchor_points.csv"
PYBAMM_PAIR_SUMMARY = DATA_DIR / "day20B_step4_full_protocol_batch_pair_summary.csv"

if not OUT_MJ1_VERDICT.exists():
    raise FileNotFoundError(f"MJ1 verdict file missing: {OUT_MJ1_VERDICT}")

for p in [
    PYBAMM_BATCH_VERDICT,
    PYBAMM_SEGMENT_SUMMARY,
    PYBAMM_ANCHOR_POINTS,
    PYBAMM_PAIR_SUMMARY,
]:
    if not p.exists():
        raise FileNotFoundError(f"Required PyBaMM Day20B file missing: {p}")

df_mj1_verdict = pd.read_csv(OUT_MJ1_VERDICT)
df_pybamm_verdict_src = pd.read_csv(PYBAMM_BATCH_VERDICT)
df_pybamm_segments = pd.read_csv(PYBAMM_SEGMENT_SUMMARY)
df_pybamm_anchors = pd.read_csv(PYBAMM_ANCHOR_POINTS)
df_pybamm_pairs = pd.read_csv(PYBAMM_PAIR_SUMMARY)

assert_exact_schema(df_mj1_verdict, UNIFIED_VERDICT_SCHEMA, "UNIFIED_VERDICT_SCHEMA")

print(f"[OK] Loaded MJ1 verdict: {OUT_MJ1_VERDICT}")
print(f"[OK] Loaded PyBaMM Day20B verdict source: {PYBAMM_BATCH_VERDICT}")


# -----------------------------------------------------------------------------
# 9C.1 Mapping helpers
# -----------------------------------------------------------------------------

PYBAMM_SEGMENT_MAP = {
    "A_pre_DCAC_Vmax": SEGMENT_A,
    "B_between_DCAC_Vmax_and_DC_Vmax": SEGMENT_B,
    "D_late_CV_feedback_region": SEGMENT_D,
}


def pybamm_source_id(param_set: str) -> str:
    return f"PyBaMM_Day20B_{param_set}_full_protocol"


def pybamm_protocol_pair(param_set: str) -> str:
    return f"{param_set}: 0.2C DC vs 0.2C+0.5C charge-first full protocol"


def one_param_row(df: pd.DataFrame, param_set: str, label: str) -> pd.Series:
    rows = df[df["param_set"] == param_set]
    if len(rows) != 1:
        raise ValueError(f"Expected one row for {label}/{param_set}, found {len(rows)}")
    return rows.iloc[0]


def segment_row(param_set: str, segment_name: str) -> pd.Series:
    rows = df_pybamm_segments[
        (df_pybamm_segments["param_set"] == param_set)
        & (df_pybamm_segments["segment"] == segment_name)
    ]
    if len(rows) != 1:
        raise ValueError(
            f"Expected one segment row for {param_set}/{segment_name}, found {len(rows)}"
        )
    return rows.iloc[0]


def anchor_row(param_set: str, anchor_label: str) -> pd.Series:
    rows = df_pybamm_anchors[
        (df_pybamm_anchors["param_set"] == param_set)
        & (df_pybamm_anchors["anchor_label"] == anchor_label)
    ]
    if len(rows) != 1:
        raise ValueError(
            f"Expected one anchor row for {param_set}/{anchor_label}, found {len(rows)}"
        )
    return rows.iloc[0]


def map_pybamm_segment(seg: str) -> str:
    if seg not in PYBAMM_SEGMENT_MAP:
        return SEGMENT_UNRESOLVED
    return PYBAMM_SEGMENT_MAP[seg]


def status_from_pybamm_q_end_diff(q_diff_Ah: float) -> str:
    if not is_finite_number(q_diff_Ah):
        return FINAL_Q_UNRESOLVED
    return (
        FINAL_Q_CONSISTENT
        if abs(float(q_diff_Ah)) <= FINAL_Q_DIFF_THRESHOLD_AH
        else FINAL_Q_MISMATCH_WARNING
    )


def pybamm_caveat_from_row(row: pd.Series) -> str:
    caveat = "source=PyBaMM_Day20B_full_protocol_batch"

    caveat = append_caveat(
        caveat,
        "segment_A_residual_not_recomputed_in_unified_adapter",
    )

    if bool(row.get("DCAC_CV_current_transient_warning", False)):
        caveat = append_caveat(caveat, "DCAC_CV_current_transient_warning")

    if bool(row.get("DCAC_CV_charge_above_CC_peak", False)):
        caveat = append_caveat(caveat, "DCAC_CV_charge_above_CC_peak")

    if not bool(row.get("DCAC_CC_peak_charge_ok", True)):
        caveat = append_caveat(caveat, "DCAC_CC_peak_charge_not_ok")

    if not bool(row.get("DCAC_CC_peak_discharge_ok", True)):
        caveat = append_caveat(caveat, "DCAC_CC_peak_discharge_not_ok")

    return caveat


# -----------------------------------------------------------------------------
# 9C.2 Adapt PyBaMM Day20B rows to UNIFIED_VERDICT_SCHEMA
# -----------------------------------------------------------------------------

pybamm_rows = []

for _, src in df_pybamm_verdict_src.iterrows():
    param_set = str(src["param_set"])
    pair = one_param_row(df_pybamm_pairs, param_set, "pair_summary")

    segA = segment_row(param_set, "A_pre_DCAC_Vmax")
    segB = segment_row(param_set, "B_between_DCAC_Vmax_and_DC_Vmax")
    segD = segment_row(param_set, "D_late_CV_feedback_region")

    a_q80_common = anchor_row(param_set, "Q80_common")
    a_q90_common = anchor_row(param_set, "Q90_common")
    a_q80_nom = anchor_row(param_set, "Q80_nominal_5Ah")
    a_q90_nom = anchor_row(param_set, "Q90_nominal_5Ah")

    row = {
        col: np.nan if col in UNIFIED_VERDICT_NUMERIC_COLUMNS else UNKNOWN
        for col in UNIFIED_VERDICT_SCHEMA
    }

    # Source / protocol identity
    row["source_type"] = "PyBaMM_Day20B_full_protocol"
    row["source_id"] = pybamm_source_id(param_set)
    row["cell_or_param_set"] = param_set
    row["chemistry_family"] = "PyBaMM_parameter_set_layered_oxide_family"
    row["protocol_pair"] = pybamm_protocol_pair(param_set)
    row["protocol_label_DC"] = "0.2C DC full CC-CV"
    row["protocol_label_DCAC"] = "0.2C+0.5C charge-first full CC-CV"
    row["phase_convention"] = "charge_first"

    # Measurement / provenance fields
    row["voltage_source"] = "PyBaMM_simulation"
    row["current_source"] = "PyBaMM_simulation"
    row["sampling_rate_Hz"] = np.nan
    row["ambient_temperature_C"] = np.nan
    row["temperature_control_type"] = "not_applicable_simulation"
    row["temperature_sensor_type"] = "not_applicable_simulation"
    row["temperature_sensor_placement"] = "not_applicable_simulation"
    row["temperature_data_source"] = "not_applicable_simulation"
    row["temperature_alignment_method"] = "not_applicable_simulation"
    row["temperature_to_NGU201_alignment_required"] = False
    row["T_surface_max_C"] = np.nan
    row["T_surface_mean_C"] = np.nan
    row["timebase_source"] = "PyBaMM_solution_timebase"
    row["time_alignment_method"] = "native_simulation_solution_timebase"
    row["voltage_current_alignment_status"] = "native_aligned_same_solution"

    # Capacity and final-Q consistency
    row["Q_nom_Ah"] = 5.0
    row["Q80_nominal_fraction_of_Q_nom"] = 0.80
    row["Q90_nominal_fraction_of_Q_nom"] = 0.90
    row["Q80_common_fraction_of_Q_nom"] = float(a_q80_common["Q_frac_nominal_5Ah"])
    row["Q90_common_fraction_of_Q_nom"] = float(a_q90_common["Q_frac_nominal_5Ah"])
    row["Q_final_DC_Ah"] = float(src["Q_end_DC_Ah"])
    row["Q_final_DCAC_Ah"] = float(src["Q_end_DCAC_Ah"])
    row["Q_final_diff_Ah"] = float(src["Q_end_diff_Ah"])
    row["Q_final_diff_status"] = status_from_pybamm_q_end_diff(src["Q_end_diff_Ah"])

    # Protocol constants
    row["Vmax_V"] = 4.2
    row["I_cutoff_A"] = np.nan
    row["I_cutoff_definition"] = "normalized_cutoff_0p05_over_3p4_times_Q_nom"
    row["I_charge_onset_threshold_A"] = np.nan

    # Event detection methods and times
    row["Vmax_detection_method_used"] = "PyBaMM_event_time_to_Vmax_from_simulation"
    row["AC_off_detection_method_used"] = "protocol_defined_AC_off_at_Vmax"
    row["t_Vmax_DC_s"] = float(pair["t_to_Vmax_DC_s"])
    row["t_Vmax_DCAC_s"] = float(pair["t_to_Vmax_DCAC_s"])
    row["t_AC_off_DC_s"] = np.nan
    row["t_AC_off_DCAC_s"] = float(pair["t_to_Vmax_DCAC_s"])
    row["AC_off_lag_s"] = 0.0
    row["t_segmentB_start_s"] = float(pair["t_to_Vmax_DCAC_s"])

    # Voltage-boundary charges and segment boundary
    row["Q_Vmax_DC_Ah"] = float(src["Q_to_Vmax_DC_Ah"])
    row["Q_Vmax_DCAC_Ah"] = float(src["Q_to_Vmax_DCAC_Ah"])
    row["Q_segmentB_start_Ah"] = float(src["Q_to_Vmax_DCAC_Ah"])
    row["segment_A_Q_hi_definition"] = "Q_segmentB_start_Ah"
    row["Q_Vmax_shift_Ah"] = float(src["Q_to_Vmax_shift_Ah"])
    row["Q_Vmax_ordering_status"] = ORDERING_EXPECTED
    row["segment_framework_status"] = SEGMENT_FRAMEWORK_OK

    # Geometry phase reference
    row["geometry_phase_offset_s"] = np.nan
    row["geometry_phase_offset_rad"] = np.nan
    row["geometry_phase_reference_status"] = "not_applicable_simulation_day20B_adapter"

    # Anchor definitions
    row["Q80_nominal_Ah"] = float(a_q80_nom["Q_Ah"])
    row["Q90_nominal_Ah"] = float(a_q90_nom["Q_Ah"])
    row["Q80_common_Ah"] = float(a_q80_common["Q_Ah"])
    row["Q90_common_Ah"] = float(a_q90_common["Q_Ah"])

    # Anchor segment assignment
    row["Q80_nominal_segment"] = map_pybamm_segment(a_q80_nom["segment"])
    row["Q90_nominal_segment"] = map_pybamm_segment(a_q90_nom["segment"])
    row["Q80_common_segment"] = map_pybamm_segment(a_q80_common["segment"])
    row["Q90_common_segment"] = map_pybamm_segment(a_q90_common["segment"])

    # Anchor raw time gains
    row["dt_Q80_nominal_raw_s"] = float(a_q80_nom["dt_full_s"])
    row["dt_Q90_nominal_raw_s"] = float(a_q90_nom["dt_full_s"])
    row["dt_Q80_common_raw_s"] = float(a_q80_common["dt_full_s"])
    row["dt_Q90_common_raw_s"] = float(a_q90_common["dt_full_s"])

    # Segment A statistics
    row["segment_A_Q_lo_Ah"] = float(segA["Q_lo_Ah"])
    row["segment_A_Q_hi_Ah"] = float(segA["Q_hi_Ah"])
    row["segment_A_Q_grid_count"] = float(segA["n_finite"])
    row["segment_A_dt_raw_median_s"] = float(segA["dt_full_median_s"])
    row["segment_A_dt_raw_max_s"] = float(segA["dt_full_max_s"])

    # PyBaMM Day20B batch does not export Segment-A geometry residual fields in this table.
    row["segment_A_dt_resid_mean_s"] = np.nan
    row["segment_A_dt_resid_max_abs_s"] = np.nan
    row["segment_A_dt_resid_p95_abs_s"] = np.nan
    row["segment_A_resid_floor_s"] = np.nan
    row["segment_A_resid_floor_type"] = "not_computed_in_day20B_unified_adapter"
    row["segment_A_resid_floor_n"] = np.nan
    row["segment_A_above_floor_status"] = "not_computed_in_day20B_unified_adapter"

    # Segment B statistics
    row["segment_B_Q_lo_Ah"] = float(segB["Q_lo_Ah"])
    row["segment_B_Q_hi_Ah"] = float(segB["Q_hi_Ah"])
    row["segment_B_dt_raw_median_s"] = float(segB["dt_full_median_s"])
    row["segment_B_dt_raw_max_s"] = float(segB["dt_full_max_s"])

    # Segment D statistics and late-CV preservation
    row["segment_D_Q_lo_Ah"] = float(segD["Q_lo_Ah"])
    row["segment_D_Q_hi_Ah"] = float(segD["Q_hi_Ah"])
    row["segment_D_dt_raw_min_s"] = float(segD["dt_full_min_s"])
    row["segment_D_dt_raw_median_s"] = float(segD["dt_full_median_s"])
    row["segment_D_dt_raw_max_s"] = float(segD["dt_full_max_s"])
    row["late_CV_preservation_threshold_s"] = LATE_CV_PRESERVATION_THRESHOLD_S
    row["late_CV_preservation_satisfied"] = (
        LATE_CV_SATISFIED
        if float(segD["dt_full_median_s"]) > LATE_CV_PRESERVATION_THRESHOLD_S
        else LATE_CV_NOT_SATISFIED
    )

    # Verdict
    row["evidence_status"] = str(src["evidence_status"])
    row["mechanism_verdict"] = str(src["mechanism_verdict"])
    row["interpretation_class"] = "boundary_control_state_mediated_first_passage_gain"
    row["caveat"] = pybamm_caveat_from_row(src)

    pybamm_rows.append(row)


df_pybamm_unified = pd.DataFrame(pybamm_rows)
df_pybamm_unified = df_pybamm_unified[UNIFIED_VERDICT_SCHEMA].copy()
df_pybamm_unified = fill_unknown_strings_and_nan_numeric(
    df_pybamm_unified,
    numeric_columns=UNIFIED_VERDICT_NUMERIC_COLUMNS,
    unknown=UNKNOWN,
)

assert_exact_schema(df_pybamm_unified, UNIFIED_VERDICT_SCHEMA, "UNIFIED_VERDICT_SCHEMA")

print("[OK] Adapted PyBaMM Day20B rows to unified schema.")
print(df_pybamm_unified[[
    "source_type",
    "cell_or_param_set",
    "evidence_status",
    "mechanism_verdict",
    "Q80_common_segment",
    "Q90_common_segment",
    "dt_Q80_common_raw_s",
    "dt_Q90_common_raw_s",
    "segment_A_above_floor_status",
    "caveat",
]].to_string(index=False))


# -----------------------------------------------------------------------------
# 9C.3 Concatenate MJ1 and PyBaMM unified rows
# -----------------------------------------------------------------------------

df_unified = pd.concat(
    [df_mj1_verdict, df_pybamm_unified],
    ignore_index=True,
)

df_unified = df_unified[UNIFIED_VERDICT_SCHEMA].copy()
df_unified = fill_unknown_strings_and_nan_numeric(
    df_unified,
    numeric_columns=UNIFIED_VERDICT_NUMERIC_COLUMNS,
    unknown=UNKNOWN,
)

assert_exact_schema(df_unified, UNIFIED_VERDICT_SCHEMA, "UNIFIED_VERDICT_SCHEMA")

df_unified.to_csv(OUT_UNIFIED_MJ1_PYBAMM, index=False)

print(f"\n[OK] Wrote unified MJ1–PyBaMM verdict table: {OUT_UNIFIED_MJ1_PYBAMM}")
print(f"[OK] unified shape = {df_unified.shape}")

display_cols = [
    "source_type",
    "cell_or_param_set",
    "protocol_pair",
    "evidence_status",
    "mechanism_verdict",
    "interpretation_class",
    "Q80_common_segment",
    "Q90_common_segment",
    "dt_Q80_common_raw_s",
    "dt_Q90_common_raw_s",
    "segment_A_above_floor_status",
    "caveat",
]

print(df_unified[display_cols].to_string(index=False))

[OK] Loaded MJ1 verdict: /Users/louislu/pybamm-dcac-superimposed/data/day21A_step6_MJ1_mechanism_verdict.csv
[OK] Loaded PyBaMM Day20B verdict source: /Users/louislu/pybamm-dcac-superimposed/data/day20B_step6_full_protocol_batch_verdict.csv
[OK] Adapted PyBaMM Day20B rows to unified schema.
                source_type cell_or_param_set                         evidence_status                                                                   mechanism_verdict                     Q80_common_segment                     Q90_common_segment  dt_Q80_common_raw_s  dt_Q90_common_raw_s           segment_A_above_floor_status                                                                                                                                                       caveat
PyBaMM_Day20B_full_protocol          Chen2020                                   valid positive_full_protocol_raw_gain_boundary_control_state_split_not_Segment_A_residual B_voltage_boundary_control_state_split B_voltage_bou

In [26]:
# Cell 10A — Day21A closure summary CSV
#
# Purpose:
# - Close Day21A audit chain in machine-readable form
# - No recomputation
# - No new thresholds
# - No new verdict logic

OUT_DAY21A_CLOSURE_CSV = DATA_DIR / "day21A_step8_closure_summary.csv"
OUT_DAY21A_CLOSURE_MD = DATA_DIR / "day21A_step8_closure_note.md"

required_closure_files = [
    OUT_AUDIT_CONTRACT_JSON,
    OUT_FILE_INVENTORY,
    OUT_LOAD_SUMMARY,
    OUT_EVENT_AUDIT,
    OUT_Q_SUMMARY,
    OUT_FINALQ_PAIR_AUDIT,
    OUT_SEGMENT_ASSIGNMENT,
    OUT_DTQ_SUMMARY,
    OUT_RESID_DIAG,
    OUT_BRANCH_JUMP_AUDIT,
    OUT_GEOM_FIDELITY_SUMMARY,
    OUT_DIAGNOSTIC_SYNTHESIS,
    OUT_MJ1_VERDICT,
    OUT_UNIFIED_MJ1_PYBAMM,
]

missing = [p for p in required_closure_files if not p.exists()]
if missing:
    raise FileNotFoundError(
        "Cannot close Day21A. Missing required files:\n"
        + "\n".join(str(p) for p in missing)
    )

df_mj1_verdict = pd.read_csv(OUT_MJ1_VERDICT)
df_unified = pd.read_csv(OUT_UNIFIED_MJ1_PYBAMM)
df_diag = pd.read_csv(OUT_DIAGNOSTIC_SYNTHESIS)

assert_exact_schema(df_mj1_verdict, UNIFIED_VERDICT_SCHEMA, "UNIFIED_VERDICT_SCHEMA")
assert_exact_schema(df_unified, UNIFIED_VERDICT_SCHEMA, "UNIFIED_VERDICT_SCHEMA")

closure_rows = []

# MJ1 compact closure rows
for _, row in df_mj1_verdict.iterrows():
    closure_rows.append({
        "source_type": row["source_type"],
        "cell_or_param_set": row["cell_or_param_set"],
        "protocol_pair": row["protocol_pair"],
        "evidence_status": row["evidence_status"],
        "mechanism_verdict": row["mechanism_verdict"],
        "interpretation_class": row["interpretation_class"],
        "Q80_common_segment": row["Q80_common_segment"],
        "Q90_common_segment": row["Q90_common_segment"],
        "dt_Q80_common_raw_s": row["dt_Q80_common_raw_s"],
        "dt_Q90_common_raw_s": row["dt_Q90_common_raw_s"],
        "segment_A_above_floor_status": row["segment_A_above_floor_status"],
        "caveat": row["caveat"],
    })

# PyBaMM compact closure rows
df_pybamm_closure = df_unified[
    df_unified["source_type"] == "PyBaMM_Day20B_full_protocol"
].copy()

for _, row in df_pybamm_closure.iterrows():
    closure_rows.append({
        "source_type": row["source_type"],
        "cell_or_param_set": row["cell_or_param_set"],
        "protocol_pair": row["protocol_pair"],
        "evidence_status": row["evidence_status"],
        "mechanism_verdict": row["mechanism_verdict"],
        "interpretation_class": row["interpretation_class"],
        "Q80_common_segment": row["Q80_common_segment"],
        "Q90_common_segment": row["Q90_common_segment"],
        "dt_Q80_common_raw_s": row["dt_Q80_common_raw_s"],
        "dt_Q90_common_raw_s": row["dt_Q90_common_raw_s"],
        "segment_A_above_floor_status": row["segment_A_above_floor_status"],
        "caveat": row["caveat"],
    })

df_closure = pd.DataFrame(closure_rows)
df_closure.to_csv(OUT_DAY21A_CLOSURE_CSV, index=False)

print(f"[OK] Wrote closure CSV: {OUT_DAY21A_CLOSURE_CSV}")
print(f"[OK] Closure rows: {df_closure.shape[0]}")
print(df_closure.to_string(index=False))

[OK] Wrote closure CSV: /Users/louislu/pybamm-dcac-superimposed/data/day21A_step8_closure_summary.csv
[OK] Closure rows: 6
                source_type cell_or_param_set                                               protocol_pair                         evidence_status                                                                   mechanism_verdict                               interpretation_class                     Q80_common_segment                     Q90_common_segment  dt_Q80_common_raw_s  dt_Q90_common_raw_s           segment_A_above_floor_status                                                                                                                                                                                             caveat
           experimental_MJ1   LG_INR18650_MJ1                                 0.3C DC vs 0.3C+0.7C 0.1tau                               ambiguous                                                ambiguous_defer_mechanism_commitment             

In [27]:
# Cell 10B — Day21A human-readable closure note
#
# Purpose:
# - Write closure note for project handoff and JES2 §4 drafting
# - No recomputation
# - No new mechanism claim

now_utc = datetime.now(timezone.utc).isoformat()

mj1_rows = df_mj1_verdict[[
    "protocol_pair",
    "evidence_status",
    "mechanism_verdict",
    "interpretation_class",
    "Q80_common_segment",
    "Q90_common_segment",
    "dt_Q80_common_raw_s",
    "dt_Q90_common_raw_s",
    "segment_A_above_floor_status",
    "caveat",
]].copy()

diag_rows = df_diag[[
    "protocol_pair",
    "segment_A_above_floor_status",
    "diagnostic_interpretation",
    "fit_quality_status",
    "dt_resid_prescribed_p95_abs_s",
    "dt_resid_fitted_p95_abs_s",
    "dt_resid_prescribed_max_abs_s",
    "dt_resid_fitted_max_abs_s",
]].copy()

pybamm_rows = df_unified[
    df_unified["source_type"] == "PyBaMM_Day20B_full_protocol"
][[
    "cell_or_param_set",
    "evidence_status",
    "mechanism_verdict",
    "Q80_common_segment",
    "Q90_common_segment",
    "dt_Q80_common_raw_s",
    "dt_Q90_common_raw_s",
    "caveat",
]].copy()

mj1_table_text = mj1_rows.to_string(index=False)
diag_table_text = diag_rows.to_string(index=False)
pybamm_table_text = pybamm_rows.to_string(index=False)

closure_lines = []

closure_lines.append("# Day21A Closure Note — MJ1 Experimental Full-Protocol Segmentation Audit")
closure_lines.append("")
closure_lines.append(f"Generated: `{now_utc}`  ")
closure_lines.append(f"Git HEAD: `{GIT_HEAD}`  ")
closure_lines.append(f"Notebook: `{NOTEBOOK_NAME}`")
closure_lines.append("")
closure_lines.append("## 1. Scope")
closure_lines.append("")
closure_lines.append("Day21A audited the experimental MJ1 full-protocol DC–AC charging data for the 0.3C reference group:")
closure_lines.append("")
closure_lines.append("- `0.3C DC`")
closure_lines.append("- `0.3C + 0.7C 0.1τ`")
closure_lines.append("- `0.3C + 0.7C 1τ`")
closure_lines.append("- `0.3C + 0.7C 10τ`")
closure_lines.append("")
closure_lines.append("The audit followed the frozen pre-registered contract in:")
closure_lines.append("")
closure_lines.append(f"`{OUT_AUDIT_CONTRACT_JSON}`")
closure_lines.append("")
closure_lines.append("The audit uses strict-net signed current integration:")
closure_lines.append("")
closure_lines.append("- no rectification")
closure_lines.append("- no cumulative maximum")
closure_lines.append("- first-passage time at equal `Q_net`")
closure_lines.append("- Segment-A residual only in the shared prescribed-current region")
closure_lines.append("")
closure_lines.append("## 2. Core MJ1 formal verdict")
closure_lines.append("")
closure_lines.append("MJ1 formal verdict rows remain conservative:")
closure_lines.append("")
closure_lines.append("```text")
closure_lines.append(mj1_table_text)
closure_lines.append("```")
closure_lines.append("")
closure_lines.append("All MJ1 formal mechanism verdicts remain:")
closure_lines.append("")
closure_lines.append("`ambiguous_defer_mechanism_commitment`")
closure_lines.append("")
closure_lines.append("This is not a failure of the audit. It is the expected conservative outcome because the formal prescribed-geometry Segment-A residual audit flags either spike-like behavior or above-floor prescribed residuals.")
closure_lines.append("")
closure_lines.append("## 3. Diagnostic interpretation")
closure_lines.append("")
closure_lines.append("Diagnostic synthesis shows:")
closure_lines.append("")
closure_lines.append("```text")
closure_lines.append(diag_table_text)
closure_lines.append("```")
closure_lines.append("")
closure_lines.append("Interpretation:")
closure_lines.append("")
closure_lines.append("- `0.1τ`: prescribed residual behaves as spike/transition artifact; fitted waveform diagnostic is unreliable, so it is not mechanism evidence.")
closure_lines.append("- `1τ` and `10τ`: prescribed Segment-A residual is above-floor, but fitted-waveform geometry collapses p95 residuals below the MJ1 floor. Remaining large extrema are localized first-passage spikes.")
closure_lines.append("")
closure_lines.append("Therefore, the above-floor prescribed residuals in `1τ` and `10τ` should not be interpreted as distributed non-geometric Segment-A acceleration.")
closure_lines.append("")
closure_lines.append("## 4. MJ1 anchor placement")
closure_lines.append("")
closure_lines.append("The key MJ1 anchor placement is:")
closure_lines.append("")
closure_lines.append("- `1τ`: Q80/Q90 common anchors lie in Segment B.")
closure_lines.append("- `10τ`: Q80/Q90 common anchors lie in Segment B.")
closure_lines.append("- `0.1τ`: Q80 common lies in Segment A, Q90 common lies in Segment B.")
closure_lines.append("")
closure_lines.append("Segment B means:")
closure_lines.append("")
closure_lines.append("`DCAC already voltage-limited / AC-off / CV-coupled while DC remains in CC approaching Vmax`")
closure_lines.append("")
closure_lines.append("This supports a boundary/control-state mediated reading for the main Q80/Q90 gains in `1τ` and `10τ`, but the formal MJ1 verdict remains conservative because Segment-A residual diagnostics introduce caveats.")
closure_lines.append("")
closure_lines.append("## 5. PyBaMM Day20B comparison")
closure_lines.append("")
closure_lines.append("Unified PyBaMM rows:")
closure_lines.append("")
closure_lines.append("```text")
closure_lines.append(pybamm_table_text)
closure_lines.append("```")
closure_lines.append("")
closure_lines.append("PyBaMM Day20B supports:")
closure_lines.append("")
closure_lines.append("`positive_full_protocol_raw_gain_boundary_control_state_split_not_Segment_A_residual`")
closure_lines.append("")
closure_lines.append("across:")
closure_lines.append("")
closure_lines.append("- Chen2020")
closure_lines.append("- OKane2022")
closure_lines.append("- ORegan2022")
closure_lines.append("")
closure_lines.append("with Q80/Q90 common anchors located in Segment B.")
closure_lines.append("")
closure_lines.append("ORegan2022 retains the caveat:")
closure_lines.append("")
closure_lines.append("`DCAC_CV_current_transient_warning`")
closure_lines.append("")
closure_lines.append("## 6. Unified interpretation")
closure_lines.append("")
closure_lines.append("The unified MJ1–PyBaMM audit supports the following bounded interpretation:")
closure_lines.append("")
closure_lines.append("The full-protocol first-passage gains are real in the measured/simulated trajectories, but they are not mechanism-pure. The dominant supported structure is a boundary/control-state mediated first-passage gain: DC–AC reaches the voltage boundary earlier, enters AC-off / voltage-limited / CV-coupled operation while the DC reference remains in CC, and the resulting advantage persists into late CV.")
closure_lines.append("")
closure_lines.append("The audit does not support reopening Interpretation B as a demonstrated non-geometric Segment-A acceleration mechanism.")
closure_lines.append("")
closure_lines.append("## 7. Allowed claims for JES2 §4")
closure_lines.append("")
closure_lines.append("Allowed:")
closure_lines.append("")
closure_lines.append("1. MJ1 exhibits real measured full-protocol state-equivalent first-passage gains at Q80/Q90.")
closure_lines.append("2. In MJ1 `1τ` and `10τ`, Q80/Q90 common anchors lie in Segment B.")
closure_lines.append("3. Segment B corresponds to boundary/control-state split: DC–AC is already voltage-limited while DC remains in CC.")
closure_lines.append("4. Formal MJ1 verdict remains conservative and ambiguous because prescribed Segment-A residuals are above-floor or spike-like.")
closure_lines.append("5. Diagnostic audits show the above-floor prescribed residuals in `1τ` and `10τ` collapse under fitted-waveform geometry except for localized first-passage spikes.")
closure_lines.append("6. PyBaMM Day20B independently supports boundary/control-state mediated full-protocol gains across three parameter sets.")
closure_lines.append("")
closure_lines.append("## 8. Prohibited claims")
closure_lines.append("")
closure_lines.append("Do not claim:")
closure_lines.append("")
closure_lines.append("1. MJ1 proves non-geometric Segment-A acceleration.")
closure_lines.append("2. Segment-A residual above-floor in the prescribed geometry audit is direct electrochemical mechanism evidence.")
closure_lines.append("3. PyBaMM proves the physical mechanism of MJ1.")
closure_lines.append("4. All full-protocol gain is “just geometry”.")
closure_lines.append("5. Event timing alone is equivalent to state advancement.")
closure_lines.append("6. The fitted-waveform diagnostic redefines the formal residual threshold.")
closure_lines.append("")
closure_lines.append("## 9. Key output files")
closure_lines.append("")
closure_lines.append(f"- Inventory: `{OUT_FILE_INVENTORY}`")
closure_lines.append(f"- Load sanity: `{OUT_LOAD_SUMMARY}`")
closure_lines.append(f"- Event audit: `{OUT_EVENT_AUDIT}`")
closure_lines.append(f"- Q integration summary: `{OUT_Q_SUMMARY}`")
closure_lines.append(f"- Final-Q pair audit: `{OUT_FINALQ_PAIR_AUDIT}`")
closure_lines.append(f"- Segment assignment: `{OUT_SEGMENT_ASSIGNMENT}`")
closure_lines.append(f"- Δt segment audit long: `{OUT_DTQ_AUDIT_LONG}`")
closure_lines.append(f"- Δt segment summary: `{OUT_DTQ_SUMMARY}`")
closure_lines.append(f"- Residual diagnostics: `{OUT_RESID_DIAG}`")
closure_lines.append(f"- Branch-jump audit: `{OUT_BRANCH_JUMP_AUDIT}`")
closure_lines.append(f"- Geometry fidelity summary: `{OUT_GEOM_FIDELITY_SUMMARY}`")
closure_lines.append(f"- Diagnostic synthesis: `{OUT_DIAGNOSTIC_SYNTHESIS}`")
closure_lines.append(f"- MJ1 verdict: `{OUT_MJ1_VERDICT}`")
closure_lines.append(f"- Unified MJ1–PyBaMM verdict: `{OUT_UNIFIED_MJ1_PYBAMM}`")
closure_lines.append("")
closure_lines.append("## 10. Closure status")
closure_lines.append("")
closure_lines.append("Day21A is closed.")
closure_lines.append("")
closure_lines.append("Next recommended action:")
closure_lines.append("")
closure_lines.append("Write JES2 §4 as a methodological upgrade:")
closure_lines.append("")
closure_lines.append("`raw Δt(Q) is real but not mechanism-pure; full-protocol gains decompose into current geometry, voltage-boundary timing, control-state split, late-CV preservation, and possible residual terms. In the current MJ1 audit, non-geometric Segment-A acceleration is not supported, while boundary/control-state mediated first-passage gain is the preferred interpretation with diagnostic caveats.`")

closure_md = "\n".join(closure_lines)
OUT_DAY21A_CLOSURE_MD.write_text(closure_md, encoding="utf-8")

print(f"[OK] Wrote closure note: {OUT_DAY21A_CLOSURE_MD}")
print(f"[OK] Day21A audit chain closed at Git HEAD: {GIT_HEAD}")
print(f"[OK] Closure CSV: {OUT_DAY21A_CLOSURE_CSV}")
print(f"[OK] Closure Markdown: {OUT_DAY21A_CLOSURE_MD}")

[OK] Wrote closure note: /Users/louislu/pybamm-dcac-superimposed/data/day21A_step8_closure_note.md
[OK] Day21A audit chain closed at Git HEAD: bf7514db35b9f4e407fb5f6a2942e519a27b0953
[OK] Closure CSV: /Users/louislu/pybamm-dcac-superimposed/data/day21A_step8_closure_summary.csv
[OK] Closure Markdown: /Users/louislu/pybamm-dcac-superimposed/data/day21A_step8_closure_note.md
